# Brain Tumor MRI — Agreement-Guided Fusion (AGF): main XAI evaluation

Run this notebook top-to-bottom (*Save & Run All*) to regenerate every result.

## Scientific question

> Under what conditions does agreement-guided attribution fusion improve explanation faithfulness and
> robustness relative to individual attribution methods and equal-weight fusion, and how does this
> behaviour vary across model architectures?

The notebook does **not** assume AGF wins. Each metric (insertion, deletion, robustness) is tested and
reported separately, per backbone, against **Equal Fusion (central baseline)** and, as a secondary
family, against the three individual methods.

## Method under test

* Base attributions: **Grad-CAM++** (final 7×7 spatial feature map), **Integrated Gradients**
  (zero baseline in normalised space, |IG| channel-mean), **SmoothGrad** (|∇| channel-mean, relative noise).
* Every map is min–max normalised, resized to 224×224 (Lanczos) and clipped at 0.
* **AGF**: per-image pairwise agreement with the **top-*k* Dice kernel** (each map binarised at its own
  top-10 % quantile), per-method agreement *a_k* = mean Dice with the other two maps, weights
  *w_k* = softmax(*a_k*/τ) with τ = 1, fused map = Σ *w_k M_k*, re-scaled by its maximum.
  Weights are computed from the three maps of the same image only — no label, no faithfulness score and
  no robustness score enters the weights (§11, leakage audit).
* **Equal Fusion**: identical pipeline with *w_k* = 1/3.
* **AGF has hyper-parameters** (τ, top-*k* fraction *q*). They were fixed before evaluation (τ = 1,
  *q* = 0.10) and are ablated, never selected, on test data (§16).

## Metric definitions (exact, used everywhere)

| Name in CSV / tables | Definition | Direction |
|---|---|---|
| `aopc_ins` | mean over steps *s* = 1…10 of softmax *p*(ĉ \| x_s), x_s = image with the top 10·*s* % pixels of the map revealed on a neutral-grey (pixel 0.5) canvas. ĉ = clean-image prediction, fixed. | ↑ |
| `aopc_del_raw` | same, top 10·*s* % pixels replaced by grey | ↓ |
| `aopc_del_comp` | 1 − `aopc_del_raw` (reported deletion score) | ↑ |
| `sensitivity` | max over 10 Gaussian perturbations (σ = 0.02, normalised input units) of ‖Φ(x+δ) − Φ(x)‖₂ / ‖Φ(x)‖₂, Φ = post-processed 224×224 map, ĉ fixed, SmoothGrad noise frozen across the pair | ↓ |

"AOPC" is kept as the column name for continuity; strictly, these are normalised areas under the
insertion/deletion curves (10-point right Riemann sum), not the Samek et al. AOPC of score drops. Use the
definition above in the paper.

## Structure

| § | Content | § | Content |
|---|---|---|---|
| 1 | Final experiment configuration | 13 | Faithfulness tables (per backbone) |
| 2 | Environment & reproducibility | 14 | Per-backbone figures |
| 3 | Output tree & utilities | 15 | Statistical analysis (from per-image CSVs) |
| 4 | Preprocessing (benchmark-identical) | 16 | Ablations: robustness (Full/Equal/Fixed), corruption probe, weighting/kernel/temperature |
| 5 | Dataset, checkpoints, calibration split | 17 | Cross-architecture analysis |
| 6 | Model architecture | 18 | Per-class analysis (descriptive) — inside §15 |
| 7 | Checkpoint, preprocessing-parity & target-layer verification | 19 | Weight diagnostics — §15 + §16c |
| 8–9 | XAI + AGF implementation (`xai_attribution.py`) | 20 | Qualitative figure |
| 10 | Sanity checks | 21 | Publication tables |
| 11 | Calibration & evaluation loops, leakage audit | 22 | Exports |
| 12 | Main evaluation (all backbones) | 23–24 | Experiment manifest, reproducibility summary |

Some code comments refer to internal development tags (v6/v7) that explain *why* a line exists.

### Inputs
1. Benchmarking notebook output (`models/{vgg16,densenet121,resnet50}_best.pth`, `*_probe.npz`,
   `*_preds.npz`, `*_probs.npz`, `benchmark_results.json`). Set `MODEL_DIR` in §1.
2. `masoudnickparvar/brain-tumor-mri-dataset` (`Training/` is needed for the calibration split,
   `Testing/` for evaluation).

Target: Kaggle, one T4 (FP32 inference), ≤ 30 GB RAM.

## §1 Experiment configuration — the single source of truth

Every later cell reads these names. No section re-defines them; an ablation that deliberately deviates
does so with a locally named variable that is printed and saved with its results.

In [1]:
# ── §1  FINAL EXPERIMENT CONFIGURATION ─────────────────────────────────────
# Pure Python: no torch import here, so this cell can be read/diffed on its own.
import os

CONFIG_VERSION = "agf-final-1.0"          # stamped into every per-image CSV + manifest

# ── Paths ──────────────────────────────────────────────────────────────────
MODEL_DIR = os.environ.get(
    "BT_MODEL_DIR",
    "/kaggle/input/notebooks/tamannajarintima/benchmarking",   # benchmarking output
)
_DATA_ROOT_CANDIDATES = [
    os.environ.get("BT_DATASET_ROOT", ""),
    "/kaggle/input/brain-tumor-mri-dataset",
    "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset",
]
DATASET_ROOT = next((p for p in _DATA_ROOT_CANDIDATES if p and os.path.exists(p)),
                    _DATA_ROOT_CANDIDATES[-1])
TEST_DIR  = os.path.join(DATASET_ROOT, "Testing")
TRAIN_DIR = os.path.join(DATASET_ROOT, "Training")      # calibration split source
OUTPUT_DIR = os.environ.get("BT_OUTPUT_DIR", "/kaggle/working")
OUTPUT_SUBDIRS = ["Results", "Tables", "Figures", "Statistics", "Logs", "Exports"]

# ── Classification pipeline (READ-ONLY mirror of benchmarking.ipynb) ───────
IMG_SIZE    = 224
NUM_CLASSES = 4
CLASS_NAMES_DEFAULT = ["glioma", "meningioma", "notumor", "pituitary"]
NORM_MEAN   = [0.485, 0.456, 0.406]
NORM_STD    = [0.229, 0.224, 0.225]
HEAD_DROPOUT = 0.4                        # benchmarking DROPOUT_RATE (inactive in eval)
BENCH_RANDOM_SEED = 42                    # benchmarking RANDOM_SEED (train/val split)
BENCH_VAL_FRAC    = 0.15                  # benchmarking VAL_FRAC
# Benchmarking preprocessing cache version this notebook must reproduce:
BENCH_PREPROC_VERSION = "v2:removeblackborder(pad4,thr15,halfopen)+clahe(clip2.0,tile8x8)"
INFERENCE_PRECISION = "fp32"              # XAI model has no autocast; gradients in FP32

MODELS = [
    {"key": "vgg16",       "backbone": "vgg16_bn",    "display": "VGG16-BN",    "ckpt": "vgg16_best.pth"},
    {"key": "densenet121", "backbone": "densenet121", "display": "DenseNet121", "ckpt": "densenet121_best.pth"},
    {"key": "resnet50",    "backbone": "resnet50",    "display": "ResNet50",    "ckpt": "resnet50_best.pth"},
]
RUN_MODELS = ["vgg16", "densenet121", "resnet50"]

# Target layers (resolved and VERIFIED in §7 against the instantiated timm model):
#   vgg16_bn    -> backbone.features      (= output of features[43] MaxPool, 512x7x7;
#                                           features[42] ReLU is 14x14)
#   densenet121 -> backbone.features      (= output of features.norm5 BatchNormAct2d, 1024x7x7)
#   resnet50    -> backbone.layer4        (post-residual, post-ReLU, 2048x7x7)
TARGET_LAYERS = {"vgg16_bn": "features", "densenet121": "features", "resnet50": "layer4"}
TARGET_EXPECTED_HW = (7, 7)

# ── Seeds (every subset has its own, all derived from SEED) ────────────────
SEED            = 42
EVAL_SEED       = SEED          # main evaluation subset
SENS_SEED       = SEED + 1      # robustness subset
ABL_SEED        = SEED + 2      # AGF Full/Equal/Fixed faithfulness ablation subset
WABL_SEED       = SEED + 3      # weighting/kernel/temperature ablation subset
ROBUST_ABL_SEED = SEED + 7      # robustness ablation subset
CORRUPT_SEED    = SEED          # corruption-probe subset
CALIB_SEED      = SEED + 11     # calibration draw from the validation split

# ── Subset sizes ───────────────────────────────────────────────────────────
EVAL_N        = 200             # faithfulness (stratified, 50/class)
GINI_SENS_N   = 50              # robustness (stratified)
ABLATION_N    = 100             # Full/Equal/Fixed faithfulness ablation
CALIB_N       = 100             # AGF-Fixed weight calibration (validation split, NOT test)
PARITY_CHECK_N = None           # None = all 1600 test images for the prediction-parity check
PARITY_WARN_AGREEMENT = 0.995   # clean FP32 vs benchmark clean (AMP) prediction agreement
PARITY_FAIL_AGREEMENT = 0.980   # below this the preprocessing is NOT the benchmark's -> stop

# ── XAI method parameters ──────────────────────────────────────────────────
SG_N_SAMPLES  = 20
SG_NOISE_STD  = 0.10            # only used when SG_NOISE_MODE == "absolute"
SG_NOISE_MODE = "relative"      # sigma = SG_NOISE_FRAC * (max - min) of the normalised input
SG_NOISE_FRAC = 0.15
SG_SEED_MODE  = "content"       # noise seed = hash(image bytes) ^ SEED
IG_N_STEPS    = 30              # trapezoidal rule, 31 evaluations
IG_BASELINE   = "zeros_normalized"   # all-zero tensor in NORMALISED space,
                                     # i.e. the ImageNet-mean colour. The only implemented option.
                                     # NOTE: differs from the AOPC grey baseline; disclosed, not changed.
MAP_NORMALIZATION = "minmax_per_map"          # base maps
MAP_POSTPROC      = "lanczos_resize_224_then_clip_ge0"   # applied identically in every loop
FUSED_NORMALIZATION = "clip_ge0_then_divide_by_max"

# ── AGF ────────────────────────────────────────────────────────────────────
AGREEMENT_KERNEL = "topk_dice"  # the configured AND executed kernel of the v7.1 run
AGREEMENT_TAU    = 1.0
TOPK_FRAC        = 0.10
AGF_WEIGHT_RULE  = "softmax"
KERNEL_ABLATION  = ["ssim_cosine", "pearson", "topk_dice"]

# ── Faithfulness (insertion / deletion) ────────────────────────────────────
AOPC_N_STEPS        = 10
AOPC_PATCH_PCT      = 0.10      # fraction of PIXELS added per step (pixel-level ranking, no patches)
AOPC_BASELINE_PIXEL = 0.5       # neutral grey in pixel space, converted per channel
AOPC_TIE_BREAK_SEED = SEED      # constant tie-break permutation, identical for all methods
AOPC_SCORE          = "softmax_probability"
AOPC_TARGET_CLASS   = "clean_prediction_fixed"       # never re-derived
AOPC_AGGREGATION    = "mean_over_steps_1_to_N"       # step 0 excluded

# ── Robustness (max-sensitivity) ───────────────────────────────────────────
SENS_RADIUS          = 0.02     # per-element Gaussian std in NORMALISED units (not an L2 radius)
SENS_SAMPLES         = 10
SENS_NOISE           = "gaussian_iid_normalized"
SENS_FREEZE_SG_NOISE = True

# ── Ablations (moved here from their cells; values unchanged) ──────────────
ROBUST_ABL_N       = 24
ROBUST_ABL_SAMPLES = 4
PROBE_BACKBONE     = "densenet121"
N_CORRUPT          = 40
CORRUPTIONS        = ["gaussian_noise", "gaussian_blur", "brightness", "contrast"]
SEVERITY = {
    "gaussian_noise": [0.10, 0.25, 0.50, 0.75],
    "gaussian_blur":  [0.50, 1.00, 2.00, 3.00],
    "brightness":     [0.25, 0.50, 0.75, 1.00],
    "contrast":       [0.75, 0.50, 0.35, 0.20],
}
N_ABL                = 40
TEMP_GRID            = [0.25, 0.10, 0.05]
WEIGHT_SPREAD_THRESH = 0.02
QUAL_BACKBONE        = "densenet121"

# ── Statistics ─────────────────────────────────────────────────────────────
ALPHA               = 0.05
STATS_TEST          = "wilcoxon_signed_rank"
STATS_ALTERNATIVE   = "two-sided"
STATS_ZERO_METHOD   = "wilcox"
MULTIPLE_CORRECTION = "holm"
BOOTSTRAP_N         = 1000
# pre-declared hypothesis families. Holm is applied across ALL tests of a family.
PRIMARY_REFERENCE   = "equal"                           # central baseline
PRIMARY_METRICS     = ["aopc_ins", "aopc_del_comp", "sensitivity"]
SECONDARY_BASELINES = ["gradcam_pp", "ig", "smoothgrad"]
METRIC_DIRECTION    = {"aopc_ins": +1, "aopc_del_comp": +1, "aopc_del_raw": -1, "sensitivity": -1}
METRIC_LABEL = {"aopc_ins": "Insertion (AOPC-Ins, higher better)",
                "aopc_del_comp": "Deletion (1 - AOPC-Del, higher better)",
                "aopc_del_raw": "Deletion (raw AOPC-Del, lower better)",
                "sensitivity": "Robustness (max-sensitivity, lower better)"}

# ── Method identity ────────────────────────────────────────────────────────
DISP = {"gradcam_pp": "GradCAM++", "ig": "IG", "smoothgrad": "SmoothGrad",
        "equal": "Equal Fusion", "agf": "AGF (Proposed)"}
ORDER      = ["gradcam_pp", "ig", "smoothgrad", "equal", "agf"]   # Equal Fusion in the main eval
PROPOSED   = "agf"
BASE_KEYS  = ["gradcam_pp", "ig", "smoothgrad"]    # individual attribution methods
FUSION_KEYS = ["equal", "agf"]
BASELINES  = ["equal"] + BASE_KEYS                  # everything AGF is compared against
MCOLOR = {"gradcam_pp": "#E63946", "ig": "#2A9D8F", "smoothgrad": "#457B9D",
          "equal": "#8D99AE", "agf": "#F4A261"}

# ── Consistency guards ─────────────────────────────────────────────────────
assert AGREEMENT_KERNEL in KERNEL_ABLATION
assert IG_BASELINE == "zeros_normalized"
assert set(SECONDARY_BASELINES) == set(BASE_KEYS)
assert PRIMARY_REFERENCE in ORDER and PROPOSED in ORDER
assert all(m in METRIC_DIRECTION for m in PRIMARY_METRICS)
assert PROBE_BACKBONE in {m["key"] for m in MODELS} and QUAL_BACKBONE in {m["key"] for m in MODELS}

_FROZEN_NAMES = [
    "CONFIG_VERSION", "MODEL_DIR", "DATASET_ROOT", "TEST_DIR", "TRAIN_DIR", "OUTPUT_DIR",
    "IMG_SIZE", "NUM_CLASSES", "CLASS_NAMES_DEFAULT", "NORM_MEAN", "NORM_STD", "HEAD_DROPOUT",
    "BENCH_RANDOM_SEED", "BENCH_VAL_FRAC", "BENCH_PREPROC_VERSION", "INFERENCE_PRECISION",
    "RUN_MODELS", "TARGET_LAYERS", "TARGET_EXPECTED_HW",
    "SEED", "EVAL_SEED", "SENS_SEED", "ABL_SEED", "WABL_SEED", "ROBUST_ABL_SEED",
    "CORRUPT_SEED", "CALIB_SEED",
    "EVAL_N", "GINI_SENS_N", "ABLATION_N", "CALIB_N", "PARITY_CHECK_N",
    "PARITY_WARN_AGREEMENT", "PARITY_FAIL_AGREEMENT",
    "SG_N_SAMPLES", "SG_NOISE_STD", "SG_NOISE_MODE", "SG_NOISE_FRAC", "SG_SEED_MODE",
    "IG_N_STEPS", "IG_BASELINE", "MAP_NORMALIZATION", "MAP_POSTPROC", "FUSED_NORMALIZATION",
    "AGREEMENT_KERNEL", "AGREEMENT_TAU", "TOPK_FRAC", "AGF_WEIGHT_RULE", "KERNEL_ABLATION",
    "AOPC_N_STEPS", "AOPC_PATCH_PCT", "AOPC_BASELINE_PIXEL", "AOPC_TIE_BREAK_SEED",
    "AOPC_SCORE", "AOPC_TARGET_CLASS", "AOPC_AGGREGATION",
    "SENS_RADIUS", "SENS_SAMPLES", "SENS_NOISE", "SENS_FREEZE_SG_NOISE",
    "ROBUST_ABL_N", "ROBUST_ABL_SAMPLES", "PROBE_BACKBONE", "N_CORRUPT", "CORRUPTIONS",
    "SEVERITY", "N_ABL", "TEMP_GRID", "WEIGHT_SPREAD_THRESH", "QUAL_BACKBONE",
    "ALPHA", "STATS_TEST", "STATS_ALTERNATIVE", "STATS_ZERO_METHOD", "MULTIPLE_CORRECTION",
    "BOOTSTRAP_N", "PRIMARY_REFERENCE", "PRIMARY_METRICS", "SECONDARY_BASELINES",
    "METRIC_DIRECTION", "ORDER", "PROPOSED", "BASE_KEYS", "FUSION_KEYS", "BASELINES",
]
FROZEN_CONFIG = {k: globals()[k] for k in _FROZEN_NAMES}

print(f"CONFIG {CONFIG_VERSION} | RUN_MODELS={RUN_MODELS}")
print(f"EVAL_N={EVAL_N} GINI_SENS_N={GINI_SENS_N} ABLATION_N={ABLATION_N} CALIB_N={CALIB_N}")
print(f"AGF kernel={AGREEMENT_KERNEL} tau={AGREEMENT_TAU} topk={TOPK_FRAC} | primary reference={DISP[PRIMARY_REFERENCE]}")
print(f"MODEL_DIR={MODEL_DIR}\nTEST_DIR ={TEST_DIR} (exists={os.path.exists(TEST_DIR)})"
      f"\nTRAIN_DIR={TRAIN_DIR} (exists={os.path.exists(TRAIN_DIR)})")

CONFIG agf-final-1.0 | RUN_MODELS=['vgg16', 'densenet121', 'resnet50']
EVAL_N=200 GINI_SENS_N=50 ABLATION_N=100 CALIB_N=100
AGF kernel=topk_dice tau=1.0 topk=0.1 | primary reference=Equal Fusion
MODEL_DIR=/kaggle/input/notebooks/tamannajarintima/benchmarking
TEST_DIR =/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing (exists=True)
TRAIN_DIR=/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training (exists=True)


## §2 Environment and reproducibility

`CUBLAS_WORKSPACE_CONFIG` is set **before** `import torch` (it must precede CUDA initialisation).
`PYTHONHASHSEED` cannot change the hash seed of an already-running interpreter, so no computation in this
notebook relies on Python's built-in `hash()`.
Inference is FP32 (the XAI model class has no autocast); one GPU (`cuda:0`) is used, no `DataParallel`.

In [2]:
# ── §2  Environment, imports, determinism ──────────────────────────────────
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # must precede CUDA init

import sys, gc, json, time, glob, random, shutil, warnings, traceback, hashlib, zlib, platform
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from tqdm.auto import tqdm
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import timm

import scipy
from scipy.stats import wilcoxon
import statsmodels
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")


def seed_everything(seed: int = SEED, deterministic: bool = True):
    """Seed Python / NumPy / torch (CPU+CUDA) and pin cuDNN to deterministic kernels.
    Deterministic kernels matter here: the effects under test (e.g. AGF vs Equal)
    are of the order 1e-3, and non-deterministic cuDNN jitter was measured at ~1e-4."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as _e:
            print(f"[warn] torch.use_deterministic_algorithms unavailable: {_e}")


def stable_seed(*parts) -> int:
    """Process-independent integer seed from arbitrary parts (replaces hash())."""
    return zlib.crc32("|".join(map(str, parts)).encode("utf-8")) & 0x7FFFFFFF


seed_everything(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ENV_INFO = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "timm": timm.__version__, "numpy": np.__version__,
    "scipy": scipy.__version__, "statsmodels": statsmodels.__version__,
    "opencv": cv2.__version__, "pandas": pd.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    "device": str(DEVICE), "precision": INFERENCE_PRECISION,
}
for k, v in ENV_INFO.items():
    print(f"{k:<24}: {v}")

python                  : 3.12.13
platform                : Linux-6.12.90+-x86_64-with-glibc2.35
torch                   : 2.10.0+cu128
timm                    : 1.0.26
numpy                   : 2.0.2
scipy                   : 1.16.3
statsmodels             : 0.14.6
opencv                  : 4.13.0
pandas                  : 2.3.3
cuda_available          : True
gpu                     : Tesla T4
cudnn_deterministic     : True
cudnn_benchmark         : False
cublas_workspace_config : :4096:8
device                  : cuda:0
precision               : fp32


## §3 Output tree, logging and save helpers

Output folders, logging, the whitelist of saved tables and a section-status registry used by the
experiment manifest.

In [3]:
# ── §3  Output tree, logging, and universal save helpers ───────────────────
# Folders required by the brief. Every artifact is written automatically;
# nothing needs manual saving.
DIRS = {name: os.path.join(OUTPUT_DIR, name) for name in OUTPUT_SUBDIRS}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

_LOG_PATH = os.path.join(DIRS["Logs"], "run.log")
def log(msg: str):
    """Print to console AND append to Logs/run.log with a timestamp."""
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line)
    with open(_LOG_PATH, "a") as f:
        f.write(line + "\n")

def free_gpu(*names, scope=None):
    """Delete named globals if present, then gc + empty CUDA cache.

    `import gc` is LOCAL on purpose. Cells that run their loops at module level
    can rebind and then `del` a global named `gc`, which silently removes the
    module and makes every later free_gpu() raise NameError -- exactly what took
    down Section 19 (and with it Sections 20/14/15) in v7. A local import makes
    this helper immune to whatever the notebook namespace does.
    """
    import gc
    g = scope if scope is not None else globals()
    for n in names:
        if n in g:
            try:    del g[n]
            except Exception: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Publication curation: emit ONLY the assets a paper/thesis needs ─────────
# Set PUBLICATION_ONLY = False to restore the full diagnostic figure/table dump.
PUBLICATION_ONLY = True
KEEP_FIGURES = {
    "aopc_curves",                       # per-backbone faithfulness curves
    "cross_arch_aopc_ins",               # cross-architecture faithfulness bar
    "agf_generalization",                # AGF vs best baseline per backbone
    "agf_generalization_forest",         # paired-delta forest plot (preferred)
    "robustness_ablation",               # Full / Equal / Fixed
    "corruption_probe_densenet121",      # faithfulness under corruption
    "weighting_ablation_densenet121",   # weight distribution + temperature
    "qualitative_grid",                  # per-backbone class x method grid
    "qualitative_grid_publication",      # Section 19 class x method overlay grid
    "cross_overall_ranking",             # FIX: the ranking TABLE was whitelisted
                                         # but its figure was silently dropped
}
KEEP_TABLES = {
    "xai_metrics", "statistics", "statistics_robustness", "per_class",  # per-backbone (S1/T3/S2/T5b)
    "cross_classification_results", "cross_xai_aopc_ins",  # T1 / T2
    "cross_overall_ranking", "cross_agf_generalization",   # rank / generalization
    "cross_robustness_sens", "cross_robustness_stats",     # T5 robustness (main body)
    "robustness_ablation_means", "robustness_ablation_stats",     # T4
    "corruption_weighting_densenet121",                   # S3
    "agf_temperature_ablation_densenet121",                # S4
    "agf_weight_diagnostic_densenet121",                   # weight diagnostic
    # FIX: these two are produced and promised in Sections 17/18's markdown
    # but were absent from the whitelist, so save_table() discarded them
    # silently and the deliverable was missing outputs the notebook claimed
    # to have written.
    "corruption_summary_densenet121",                      # Section 17
    "agf_kernel_ablation_densenet121",                 # Section 18
    "runtime", "ablation",                                 # cost + ablation tables
}

# ── v7.1  Per-backbone artifact names, generated for EVERY backbone ────────
#   The literal sets above hard-code "..._densenet121". save_table() and
#   savefig_all() therefore SILENTLY DISCARD the Section 17/18 outputs of any
#   other backbone: a VGG or ResNet run prints its tables, logs success, and
#   writes nothing. The same bug already cost the v7 DenseNet run two
#   artifacts -- there was no entry for `agf_kernel_scores_densenet121` or for
#   the `kernel_ablation_densenet121` FIGURE, so both were computed and thrown
#   away.
#
#   This block is STRICTLY ADDITIVE: every name the literal sets already
#   contain is regenerated identically, so the DenseNet121 curation is
#   unchanged and no previously-emitted artifact can disappear.
for _bb in [_s["key"] for _s in MODELS]:
    KEEP_FIGURES.update({
        f"corruption_probe_{_bb}",          # Section 17
        f"weighting_ablation_{_bb}",        # Section 18 fig 1
        f"kernel_ablation_{_bb}",           # Section 18 fig 2  (was missing)
    })
    KEEP_TABLES.update({
        f"corruption_summary_{_bb}", f"corruption_weighting_{_bb}",     # Section 17
        f"agf_weight_diagnostic_{_bb}", f"agf_temperature_ablation_{_bb}",
        f"agf_kernel_ablation_{_bb}",
        f"agf_kernel_scores_{_bb}",         # Section 18            (was missing)
    })
KEEP_FIGURES.update({"kernel_ablation_cross"})
KEEP_TABLES.update({"agf_kernel_ablation_cross",
                    "agf_temperature_ablation_cross",
                    "agf_weight_diagnostic_cross"})
del _bb

def savefig_all(fig, name: str, subdir: str = None, dpi: int = 300):
    """Save a figure to Figures/ as PNG + PDF + SVG at >=300 DPI, then close it."""
    if PUBLICATION_ONLY and name not in KEEP_FIGURES:
        plt.close(fig); return []
    base = DIRS["Figures"] if subdir is None else os.path.join(DIRS["Figures"], subdir)
    os.makedirs(base, exist_ok=True)
    paths = []
    for ext in ("png", "pdf", "svg"):
        p = os.path.join(base, f"{name}.{ext}")
        fig.savefig(p, dpi=dpi, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        paths.append(p)
    plt.close(fig)
    del fig
    return paths

def _keep_table(name: str) -> bool:
    """Whitelist test for tables.

    FIX: per-backbone tables are saved as "<Tag>_<base>" (e.g.
    "VGG16BN_statistics") while KEEP_TABLES stores the bare base name
    ("statistics"). A plain `name in KEEP_TABLES` test therefore dropped
    EVERY per-backbone table, and Section 20 later crashed with
    FileNotFoundError when it tried to read them. Accept an exact match or
    a "<prefix>_<base>" suffix match.
    """
    if not PUBLICATION_ONLY:
        return True
    if name in KEEP_TABLES:
        return True
    return any(name.endswith("_" + k) for k in KEEP_TABLES)

def save_table(df: pd.DataFrame, name: str, subdir: str = None,
               float_fmt: str = "%.4f", index: bool = False):
    """Save a DataFrame to Tables/ as CSV + LaTeX (booktabs). Returns paths."""
    if not _keep_table(name):
        return []
    base = DIRS["Tables"] if subdir is None else os.path.join(DIRS["Tables"], subdir)
    os.makedirs(base, exist_ok=True)
    csv_p = os.path.join(base, f"{name}.csv")
    tex_p = os.path.join(base, f"{name}.tex")
    df.to_csv(csv_p, index=index)
    try:
        df.to_latex(tex_p, index=index, float_format=lambda x: float_fmt % x,
                    escape=True, bold_rows=False)
    except Exception as e:
        with open(tex_p, "w") as f:
            f.write(f"% LaTeX export failed: {e}\n")
    return csv_p, tex_p

# One Excel workbook accumulates every table as a sheet (written at the end).
_EXCEL_SHEETS: Dict[str, pd.DataFrame] = {}
def register_excel(sheet_name: str, df: pd.DataFrame):
    """Queue a DataFrame to become a sheet in Exports/all_tables.xlsx."""
    _EXCEL_SHEETS[sheet_name[:31]] = df.copy()      # Excel sheet-name cap = 31 chars

# ── final-version additions to the whitelist (reporting outputs only) ───────
KEEP_FIGURES.update({
    "primary_agf_vs_equal_forest",       # §17 primary-family forest plot
    "cross_arch_metric_means",           # §17 ins / del / sens means per backbone
    "weight_distribution_main",          # §15 main-loop AGF weights
})
KEEP_TABLES.update({
    "stats_primary", "stats_secondary",                  # §15 inferential results
    "per_class_descriptive", "weight_diagnostic_main",   # §15 descriptive
    "cross_metric_means", "calibration_weights",         # §15 / §11
    "prediction_parity", "target_layer_verification",    # §7
})
for _sub in ("calibration", "per_image"):
    os.makedirs(os.path.join(DIRS["Statistics"], _sub), exist_ok=True)

# Section status registry -> written into the experiment manifest (§23).
SECTION_STATUS: Dict[str, str] = {}
def mark_section(name: str, status: str = "completed"):
    SECTION_STATUS[name] = f"{status} @ {time.strftime('%Y-%m-%d %H:%M:%S')}"

log("Output tree ready: " + ", ".join(DIRS.keys()))

[13:23:37] Output tree ready: Results, Tables, Figures, Statistics, Logs, Exports


## §4 Preprocessing — identical to `benchmarking.ipynb` (source of truth)

The benchmarking classes are copied below verbatim, so every XAI input is exactly the input the
classifiers were trained and evaluated on. `RemoveBlackBorder` uses a half-open PIL crop box (`+1` on the
right/lower bound) and the `MIN_SIDE` guard; `ApplyCLAHE` is stateless. Order: `convert("RGB")` → crop →
CLAHE → `Resize(224)` → `ToTensor` → `Normalize`, exactly as cache build + `val_test_transform`.
Parity with the benchmark's own clean predictions is verified in §7.

In [4]:
# ── §4  Preprocessing transforms (verbatim from benchmarking.ipynb §3.1/§3.2) ──
class ApplyCLAHE:
    """CLAHE on the L channel of LAB (stateless; identical to benchmarking.ipynb)."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit     = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, pil_img):
        img_np = np.array(pil_img.convert("RGB"))
        lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        clahe  = cv2.createCLAHE(clipLimit=self.clip_limit,
                                 tileGridSize=self.tile_grid_size)
        lab[:, :, 0] = clahe.apply(np.ascontiguousarray(lab[:, :, 0]))
        return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))


class RemoveBlackBorder:
    """Crop the black scanner border (identical to benchmarking.ipynb: half-open
    PIL box with +1, MIN_SIDE guard)."""
    MIN_SIDE = 8

    def __call__(self, img):
        arr  = np.array(img.convert("L"))
        mask = arr > 15
        rows, cols = np.any(mask, axis=1), np.any(mask, axis=0)
        if not rows.any() or not cols.any():
            return img
        r_min, r_max = np.where(rows)[0][[0, -1]]
        c_min, c_max = np.where(cols)[0][[0, -1]]
        pad  = 4
        h, w = arr.shape
        r_min, r_max = max(0, r_min - pad), min(h - 1, r_max + pad)
        c_min, c_max = max(0, c_min - pad), min(w - 1, c_max + pad)
        box = (int(c_min), int(r_min), int(c_max) + 1, int(r_max) + 1)
        if (box[2] - box[0]) < self.MIN_SIDE or (box[3] - box[1]) < self.MIN_SIDE:
            return img
        return img.crop(box)


PREPROC_VERSION = "v2:removeblackborder(pad4,thr15,halfopen)+clahe(clip2.0,tile8x8)"
assert PREPROC_VERSION == BENCH_PREPROC_VERSION, "preprocessing drifted from the benchmark"

val_test_transform = transforms.Compose([
    RemoveBlackBorder(),
    ApplyCLAHE(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])


def load_image_tensor(path: str) -> torch.Tensor:
    """The ONLY image loader used by every XAI loop (PIL convert('RGB') first,
    as ImageFolder's default loader does in the benchmark)."""
    with Image.open(path) as im:
        return val_test_transform(im.convert("RGB"))

print(f"Transforms ready ({PREPROC_VERSION}).")

Transforms ready (v2:removeblackborder(pad4,thr15,halfopen)+clahe(clip2.0,tile8x8)).


## §5 Dataset, checkpoints and calibration split

* **Evaluation data (B):** `Testing/` (1 600 images, 400/class) — the benchmark's held-out test set.
  Evaluation subsets are stratified draws with the seeds in §1.
* **Calibration data (A):** the benchmark's own **validation split** (15 % of `Training/`,
  `StratifiedShuffleSplit(random_state=42)`), reconstructed below and checked against the benchmark's saved
  validation labels. It is used for exactly one thing: the static weight vector of the **AGF-Fixed**
  ablation arm, so calibration and evaluation data are disjoint.
* Checkpoint loading asserts `backbone_name`, `num_classes`, `class_names`, `img_size`, `in_features`
  against the benchmark contract and loads with `strict=True`.

In [5]:
# ── §5  Test data, checkpoint discovery, per-model loader ────────────────
if not os.path.exists(TEST_DIR):
    raise FileNotFoundError(
        f"TEST_DIR not found: {TEST_DIR}. Attach 'masoudnickparvar/"
        f"brain-tumor-mri-dataset' or set BT_TEST_DIR.")

test_dataset  = datasets.ImageFolder(TEST_DIR)
sample_paths  = [s[0] for s in test_dataset.samples]
sample_labels = [s[1] for s in test_dataset.samples]
# Stable image ID = "<class>/<file stem>" (extension dropped: the benchmark cache
# stores the same images as .png, ImageFolder order is identical, see §7).
sample_ids    = [os.path.join(os.path.basename(os.path.dirname(p)),
                              os.path.splitext(os.path.basename(p))[0]) for p in sample_paths]

# class names / count: prefer the benchmarking manifest, else ImageFolder order
BENCH_JSON = None
_bench_candidates = [os.path.join(MODEL_DIR, "models", "benchmark_results.json"),
                     os.path.join(MODEL_DIR, "benchmark_results.json")]
_bench_candidates += glob.glob(os.path.join(MODEL_DIR, "**", "benchmark_results.json"),
                               recursive=True)
for _cand in _bench_candidates:
    if os.path.exists(_cand):
        BENCH_JSON = json.load(open(_cand)); break

class_names = list(test_dataset.classes)
assert class_names == CLASS_NAMES_DEFAULT, f"class order {class_names} != benchmark {CLASS_NAMES_DEFAULT}"
num_classes = len(class_names)
assert num_classes == NUM_CLASSES

log(f"Test samples: {len(sample_paths)} | classes={class_names}")
for i, c in enumerate(class_names):
    log(f"    {c}: {sample_labels.count(i)}")

# ── stratified, seeded sampler bound to this dataset ───────────────────────
_cls_to_idxs = defaultdict(list)
for _i, (_, _lbl) in enumerate(test_dataset.samples):
    _cls_to_idxs[int(_lbl)].append(_i)

def stratified_sample(total_n, seed):
    rng = np.random.default_rng(seed)
    base, rem = divmod(total_n, num_classes)
    chosen = []
    for cls_idx in range(num_classes):
        want = base + (1 if cls_idx < rem else 0)
        pool = _cls_to_idxs[cls_idx]
        want = min(want, len(pool))
        chosen.extend(rng.choice(pool, size=want, replace=False).tolist())
    return sorted(chosen)

def class_counts(idxs):
    c = {cn: 0 for cn in class_names}
    for i in idxs:
        c[class_names[sample_labels[i]]] += 1
    return c

# ── checkpoint discovery ───────────────────────────────────────────────────
def find_artifact(fname):
    """Locate a benchmarking artifact under MODEL_DIR (models/ first, then recursive)."""
    for cand in [os.path.join(MODEL_DIR, "models", fname), os.path.join(MODEL_DIR, fname)]:
        if os.path.exists(cand):
            return cand
    hits = sorted(glob.glob(os.path.join(MODEL_DIR, "**", fname), recursive=True))
    return hits[0] if hits else None

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def find_ckpt(spec):
    for cand in [os.path.join(MODEL_DIR, "models", spec["ckpt"]),
                 os.path.join(MODEL_DIR, spec["ckpt"])]:
        if os.path.exists(cand):
            return cand
    hits = glob.glob(os.path.join(MODEL_DIR, "**", spec["ckpt"]), recursive=True)
    return hits[0] if hits else None

print("\nCheckpoint discovery:")
_missing = []
for spec in MODELS:
    p = find_ckpt(spec)
    tag = p if p else "*** NOT FOUND ***"
    print(f"  {spec['display']:<14} -> {tag}")
    if p is None and spec["key"] in RUN_MODELS:
        _missing.append(spec["key"])
if _missing:
    _tree = "\n".join(sorted(glob.glob(os.path.join(MODEL_DIR, "**", "*.pth"),
                                        recursive=True)) or ["  (no .pth found)"])
    raise FileNotFoundError(
        f"Missing checkpoints for {_missing} under MODEL_DIR={MODEL_DIR}.\n"
        f".pth files present:\n{_tree}\n"
        f"Fix MODEL_DIR in Section 2 Config to your benchmarking-output slug.")

# ── per-model loader: reconstruct backbone from checkpoint metadata ────────
def disable_inplace(model):
    """Turn off every in-place activation (ReLU(inplace=True) etc.). VGG16-BN's
    in-place ReLUs modify views that GradCAM++'s full backward hook then tries
    to differentiate, which PyTorch forbids ('...view is being modified
    inplace...'). Non-inplace is numerically identical, just hook-safe."""
    n = 0
    for m in model.modules():
        if getattr(m, "inplace", False):
            m.inplace = False; n += 1
    return n

def load_model(spec):
    """Load one backbone onto DEVICE from its checkpoint. Returns
    (model, eval_model, in_features, backbone_name, ckpt_meta)."""
    path = find_ckpt(spec)
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    assert "model_state_dict" in ckpt, f"{path}: no model_state_dict"
    backbone_name = ckpt.get("backbone_name", spec["backbone"])
    ncls = ckpt.get("num_classes", num_classes)
    # FINAL: checkpoint metadata must match the benchmark contract exactly.
    assert backbone_name == spec["backbone"], f"{path}: backbone {backbone_name} != {spec['backbone']}"
    assert int(ncls) == NUM_CLASSES, f"{path}: num_classes {ncls}"
    if ckpt.get("class_names") is not None:
        assert list(ckpt["class_names"]) == class_names, f"{path}: class_names {ckpt['class_names']}"
    if ckpt.get("img_size") is not None:
        assert int(ckpt["img_size"]) == IMG_SIZE, f"{path}: img_size {ckpt['img_size']}"

    backbone = timm.create_model(backbone_name, pretrained=False,
                                 num_classes=0, global_pool="").to(DEVICE)
    with torch.no_grad():
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
        in_features = backbone(dummy).shape[1]
    del dummy
    if ckpt.get("in_features") is not None:
        assert int(ckpt["in_features"]) == int(in_features), \
            f"{path}: in_features {ckpt['in_features']} != rebuilt {in_features}"
    head  = BrainTumorHead(in_features, ncls, dropout=HEAD_DROPOUT).to(DEVICE)
    model = BrainTumorModel(backbone, head).to(DEVICE)
    unwrap(model).load_state_dict(ckpt["model_state_dict"], strict=True)
    disable_inplace(model)          # VGG16-BN uses inplace ReLU -> clashes with GradCAM++ hooks
    model.eval()
    eval_model = unwrap(model); eval_model.eval()
    meta = {k: ckpt.get(k) for k in
            ("backbone_name", "num_classes", "in_features", "img_size", "eval_choice",
             "best_f1", "best_acc", "best_auc", "best_mcc")}
    meta["ckpt_path"] = path
    del ckpt
    free_gpu()
    return model, eval_model, in_features, backbone_name, meta

# ── classification predictions for confusion/ROC/PR (npz reuse or 1 pass) ──
@torch.inference_mode()
def get_predictions(spec, eval_model):
    """Return (y_true, y_pred, y_prob[N,C]) for the classification figures.
    NOTE: when the benchmarking npz is attached these are the benchmark's
    REPORTED TTA x5 predictions (the source of truth for classification
    metrics); they are NOT the clean FP32 predictions that define the XAI
    target class. The latter are recorded per image in the per-image CSVs."""
    key = spec["key"]
    pnpz = os.path.join(MODEL_DIR, "models", f"{key}_preds.npz")
    qnpz = os.path.join(MODEL_DIR, "models", f"{key}_probs.npz")
    if not os.path.exists(pnpz):
        _h = glob.glob(os.path.join(MODEL_DIR, "**", f"{key}_preds.npz"), recursive=True)
        if _h: pnpz = _h[0]
    if not os.path.exists(qnpz):
        _h = glob.glob(os.path.join(MODEL_DIR, "**", f"{key}_probs.npz"), recursive=True)
        if _h: qnpz = _h[0]
    if os.path.exists(pnpz) and os.path.exists(qnpz):
        try:
            pd_ = np.load(pnpz); qd_ = np.load(qnpz)
            def _pick(npz, *names):
                for n in names:
                    if n in npz.files:
                        return npz[n]
                return npz[npz.files[0]]           # fall back to first array
            y_true = _pick(pd_, "true", "y_true", "labels", "targets")
            y_pred = _pick(pd_, "preds", "pred", "y_pred", "predictions")
            y_prob = _pick(qd_, "probs", "prob", "y_prob", "probabilities")
            if len(y_true) == len(y_pred) == len(y_prob):
                return np.asarray(y_true), np.asarray(y_pred), np.asarray(y_prob)
            log(f"  [preds] npz length mismatch for {key}; recomputing.")
        except Exception as e:
            log(f"  [preds] npz load failed for {key} ({e}); recomputing.")
    y_true, y_pred, y_prob = [], [], []
    for path, lbl in tqdm(list(zip(sample_paths, sample_labels)),
                          desc=f"{spec['display']} classify", leave=False):
        t = load_image_tensor(path).unsqueeze(0).to(DEVICE)
        p = torch.softmax(eval_model(t), dim=1)[0].cpu().numpy()
        y_prob.append(p); y_pred.append(int(p.argmax())); y_true.append(int(lbl))
        del t
    return np.array(y_true), np.array(y_pred), np.array(y_prob)

print("\nData + loader ready.")

[13:23:41] Test samples: 1600 | classes=['glioma', 'meningioma', 'notumor', 'pituitary']
[13:23:41]     glioma: 400
[13:23:41]     meningioma: 400
[13:23:41]     notumor: 400
[13:23:41]     pituitary: 400

Checkpoint discovery:
  VGG16-BN       -> /kaggle/input/notebooks/tamannajarintima/benchmarking/models/vgg16_best.pth
  DenseNet121    -> /kaggle/input/notebooks/tamannajarintima/benchmarking/models/densenet121_best.pth
  ResNet50       -> /kaggle/input/notebooks/tamannajarintima/benchmarking/models/resnet50_best.pth

Data + loader ready.


In [6]:
# ── §5b  Calibration split = benchmark validation split (never evaluated on) ──
from sklearn.model_selection import StratifiedShuffleSplit

if not os.path.exists(TRAIN_DIR):
    raise FileNotFoundError(f"TRAIN_DIR not found: {TRAIN_DIR} (needed for the calibration split)")

_train_folder = datasets.ImageFolder(TRAIN_DIR)
assert list(_train_folder.classes) == class_names, "Training/ class order differs from Testing/"
_train_labels = np.asarray(_train_folder.targets)
_sss = StratifiedShuffleSplit(n_splits=1, test_size=BENCH_VAL_FRAC, random_state=BENCH_RANDOM_SEED)
_train_idx, _val_idx = next(_sss.split(np.zeros(len(_train_labels)), _train_labels))
VAL_PATHS  = [_train_folder.samples[i][0] for i in _val_idx]
VAL_LABELS = _train_labels[_val_idx].astype(int)
VAL_IDS    = [os.path.join(os.path.basename(os.path.dirname(p)),
                           os.path.splitext(os.path.basename(p))[0]) for p in VAL_PATHS]
log(f"Benchmark split reconstructed: train={len(_train_idx)} val={len(_val_idx)} "
    f"(benchmark log: 4760 / 840)")
assert len(_train_idx) == 4760 and len(_val_idx) == 840, "split sizes differ from the benchmark"

# Verify against the benchmark's saved validation labels (order-sensitive).
VAL_SPLIT_VERIFIED = None
for _spec in MODELS:
    _p = find_artifact(f"{_spec['key']}_probe.npz")
    if _p:
        _vt = np.load(_p)["val_true"].astype(int)
        VAL_SPLIT_VERIFIED = bool(np.array_equal(_vt, VAL_LABELS))
        assert VAL_SPLIT_VERIFIED, (f"validation label sequence differs from {_p}: the calibration split "
                                    f"is not the benchmark's validation split")
        break
log(f"Validation split label sequence matches benchmark probe: {VAL_SPLIT_VERIFIED} "
    f"(None = probe npz not attached, could not verify)")

# Stratified calibration draw, positions refer to VAL_PATHS.
_rng_cal = np.random.default_rng(CALIB_SEED)
CALIB_POSITIONS = []
for _c in range(num_classes):
    _pool = np.flatnonzero(VAL_LABELS == _c)
    _want = min(len(_pool), CALIB_N // num_classes + (1 if _c < CALIB_N % num_classes else 0))
    CALIB_POSITIONS.extend(_rng_cal.choice(_pool, size=_want, replace=False).tolist())
CALIB_POSITIONS = sorted(CALIB_POSITIONS)
log(f"Calibration images: {len(CALIB_POSITIONS)} from the validation split "
    f"({ {class_names[c]: int((VAL_LABELS[CALIB_POSITIONS] == c).sum()) for c in range(num_classes)} })")

[13:23:50] Benchmark split reconstructed: train=4760 val=840 (benchmark log: 4760 / 840)
[13:23:50] Validation split label sequence matches benchmark probe: True (None = probe npz not attached, could not verify)
[13:23:50] Calibration images: 100 from the validation split ({'glioma': 25, 'meningioma': 25, 'notumor': 25, 'pituitary': 25})


## §6 Model architecture

Same `BrainTumorHead` / `BrainTumorModel` as the benchmark. The XAI copy omits `autocast`, so all
attributions are computed in FP32 (the benchmark's reported metrics were computed with AMP; §7 measures
the resulting prediction agreement). Note for VGG16-BN: timm's `vgg16_bn(num_classes=0, global_pool="")`
still applies `pre_logits` (a 7×7 conv), so the head receives a 4096×1×1 tensor (`in_features=4096`),
while the 512×7×7 spatial map lives at `backbone.features`.

In [7]:
# ── §6  Model architecture (verbatim from benchmarking.ipynb §4) ──────────
# Deliberate difference: no autocast inside BrainTumorModel.forward -> the XAI
# forward/backward passes run in FP32. Weights/architecture are identical.
class BrainTumorHead(nn.Module):
    def __init__(self, in_features, num_classes, dropout=HEAD_DROPOUT):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Linear(in_features * 2, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(256, num_classes)

    def forward(self, x):
        avg = self.avg_pool(x).flatten(1)
        mx  = self.max_pool(x).flatten(1)
        x = torch.cat([avg, mx], dim=1)
        x = F.gelu(self.bn1(self.fc1(x))); x = self.dropout(x)
        x = F.gelu(self.bn2(self.fc2(x))); x = self.dropout(x)
        return self.out(x)


class BrainTumorModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head
    def forward(self, x):
        return self.head(self.backbone(x))


def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model

print("Architecture ready.")

Architecture ready.


## §7 Checkpoint, preprocessing-parity and target-layer verification

Loads each backbone once and **fails loudly** if any of the following is not true:

1. The test-set label order equals the benchmark's `clean_true` (so image index *i* here is image *i* there).
2. Clean FP32 predictions with the §4 preprocessing agree with the benchmark's saved clean
   (no-TTA) predictions on ≥ `PARITY_FAIL_AGREEMENT` of test images (small disagreement is expected from
   AMP vs FP32 on borderline images; it is reported). The same check on the calibration images uses the
   benchmark's validation predictions.
3. The resolved Grad-CAM++ target module is the structurally expected one for the *instantiated* timm
   model (VGG16-BN: `features` whose last child `features[43]` is `MaxPool2d`, with `features[42]` a 14×14
   ReLU; DenseNet121: `features` ending in `norm5`; ResNet50: `layer4`), its output is 4-D, 7×7 and ≥ 99 %
   non-negative, and it is not the classifier input.

Verification only; writes `Tables/prediction_parity.csv` and `Tables/target_layer_verification.csv`.

In [8]:
# ── §7a  Target-layer resolution (used by §7 verification and every loop) ──
# ---- GradCAM++ target layer resolution (backbone-aware) --------------------
def get_target_layer(model, backbone_name: str):
    """Resolve the GradCAM++ target layer: the last SPATIAL, POST-ACTIVATION
    feature map of the backbone.

    WHY THE PREVIOUS VERSION WAS WRONG
    ----------------------------------
    It resolved, per backbone:

        vgg      ->  [m for m in bb.modules() if isinstance(m, Conv2d)][-1]
        densenet ->  bb.features.denseblock4.denselayer16.conv2
        resnet   ->  bb.layer4[-1].conv2
        fallback ->  the last Conv2d found anywhere in the tree

    Each of those is a different failure:

    * VGG16-BN. `convs[-1]` walks the whole module tree in registration
      order, so the last Conv2d is NOT the last conv of `features` -- it is
      `pre_logits.fc2`, a Conv2d(4096, 4096, 1x1) in the classifier head.
      Its activation map is 1x1 SPATIAL. A 1x1 CAM is constant, and the old
      `_normalize` mapped a constant array to exactly 0.0. Result: an
      identically-zero attribution map on 100% of images, which then flowed
      into AOPC (where argsort of zeros is a reverse-raster scan), Gini,
      Entropy, and the AGF weight vector -- where it still received w=0.293.

    * ResNet50. `layer4[-1].conv2` sits BEFORE bn2/act2 in timm's Bottleneck
      (conv2 -> bn2 -> act2), so its output is SIGNED. GradCAM++ derives
      alpha assuming A >= 0; with negative activations `spatial_sum` flips
      the denominator's sign, alpha flips, and np.maximum(cam, 0) zeroes the
      map on roughly 72-79% of images.

    * DenseNet121. `denseblock4.denselayer16.conv2` emits only the 32 NEW
      channels that the last dense layer contributes, not the 1024-channel
      concatenated block output. It does not collapse, but it is not the
      DenseNet feature map either.

    * The Conv2d fallback silently reproduced the VGG failure on any
      unrecognised backbone. It is removed: an unknown backbone now raises.

    Correct targets: bb.features (VGG, 512x7x7), bb.features (DenseNet,
    1024x7x7 via norm5), bb.layer4 (ResNet, 2048x7x7 post-residual).
    """
    bb = unwrap(model).backbone
    name = backbone_name.lower()
    # FINAL: the attribute path comes from CONFIG (TARGET_LAYERS) and is verified
    # structurally in §7. The branches below are the audited v7.1 logic and must
    # agree with CONFIG; a disagreement raises.
    if name in TARGET_LAYERS:
        _cfg_attr = TARGET_LAYERS[name]
        _cfg_mod = bb
        for _part in _cfg_attr.split("."):
            _cfg_mod = getattr(_cfg_mod, _part)
    else:
        raise ValueError(f"backbone '{backbone_name}' has no verified target layer in TARGET_LAYERS")

    if "vgg" in name:
        # timm vgg16_bn: features -> 512 x 7 x 7, post-ReLU/post-pool.
        # NOT convs[-1], which is pre_logits.fc2 (a 1x1 classifier conv).
        assert _cfg_mod is bb.features
        return bb.features

    if "densenet" in name:
        # timm densenet121: features ends in norm5 (BatchNormAct2d, i.e.
        # norm+act fused) -> 1024 x 7 x 7, non-negative.
        assert _cfg_mod is bb.features
        return bb.features

    if "resnet" in name:
        # timm resnet50: layer4 output -> 2048 x 7 x 7, post-residual,
        # post-ReLU. NOT layer4[-1].conv2, which is pre-BatchNorm.
        assert _cfg_mod is bb.layer4
        return bb.layer4

    raise ValueError(
        f"Unresolved target layer for backbone '{backbone_name}'. Add an "
        f"explicit branch to get_target_layer(); the old last-Conv2d "
        f"fallback silently resolved to classifier-head convs and produced "
        f"identically-zero CAMs."
    )


def assert_spatial_target(eval_model, target_layer, img_size=None, device=None,
                          min_hw: int = 7, min_nonneg_frac: float = 0.99,
                          label: str = ""):
    """Probe the resolved target layer once and HARD-FAIL if it cannot
    support GradCAM++. Call immediately after get_target_layer(), before any
    attribution is computed.

    Guards exactly the two failure modes above:
      (1) insufficient spatial extent  -> constant CAM  (the VGG bug)
      (2) signed activations           -> zeroed CAM    (the ResNet bug)

    This is the assertion whose absence let a broken hook produce a full set
    of publication tables and a p = 1.1e-20.
    """
    img_size = IMG_SIZE if img_size is None else img_size
    device   = DEVICE   if device   is None else device
    captured = {}

    def _probe_hook(module, inputs, output):
        if isinstance(output, (tuple, list)):
            output = output[0]
        o = output.detach()
        captured["shape"] = tuple(o.shape)
        captured["min"] = float(o.min())
        captured["nonneg_frac"] = float((o >= 0).float().mean())

    handle = target_layer.register_forward_hook(_probe_hook)
    was_training = eval_model.training
    try:
        eval_model.eval()
        g = torch.Generator(device="cpu").manual_seed(SEED)
        probe_x = torch.randn(1, 3, img_size, img_size, generator=g).to(device)
        with torch.no_grad():
            eval_model(probe_x)
    finally:
        handle.remove()
        if was_training:
            eval_model.train()
        del probe_x

    tag = f" [{label}]" if label else ""
    if "shape" not in captured:
        raise RuntimeError(f"Target layer{tag}: forward hook never fired -- the "
                           f"module is not on this model's forward path.")
    shape = captured["shape"]
    if len(shape) != 4:
        raise RuntimeError(f"Target layer{tag}: output shape {shape}; GradCAM++ "
                           f"requires a 4-D (N, C, H, W) feature map.")
    h, w = shape[-2], shape[-1]
    if h < min_hw or w < min_hw:
        raise RuntimeError(
            f"Target layer{tag}: output {shape} -> spatial extent {h}x{w}, below "
            f"the required {min_hw}x{min_hw}. This is a classifier-head conv, "
            f"not a feature map; the CAM would be constant.")
    if captured["nonneg_frac"] < min_nonneg_frac:
        raise RuntimeError(
            f"Target layer{tag}: only {captured['nonneg_frac']:.1%} of activations "
            f"are non-negative (min={captured['min']:.4f}). GradCAM++ assumes "
            f"A >= 0; hook a post-activation module, not a pre-BatchNorm conv.")
    log(f"[target-layer]{tag} {type(target_layer).__name__} -> {shape} | "
        f"non-negative {captured['nonneg_frac']:.1%} | OK")
    return shape

In [9]:
# ── §7  Verification: checkpoints, preprocessing parity, target layers ─────
@torch.inference_mode()
def _predict_tensors(eval_model, X, bs=32):
    probs = []
    for i in range(0, len(X), bs):
        xb = X[i:i + bs].to(DEVICE)
        probs.append(torch.softmax(eval_model(xb).float(), dim=1).cpu())
    return torch.cat(probs).numpy()

def _hook_shape(eval_model, module, x):
    cap = {}
    h = module.register_forward_hook(lambda m, i, o: cap.__setitem__("s", tuple(o.shape)))
    try:
        with torch.inference_mode():
            eval_model(x)
    finally:
        h.remove()
    return cap.get("s")

_parity_idx = (list(range(len(sample_paths))) if PARITY_CHECK_N is None
               else stratified_sample(PARITY_CHECK_N, SEED))
log(f"[verify] preprocessing {len(_parity_idx)} test + {len(CALIB_POSITIONS)} calibration images once...")
_X_TEST = torch.stack([load_image_tensor(sample_paths[i]) for i in tqdm(_parity_idx, leave=False)])
_X_CAL  = torch.stack([load_image_tensor(VAL_PATHS[p]) for p in tqdm(CALIB_POSITIONS, leave=False)])

PARITY_ROWS, LAYER_ROWS, CKPT_SHA = [], [], {}
for spec in MODELS:
    if spec["key"] not in RUN_MODELS:
        continue
    model, eval_model, in_features, backbone_name, meta = load_model(spec)
    CKPT_SHA[spec["key"]] = sha256_file(meta["ckpt_path"])
    bb = eval_model.backbone
    tl = get_target_layer(eval_model, backbone_name)
    x1 = _X_TEST[:1].to(DEVICE)

    # ---- structural target-layer checks on the instantiated model ----------
    extra = {}
    if backbone_name == "vgg16_bn":
        assert len(bb.features) == 44, f"vgg16_bn.features has {len(bb.features)} modules, expected 44"
        assert isinstance(bb.features[43], nn.MaxPool2d) and isinstance(bb.features[42], nn.ReLU)
        extra["features[42]_shape"] = str(_hook_shape(eval_model, bb.features[42], x1))
        extra["features[43]_shape"] = str(_hook_shape(eval_model, bb.features[43], x1))
        extra["pre_logits_out_shape"] = str(_hook_shape(eval_model, bb.pre_logits, x1))
    elif backbone_name == "densenet121":
        assert list(dict(bb.features.named_children()))[-1] == "norm5"
        extra["norm5_shape"] = str(_hook_shape(eval_model, bb.features.norm5, x1))
    elif backbone_name == "resnet50":
        extra["layer4[-1]_shape"] = str(_hook_shape(eval_model, bb.layer4[-1], x1))
    tl_shape = assert_spatial_target(eval_model, tl, IMG_SIZE, DEVICE, label=spec["display"])
    assert tuple(tl_shape[-2:]) == TARGET_EXPECTED_HW, f"target layer spatial size {tl_shape}"
    head_in = _hook_shape(eval_model, bb, x1)
    LAYER_ROWS.append({"Backbone": spec["display"], "backbone_name": backbone_name,
                       "target_attr": f"backbone.{TARGET_LAYERS[backbone_name]}",
                       "target_module": type(tl).__name__, "target_output_shape": str(tl_shape),
                       "backbone_output_shape(head_input)": str(head_in),
                       "in_features": int(in_features), **extra})
    del x1

    # ---- prediction parity vs the benchmark's saved clean predictions ------
    probe_p = find_artifact(f"{spec['key']}_probe.npz")
    row = {"Backbone": spec["display"], "n_test": len(_parity_idx), "probe_npz": probe_p}
    prob = _predict_tensors(eval_model, _X_TEST)
    pred = prob.argmax(1)
    row["xai_fp32_accuracy_on_checked"] = float((pred == np.asarray(sample_labels)[_parity_idx]).mean())
    if probe_p:
        pz = np.load(probe_p)
        ct = pz["clean_true"].astype(int); cp = pz["clean_pred"].astype(int); cq = pz["clean_prob"]
        assert np.array_equal(ct, np.asarray(sample_labels)), \
            f"{spec['display']}: test label ORDER differs from the benchmark probe"
        row["label_order_identical"] = True
        row["pred_agreement_vs_bench_clean"] = float((pred == cp[_parity_idx]).mean())
        row["n_pred_disagree"] = int((pred != cp[_parity_idx]).sum())
        row["max_abs_prob_diff"] = float(np.abs(prob - cq[_parity_idx]).max())
        row["mean_abs_prob_diff"] = float(np.abs(prob - cq[_parity_idx]).mean())
        cal_prob = _predict_tensors(eval_model, _X_CAL)
        row["calib_pred_agreement_vs_bench_val"] = float(
            (cal_prob.argmax(1) == pz["val_pred"].astype(int)[CALIB_POSITIONS]).mean())
        agr = row["pred_agreement_vs_bench_clean"]
        if agr < PARITY_FAIL_AGREEMENT:
            raise RuntimeError(f"{spec['display']}: prediction agreement {agr:.4f} < "
                               f"{PARITY_FAIL_AGREEMENT}: preprocessing/weights are NOT the benchmark's")
        if agr < PARITY_WARN_AGREEMENT:
            log(f"[verify][WARN] {spec['display']}: agreement {agr:.4f} below {PARITY_WARN_AGREEMENT}")
    else:
        row["label_order_identical"] = None
        log(f"[verify][WARN] {spec['display']}: {spec['key']}_probe.npz not attached -- parity NOT verified")
    PARITY_ROWS.append(row)
    log(f"[verify] {spec['display']}: " + ", ".join(f"{k}={v}" for k, v in row.items() if k != "probe_npz"))
    del model, eval_model, tl, bb
    free_gpu()

del _X_TEST, _X_CAL
PARITY_DF = pd.DataFrame(PARITY_ROWS); LAYER_DF = pd.DataFrame(LAYER_ROWS)
save_table(PARITY_DF, "prediction_parity"); save_table(LAYER_DF, "target_layer_verification")
PARITY_DF.to_csv(os.path.join(DIRS["Statistics"], "prediction_parity.csv"), index=False)
LAYER_DF.to_csv(os.path.join(DIRS["Statistics"], "target_layer_verification.csv"), index=False)
print(PARITY_DF.to_string(index=False)); print(LAYER_DF.to_string(index=False))
mark_section("07_verification")

[13:23:50] [verify] preprocessing 1600 test + 100 calibration images once...


  0%|          | 0/1600 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

[13:24:22] [target-layer] [VGG16-BN] Sequential -> (1, 512, 7, 7) | non-negative 100.0% | OK
[13:24:31] [verify] VGG16-BN: Backbone=VGG16-BN, n_test=1600, xai_fp32_accuracy_on_checked=0.9525, label_order_identical=True, pred_agreement_vs_bench_clean=0.999375, n_pred_disagree=1, max_abs_prob_diff=0.005978494882583618, mean_abs_prob_diff=6.1008991906419396e-05, calib_pred_agreement_vs_bench_val=1.0
[13:24:32] [target-layer] [DenseNet121] Sequential -> (1, 1024, 7, 7) | non-negative 100.0% | OK
[13:24:38] [verify] DenseNet121: Backbone=DenseNet121, n_test=1600, xai_fp32_accuracy_on_checked=0.953125, label_order_identical=True, pred_agreement_vs_bench_clean=0.999375, n_pred_disagree=1, max_abs_prob_diff=0.010363847017288208, mean_abs_prob_diff=0.00034023457556031644, calib_pred_agreement_vs_bench_val=1.0
[13:24:40] [target-layer] [ResNet50] Sequential -> (1, 2048, 7, 7) | non-negative 100.0% | OK
[13:24:45] [verify] ResNet50: Backbone=ResNet50, n_test=1600, xai_fp32_accuracy_on_checked=0.8

## §8–9 XAI method and AGF implementation — `xai_attribution.py`

Summary of the implementation:

* **Grad-CAM++** — forward hook caches activations and a full backward hook caches ∂y_c/∂A of the target
  module; `y_c` is the **logit** of the fixed class; α = g²/(2g² + ΣA·g³) (denominator floored at 1e-8),
  channel weights Σ α·ReLU(g), CAM = ReLU(Σ_c w_c A_c) at 7×7, min–max normalised. A constant CAM raises
  (never silently zero). Hooks are removed in `finally` blocks in every loop.
* **SmoothGrad** — 20 noisy copies, σ = 0.15·(max−min) of the normalised input, noise drawn on CPU from a
  generator seeded by a hash of the image bytes (same noise for the same image in every arm); saliency =
  mean over samples of |∇x y_c|, averaged over RGB channels.
* **Integrated Gradients** — zero baseline in normalised space, 30 steps, trapezoidal weights, attribution
  = (x − x′)·avg-grad, then |·| averaged over channels.
* `model.eval()` is called in every `generate`; no autocast anywhere → FP32 gradients; `disable_inplace`
  removes in-place ReLUs (needed for VGG16-BN hooks; numerically identical).
* **AGF** — `fuse_maps` clips each resized map at 0, computes pairwise kernel similarities, per-method mean
  agreement, `softmax(a/τ)` with max-subtraction for stability, weighted sum, clip, divide by max,
  `nan_to_num`. Weights sum to 1 by construction (asserted in §10); a constant map scores Dice 0 and cannot
  produce NaN. With three methods and Dice ∈ [0, 1], the softmax at τ = 1 bounds every weight to
  [e⁻¹/(e⁻¹+2), 1/(1+2e⁻¹)] ≈ [0.155, 0.576], so weights cannot become degenerate (one-hot).

**Kernel decision.** `topk_dice` is retained. It is the kernel in the configuration and was chosen by a data-free
synthetic control (§10) rather than by test performance. On real data it is *not* the best-scoring
kernel (§16c); that is reported, not acted upon.

**Known limitations:** IG baseline (ImageNet-mean colour)
≠ insertion/deletion baseline (neutral grey); the Dice quantile threshold admits >10 % of pixels when a map
has many ties at the threshold (e.g. clipped Grad-CAM++ zeros); fused maps are max-scaled while base maps
are min–max scaled, which leaves insertion/deletion rankings unaffected but can shift relative-L2
sensitivity between fused and individual maps (not between AGF and Equal Fusion, which share it).

In [10]:
%%writefile /kaggle/working/xai_attribution.py
"""
xai_attribution.py  — final-1.0 (algorithmically identical to v7.1)
===================================================================
FINAL-VERSION NOTE: no numerical behaviour of any class in this module was
changed relative to the v7.1 run. Only docstrings that contradicted the code
were corrected (the default kernel, and the definition of max-sensitivity).
Brain Tumor MRI Explainability — pruned XAI core.

Provides:
  BaseAttributor, resize_map, _safe_entropy
  GradCAMPlusPlus, SmoothGrad, IntegratedGradients
  SaliencyMetrics (.sensitivity, .gini)
  AgreementGuidedFusion    (AGF — the proposed method)

v7 change (the substantive one):
  0. The agreement kernel is now PLUGGABLE. [CORRECTED DOCSTRING: the default,
     the notebook CONFIG and the executed v7.1 run all use "topk_dice"; an
     earlier draft of this note said "pearson".] The v6 kernel -- 0.5*(SSIM + raw cosine) on
     min-max-normalised NON-NEGATIVE maps -- is bounded well above zero for
     any two maps sharing a coarse spatial prior. Observed score spread was
     ~0.02, so softmax at tau=1 produced a weight spread of ~0.007 and the
     "agreement-guided" fusion was an arithmetic mean to within 2%. That was
     a property of the kernel, not of brain-tumour MRI. "pearson" (centring)
     and "topk_dice" (thresholding) were both implemented; on the synthetic
     shared-prior control only topk_dice produced usable dynamic range, so it
     was fixed as the kernel BEFORE any test-set evaluation. "ssim_cosine" and
     "pearson" are retained as ablation arms.

v6 changes (correctness, not cosmetics):
  1. BaseAttributor._normalize now RAISES on a degenerate (constant) map
     instead of silently returning all zeros. A constant map is never a
     valid attribution: it means the target layer, class index or gradient
     path is broken. The old silent-zero path let a broken GradCAM++ hook
     propagate through AOPC, Gini, entropy and the fusion weights as if it
     were a real explanation.
  2. SmoothGrad gained freeze_noise(): the max-sensitivity metric compares
     phi(x) against phi(x+delta), and with seed_mode="content" those two
     calls hashed to different seeds and drew independent noise sets. That
     made the robustness column measure Monte-Carlo estimator variance on
     top of input sensitivity for the stochastic methods only, while
     GradCAM++ and IG were deterministic. Freezing the draw across the pair
     makes the comparison apples-to-apples.
  3. SaliencyMetrics.sensitivity returns NaN when every perturbation sample
     is non-finite. It previously returned max_rel_diff == 0.0, i.e. it
     scored a total failure as PERFECT robustness.
  4. AgreementGuidedFusion.fuse_maps() fuses PRE-COMPUTED maps, so the
     evaluation loop no longer computes every base method twice (once
     standalone for its own row, once again inside fuse()). It also
     guarantees the standalone maps and the fusion constituents are the
     identical arrays, which the old path did not: fuse() clipped its
     constituents after resize_map and the standalone arm did not, so Gini
     and Sparsity were computed on subtly different data.

REMOVED in v6: two earlier fusion variants that were retired from the
experiment but still occupied ~250 lines of unreachable code. Recover them
from version control if they are ever re-enabled.
"""

from __future__ import annotations
import hashlib
import numpy as np
import torch
from abc import ABC, abstractmethod
from typing import Dict, Optional, Tuple, List
import cv2

# SSIM is needed by the ssim_cosine ablation kernel. skimage ships on Kaggle,
# but we fall back to a self-contained windowed-SSIM implementation (scipy
# uniform_filter, standard SSIM formula) so this module still works if
# scikit-image is ever missing -- no silent behaviour change either way.
try:
    from skimage.metrics import structural_similarity as _skimage_ssim
    _HAS_SKIMAGE_SSIM = True
except ImportError:
    _HAS_SKIMAGE_SSIM = False
    from scipy.ndimage import uniform_filter

    def _skimage_ssim(a: np.ndarray, b: np.ndarray, data_range: float = 1.0,
                       win_size: int = 7, **_ignored) -> float:
        """Fallback single-scale SSIM (standard formula, uniform window).
        Only used if scikit-image is unavailable. a, b assumed 2-D float32
        in [0, data_range]."""
        C1, C2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
        a, b = a.astype(np.float64), b.astype(np.float64)
        mu_a, mu_b = uniform_filter(a, win_size), uniform_filter(b, win_size)
        mu_a2, mu_b2, mu_ab = mu_a * mu_a, mu_b * mu_b, mu_a * mu_b
        var_a  = uniform_filter(a * a, win_size) - mu_a2
        var_b  = uniform_filter(b * b, win_size) - mu_b2
        cov_ab = uniform_filter(a * b, win_size) - mu_ab
        num = (2 * mu_ab + C1) * (2 * cov_ab + C2)
        den = (mu_a2 + mu_b2 + C1) * (var_a + var_b + C2)
        return float(np.mean(num / (den + 1e-12)))


# ── Utilities ─────────────────────────────────────────────────────────────

def resize_map(attr_map: np.ndarray, size: Tuple[int, int],
               anti_alias: bool = True) -> np.ndarray:
    h, w = size
    interp = cv2.INTER_LANCZOS4 if anti_alias else cv2.INTER_LINEAR
    return cv2.resize(attr_map.astype(np.float32), (w, h), interpolation=interp)


def _safe_entropy(arr: np.ndarray) -> float:
    if float(arr.max()) < 1e-6:
        return 0.0
    p = arr / (arr.sum() + 1e-8)
    return float(-np.sum(p * np.log(p + 1e-8)))


# ── Abstract base ─────────────────────────────────────────────────────────

class BaseAttributor(ABC):
    def __init__(self, model: torch.nn.Module):
        self.model  = model
        self.device = next(model.parameters()).device

    @abstractmethod
    def generate(self, tensor: torch.Tensor,
                 class_idx: Optional[int] = None) -> Tuple[np.ndarray, int]:
        ...

    def __call__(self, tensor: torch.Tensor,
                 class_idx: Optional[int] = None) -> Tuple[np.ndarray, int]:
        return self.generate(tensor, class_idx)

    @staticmethod
    def _normalize(arr: np.ndarray, *, context: str = "") -> np.ndarray:
        """Min-max normalise an attribution map to [0, 1].

        FAILS LOUDLY on degenerate (constant) input. The previous version
        was `arr -= arr.min(); return arr / (arr.max() + 1e-8)`, which maps
        a constant array to EXACTLY zero and returns it without a warning.
        Every downstream consumer -- aopc_curve, gini, entropy, the
        agreement scores, the weighted fusion sum -- then operates on an
        all-zero map as if it were a real explanation. That single silent
        path is how a mis-resolved GradCAM++ target layer (a 1x1 classifier
        conv) produced a full set of publication tables and a p = 1e-20.

        A constant attribution map is never legitimate. It means one of:
          * the target layer has no spatial extent (classifier head conv),
          * the target layer emits signed activations and GradCAM++'s
            np.maximum(cam, 0) zeroed everything,
          * the class index is wrong, or gradients never reached the hook.
        All of those are bugs, so this raises and lets the caller's
        try/except log a SKIP rather than fabricating a data point.

        `context` is a free-text tag (e.g. "gradcam_pp/vgg16_bn") echoed in
        the error message. It is keyword-only with a default, so existing
        `self._normalize(x)` call sites keep working unchanged.
        """
        arr = np.asarray(arr, dtype=np.float64)
        tag = f" [{context}]" if context else ""
        if not np.isfinite(arr).all():
            n_bad = int(np.count_nonzero(~np.isfinite(arr)))
            raise ValueError(
                f"Non-finite attribution map{tag}: {n_bad} of {arr.size} "
                f"entries are NaN or Inf."
            )
        lo, hi = float(arr.min()), float(arr.max())
        if hi - lo < 1e-12:
            raise ValueError(
                f"Degenerate attribution map{tag}: constant value {lo:.6g} "
                f"(dynamic range {hi - lo:.3g}). Check the target layer "
                f"(spatial extent AND non-negativity), the class index, and "
                f"that gradients reach the hooked module."
            )
        return (arr - lo) / (hi - lo)

    def _resolve_class(self, inp: torch.Tensor,
                       class_idx: Optional[int]) -> int:
        if class_idx is not None:
            return class_idx
        with torch.no_grad():
            return int(self.model(inp).argmax(dim=1).item())


# ── GradCAM++ ─────────────────────────────────────────────────────────────

class GradCAMPlusPlus(BaseAttributor):
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        super().__init__(model)
        self._acts = self._grads = None
        self._fwd_hook = target_layer.register_forward_hook(self._cache_acts)
        self._bwd_hook = target_layer.register_full_backward_hook(self._cache_grads)

    def _cache_acts(self, _, __, output):
        self._acts = output.detach()

    def _cache_grads(self, _, __, grad_output):
        self._grads = grad_output[0].detach()

    def __enter__(self): return self
    def __exit__(self, *_): self.remove()

    def remove(self):
        self._fwd_hook.remove()
        self._bwd_hook.remove()

    def generate(self, tensor: torch.Tensor,
                 class_idx: Optional[int] = None) -> Tuple[np.ndarray, int]:
        self.model.eval()
        inp       = tensor.unsqueeze(0).to(self.device).requires_grad_(True)
        out       = self.model(inp)
        class_idx = self._resolve_class(inp, class_idx)
        self.model.zero_grad()
        out[0, class_idx].backward()

        grads = self._grads[0].cpu().numpy()
        acts  = self._acts[0].cpu().numpy()
        g2, g3     = grads ** 2, grads ** 3
        spatial_sum = acts.sum(axis=(1, 2), keepdims=True)
        denom  = 2.0 * g2 + spatial_sum * g3
        denom  = np.where(np.abs(denom) > 1e-8, denom, 1e-8)
        alpha  = g2 / denom
        weights = (alpha * np.maximum(grads, 0)).sum(axis=(1, 2))
        cam = np.einsum("c,chw->hw", weights, acts)
        cam = np.maximum(cam, 0)
        return self._normalize(cam, context="gradcam_pp"), class_idx


# ── SmoothGrad ────────────────────────────────────────────────────────────

class SmoothGrad(BaseAttributor):
    """SmoothGrad with reproducible, arm-independent noise.

    seed_mode="content": the generator is reseeded per call from a hash of
    the input tensor's bytes. Consequences that matter for paired testing:
      * the same image draws the same noise whether SmoothGrad is called
        standalone or from inside AGF, so the AGF-vs-SmoothGrad comparison
        is not confounded by a noise mismatch;
      * different images draw different noise, so the fusion arms are not
        reusing one fixed noise set for all N images;
      * results do not depend on call order, so RUN_MODELS staging and
        re-runs reproduce exactly.
    seed_mode="stream": legacy single advancing generator.

    noise_mode="relative": sigma = noise_frac * (input max - min), i.e. the
    fraction-of-range convention SmoothGrad actually specifies.
    noise_mode="absolute": sigma = noise_std, interpreted in whatever space
    the input already lives in (here: ImageNet-normalised).

    NOISE FREEZING (v6) -- required for an honest max-sensitivity metric
    --------------------------------------------------------------------
    Max-sensitivity evaluates  ||phi(x + delta) - phi(x)|| / ||phi(x)||.
    Under seed_mode="content", x and x+delta hash to DIFFERENT seeds, so the
    two calls drew entirely independent noise sets. The measured quantity
    was therefore

        input sensitivity  +  Monte-Carlo estimator variance

    for SmoothGrad and for any fusion containing it, while GradCAM++ and IG
    are deterministic and contributed only the first term. Methods were
    being compared on different measurands.

    freeze_noise(True) caches the noise draw from the next generate() call
    and reuses it for every subsequent call until freeze_noise(False). Wrap
    the (base, perturbed) pair in a freeze and the estimator variance
    cancels, leaving input sensitivity alone. The cache is keyed on tensor
    SHAPE only -- freezing across differently-shaped inputs is a usage
    error and raises rather than silently re-drawing.
    """

    def __init__(self, model: torch.nn.Module, n_samples: int = 50,
                 noise_std: float = 0.10, squared: bool = False,
                 seed: Optional[int] = None,
                 noise_mode: str = "absolute", noise_frac: float = 0.15,
                 seed_mode: str = "content"):
        super().__init__(model)
        self.n_samples  = n_samples
        self.noise_std  = noise_std
        self.squared    = squared
        self.noise_mode = noise_mode
        self.noise_frac = noise_frac
        self.seed_mode  = seed_mode
        self._base_seed = 0 if seed is None else int(seed)
        self._rng = torch.Generator(device="cpu")
        if seed is not None:
            self._rng.manual_seed(self._base_seed)
        # --- noise-freezing state (v6) ---
        self._freeze          = False
        self._frozen_noise    = None      # list[Tensor] on CPU, len == n_samples
        self._frozen_shape    = None
        self._frozen_sigma    = None

    def freeze_noise(self, enable: bool = True) -> "SmoothGrad":
        """Enable/disable noise freezing. Enabling CLEARS any existing cache,
        so the next generate() call defines the frozen draw. Returns self so
        it can be chained: sg.freeze_noise(True).generate(x, c)."""
        self._freeze = bool(enable)
        self._frozen_noise = None
        self._frozen_shape = None
        self._frozen_sigma = None
        return self

    def _content_seed(self, inp: torch.Tensor) -> int:
        """Stable 63-bit seed from the input bytes, mixed with the base seed."""
        buf = inp.detach().to("cpu", torch.float32).contiguous().numpy().tobytes()
        h   = hashlib.blake2b(buf, digest_size=8,
                              person=b"smoothgr").digest()
        return (int.from_bytes(h, "big") ^ (self._base_seed * 0x9E3779B97F4A7C15)) \
               % ((1 << 63) - 1)

    def _draw_noise(self, inp: torch.Tensor, sigma: float) -> List[torch.Tensor]:
        """Return the n_samples noise tensors for this call, honouring the
        freeze cache. Noise is generated on CPU with self._rng (so it is
        reproducible independently of the CUDA RNG) and moved to device."""
        shape = tuple(inp.shape)
        if self._freeze and self._frozen_noise is not None:
            if self._frozen_shape != shape:
                raise ValueError(
                    f"SmoothGrad noise is frozen for shape {self._frozen_shape} "
                    f"but was called with {shape}. Call freeze_noise(True) again "
                    f"to re-arm, or freeze_noise(False) to disable."
                )
            # Reuse the FROZEN sigma as well as the frozen draw. Under
            # noise_mode="relative" sigma is derived from the input's own
            # range, which shifts slightly under perturbation; holding both
            # the draw and its scale fixed is what makes phi(x) and
            # phi(x+delta) differ only by delta.
            return [n.to(inp.device) * self._frozen_sigma
                    for n in self._frozen_noise]
        # Unit-variance draws are cached; the sigma scaling is applied on
        # the way out so the cache stays independent of the noise mode.
        draws = [torch.randn(shape, generator=self._rng)
                 for _ in range(self.n_samples)]
        if self._freeze:
            self._frozen_noise = [d.clone() for d in draws]
            self._frozen_shape = shape
            self._frozen_sigma = float(sigma)
        return [d.to(inp.device) * sigma for d in draws]

    def generate(self, tensor: torch.Tensor,
                 class_idx: Optional[int] = None) -> Tuple[np.ndarray, int]:
        self.model.eval()
        inp       = tensor.unsqueeze(0).to(self.device)
        class_idx = self._resolve_class(inp, class_idx)
        grad_acc  = torch.zeros_like(inp)
        # Reseeding is skipped while frozen: the cached draw already fixes
        # the noise, and reseeding from the PERTURBED input's bytes is
        # exactly the behaviour freezing exists to suppress.
        if self.seed_mode == "content" and not (self._freeze and
                                                self._frozen_noise is not None):
            self._rng.manual_seed(self._content_seed(inp))
        if self.noise_mode == "relative":
            span  = float(inp.max().item() - inp.min().item())
            sigma = self.noise_frac * span
        else:
            sigma = self.noise_std
        for noise in self._draw_noise(inp, sigma):
            noisy = (inp + noise).requires_grad_(True)
            out   = self.model(noisy)
            self.model.zero_grad()
            out[0, class_idx].backward()
            if noisy.grad is not None:
                g = noisy.grad.detach()
                grad_acc += (g ** 2) if self.squared else g.abs()
            del noise, noisy, out
        saliency = grad_acc[0].mean(dim=0).cpu().numpy()
        return self._normalize(saliency, context="smoothgrad"), class_idx

# ── Integrated Gradients ──────────────────────────────────────────────────

class IntegratedGradients(BaseAttributor):
    def __init__(self, model: torch.nn.Module, n_steps: int = 50):
        super().__init__(model)
        self.n_steps = n_steps

    def generate(self, tensor: torch.Tensor,
                 class_idx: Optional[int] = None,
                 baseline: Optional[torch.Tensor] = None) -> Tuple[np.ndarray, int]:
        self.model.eval()
        inp      = tensor.unsqueeze(0).to(self.device)
        baseline = (torch.zeros_like(inp) if baseline is None
                    else baseline.to(self.device))
        class_idx = self._resolve_class(inp, class_idx)
        grad_sum  = torch.zeros_like(inp)
        for k in range(self.n_steps + 1):
            alpha  = k / self.n_steps
            interp = (baseline + alpha * (inp - baseline)).requires_grad_(True)
            out    = self.model(interp)
            self.model.zero_grad()
            out[0, class_idx].backward()
            if interp.grad is not None:
                w = 0.5 if k in (0, self.n_steps) else 1.0
                grad_sum += w * interp.grad.detach()
        ig     = (inp - baseline) * grad_sum / self.n_steps
        ig_map = ig[0].abs().mean(dim=0).cpu().numpy()
        return self._normalize(ig_map, context="ig"), class_idx


# ── SaliencyMetrics ───────────────────────────────────────────────────────

class SaliencyMetrics:
    def __init__(self, model: torch.nn.Module):
        self.model  = model
        self.device = next(model.parameters()).device

    # NOTE (v08): the old SaliencyMetrics.aopc() method was removed. It was
    # dead code (never called; the evaluation loop uses _aopc_curve in section 4,
    # which correctly uses a grey baseline + softmax + insertion/deletion).
    # The old method masked with raw logits and no grey baseline, so keeping it
    # risked being mistaken for the metric actually reported in Table 1.

    def sensitivity(self, tensor: torch.Tensor, attributor,
                    class_idx: int, radius: float = 0.02,
                    n_samples: int = 20, seed: Optional[int] = None,
                    freeze_stochastic: bool = True) -> float:
        """Relative max-sensitivity, adapted from Yeh et al. 2019.

            Sens_rel = max_{s=1..S} ||phi(x+d_s) - phi(x)||_2 / (||phi(x)||_2 + eps),
            d_s ~ N(0, radius^2 I) i.i.d. per element in NORMALISED input space.

        [CORRECTED DOCSTRING] This is a Monte-Carlo maximum over S Gaussian
        perturbations with per-element std `radius`; it is NOT a supremum over
        an L2 ball of radius r (Yeh et al. sample uniformly in an L-inf ball).
        phi(.) is whatever the attributor returns; the notebook wraps every
        method so that phi is the post-processed 224x224 map (min-max ->
        Lanczos resize -> clip >= 0), identical to what the faithfulness
        metrics consume. The class index is fixed by the caller.

        Dividing by the base-map norm makes the score comparable across
        methods whose maps live on different numeric scales; the raw L2
        version ranked methods by MAP MAGNITUDE, not by robustness.

        v6 FIX 1 -- all-NaN case. The loop below only updates max_rel_diff
        on finite samples, and max_rel_diff starts at 0.0. If EVERY sample
        was non-finite the function returned 0.0, i.e. it reported PERFECT
        robustness -- the best possible score -- for a method that failed
        completely on that image. It now returns NaN so the image is
        excluded from the paired comparison instead of silently winning it.

        v6 FIX 2 -- stochastic/deterministic asymmetry. If `attributor`
        exposes freeze_noise() (SmoothGrad, or a fusion that forwards it),
        its noise draw is frozen across the whole (base, perturbed) family.
        Without this, phi(x) and phi(x+d) drew independent SmoothGrad noise
        and the metric measured input sensitivity PLUS Monte-Carlo
        estimator variance -- but only for the stochastic methods, while
        GradCAM++ and IG contributed input sensitivity alone. Set
        freeze_stochastic=False to reproduce the pre-v6 (confounded)
        numbers.

        NOTE: absolute values are not comparable to any pre-v08 run.
        """
        if seed is not None:
            torch.manual_seed(seed)

        # --- freeze stochastic estimators across the whole pair family ----
        _froze = False
        if freeze_stochastic and hasattr(attributor, "freeze_noise"):
            try:
                attributor.freeze_noise(True)
                _froze = True
            except Exception:
                _froze = False
        try:
            base_attr, _ = attributor(tensor, class_idx)
            base_l2      = float(np.linalg.norm(base_attr))
            # A near-zero / degenerate base map drives the relative
            # denominator toward 0 and the ratio explodes (observed pre-fix:
            # ResNet50 GradCAM++ ~8.7e7). Such a map carries no meaningful
            # stability to measure. NOTE: after the v6 target-layer fix this
            # branch should essentially never fire -- if it does, the target
            # layer is still wrong, so it is logged rather than swallowed.
            if base_l2 < 1e-6:
                print("  [WARN] sensitivity(): degenerate base map "
                      f"(L2={base_l2:.3g}) -- image excluded. Check "
                      "get_target_layer / assert_spatial_target.")
                return float("nan")
            base_norm    = base_l2 + 1e-8   # scale reference
            max_rel_diff = 0.0
            n_nan        = 0
            for _ in range(n_samples):
                noise   = torch.randn_like(tensor) * radius
                pert, _ = attributor(tensor + noise, class_idx)
                diff    = float(np.linalg.norm(pert - base_attr)) / base_norm
                # NaN comparisons are always False in Python, so a NaN diff
                # used to silently freeze the max at 0.0 forever.
                if np.isnan(diff):
                    n_nan += 1
                    continue
                if diff > max_rel_diff:
                    max_rel_diff = diff
                del noise, pert
        finally:
            if _froze:
                try:    attributor.freeze_noise(False)
                except Exception: pass

        if n_nan >= n_samples:
            # Every sample failed: returning max_rel_diff (== 0.0) here would
            # score total failure as perfect robustness. Return NaN and let
            # the nan-aware aggregate exclude the image.
            print(f"  [WARN] sensitivity(): all {n_samples} perturbation "
                  "samples were non-finite -- returning NaN (image excluded).")
            return float("nan")
        if n_nan > 0:
            print(f"  [WARN] sensitivity(): {n_nan}/{n_samples} NaN diffs skipped; "
                  f"max taken over the remaining {n_samples - n_nan}.")
        return float(max_rel_diff)

    @staticmethod
    def gini(attr_map: np.ndarray) -> float:
        arr = np.abs(attr_map).flatten().astype(np.float64)
        if arr.sum() < 1e-6:
            return 0.0
        arr = np.sort(arr)
        n   = len(arr)
        idx = np.arange(1, n + 1, dtype=np.float64)
        num = 2.0 * float((idx * arr).sum())
        den = float(n * arr.sum()) + 1e-8
        g   = num / den - (n + 1.0) / n
        return float(max(0.0, g))


# ── ★ AGF — Agreement-Guided Fusion (proposed method) ──────────────────────

# Defaults; Section 2 CONFIG overrides these on the imported module so the
# kernel is set in exactly one place.
_AGF_DEFAULT_KERNEL = "topk_dice"
_AGF_DEFAULT_TAU    = 1.0
_AGF_DEFAULT_TOPK   = 0.10


class AgreementGuidedFusion:
    """
    ★ AGF — Agreement-Guided Fusion.

    Mechanism
    ---------
    For each image, score how much each base method's attribution map agrees
    with the other two, then set per-image fusion weights from those scores.
    A method that agrees with the consensus is trusted more; a method that
    diverges is down-weighted. Weights are recomputed for every image.

    Mathematical definition
    ------------------------
    Let M_1, M_2, M_3 be the resized, non-negative maps for one image and let
    Sim(.,.) be the agreement kernel (below).

    Per-method agreement (mean similarity with the OTHER K-1 methods):
        a_k = (1 / (K-1)) * sum_{j != k} Sim(M_k, M_j)          (K = 3)

    Weights (softmax at temperature tau, tau = 1 in the reported runs):
        w_k = exp(a_k / tau) / sum_l exp(a_l / tau)

    Fused map (weighted arithmetic mean, then re-normalised to [0, 1]):
        AGF(i,j) = sum_k  w_k * M_k(i,j)

    Agreement kernels
    -----------------
    The kernel controls whether the weights can separate at all, so it is a
    first-class, ablatable choice (`agreement_kernel`):

    "topk_dice"  (DEFAULT, proposed)
        Binarise each map at its OWN top-q quantile (q = `topk_frac`) and take
        the Dice coefficient of the two masks, in [0, 1]. Thresholding is the
        operative step: it discards the shared low-level bulk and compares
        only WHERE each method concentrates its evidence. One parameter (q),
        and q is the same quantity a reader already interprets when they look
        at a thresholded saliency overlay.

    "pearson"
        Sim = mean-centred cosine, i.e. the Pearson correlation between the
        two maps, in [-1, 1]. Parameter-free, and it removes the constant
        offset that inflates raw cosine -- but on a synthetic control where
        all three maps carry the SAME coarse spatial prior and differ only in
        fine structure, it scored a spread of 0.022 against ssim_cosine's
        0.023. Centring removes a DC *offset*, not a shared spatial
        *component*, so it does NOT on its own cure the floor effect. Kept as
        an ablation arm because that negative result is worth reporting.

    "ssim_cosine"  (v6 legacy — the ablation control)
        Sim = 0.5 * [SSIM + raw cosine] on min-max-normalised non-negative
        maps. Both terms are bounded well above zero for any two maps sharing
        a coarse spatial prior; the observed score spread on real data was
        ~0.02, and softmax at tau=1 turned that into a weight spread of
        ~0.007, i.e. the fusion was an arithmetic mean to within 2%.

    On the synthetic shared-prior control (three maps, one fine-structure
    outlier) the agreement-score spreads were:
        ssim_cosine 0.023 | pearson 0.022 | topk_dice 0.110
    All three rank the outlier last; they differ in how much dynamic range
    they give the softmax. Section 18 repeats this on real attributions.

    Note that near-uniform weights are NOT automatically a defect: when the
    three base methods genuinely agree, uniform weights are the correct
    answer, and topk_dice returns them (weight std < 0.01 on the all-agree
    control). What was wrong in v6 is that the kernel returned near-uniform
    weights whether the methods agreed or not. Do not claim agreement-guided
    weighting without reporting the weight-spread diagnostic for the kernel
    actually used; Section 18 is that test.

    Returns
    -------
    fuse() returns a 5-tuple:
        (fused_map, resolved_class_idx, individual_maps, weights, agreement_scores)
    fuse_maps() fuses ALREADY-COMPUTED maps and returns a 4-tuple:
        (fused_map, individual_maps, weights, agreement_scores)
    """

    KERNELS = ("topk_dice", "pearson", "ssim_cosine")

    def __init__(self, methods: Dict[str, "BaseAttributor"],
                 ssim_win_size: int = 7,
                 agreement_kernel: Optional[str] = None,
                 temperature: Optional[float] = None,
                 topk_frac: Optional[float] = None,
                 weight_rule: str = "softmax"):
        agreement_kernel = agreement_kernel or _AGF_DEFAULT_KERNEL
        temperature = _AGF_DEFAULT_TAU if temperature is None else temperature
        topk_frac   = _AGF_DEFAULT_TOPK if topk_frac is None else topk_frac
        if agreement_kernel not in self.KERNELS:
            raise ValueError(f"agreement_kernel must be one of {self.KERNELS}, "
                             f"got {agreement_kernel!r}")
        if weight_rule not in ("softmax", "linear"):
            raise ValueError("weight_rule must be 'softmax' or 'linear'")
        if not (0.0 < float(topk_frac) < 1.0):
            raise ValueError("topk_frac must lie strictly in (0, 1)")
        if float(temperature) <= 0.0:
            raise ValueError("temperature must be > 0")
        self.methods          = methods
        self.ssim_win_size    = ssim_win_size
        self.agreement_kernel = agreement_kernel
        self.temperature      = float(temperature)
        self.topk_frac        = float(topk_frac)
        self.weight_rule      = weight_rule

    # ---- similarity primitives ------------------------------------------

    @staticmethod
    def _cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        """Raw cosine on the non-negative maps (v6 behaviour)."""
        a_flat = a.flatten().astype(np.float64)
        b_flat = b.flatten().astype(np.float64)
        num = float(np.dot(a_flat, b_flat))
        den = float(np.linalg.norm(a_flat) * np.linalg.norm(b_flat)) + 1e-8
        return num / den

    @staticmethod
    def _pearson_similarity(a: np.ndarray, b: np.ndarray) -> float:
        """Mean-centred cosine == Pearson r, in [-1, 1].

        A constant map (e.g. an identically-zero GradCAM++ on a layer that
        never activates) has zero variance and undefined correlation; it is
        scored 0.0, i.e. 'carries no agreement information', so the softmax
        pushes weight onto the two methods that do agree.
        """
        a_flat = a.flatten().astype(np.float64)
        b_flat = b.flatten().astype(np.float64)
        a_c = a_flat - a_flat.mean()
        b_c = b_flat - b_flat.mean()
        na = float(np.linalg.norm(a_c))
        nb = float(np.linalg.norm(b_c))
        if na < 1e-12 or nb < 1e-12:
            return 0.0
        r = float(np.dot(a_c, b_c) / (na * nb))
        return float(np.clip(r, -1.0, 1.0))

    def _dice_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """Dice overlap of the two top-`topk_frac` saliency masks, in [0, 1].

        Each map is thresholded at its OWN quantile, so the two masks always
        have (near-)equal cardinality and Dice is not biased by one method
        being flatter than the other.
        """
        a_flat = a.flatten().astype(np.float64)
        b_flat = b.flatten().astype(np.float64)
        q = 1.0 - self.topk_frac
        ta, tb = np.quantile(a_flat, q), np.quantile(b_flat, q)
        ma, mb = a_flat >= ta, b_flat >= tb
        # A constant map thresholds to the whole image; treat as uninformative.
        if ma.all() or mb.all() or not ma.any() or not mb.any():
            return 0.0
        inter = float(np.logical_and(ma, mb).sum())
        return float(2.0 * inter / (float(ma.sum()) + float(mb.sum()) + 1e-12))

    def _ssim_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        # a, b are already clipped to >= 0 and roughly in [0, 1] (see fuse()).
        a32, b32 = a.astype(np.float32), b.astype(np.float32)
        win = self.ssim_win_size
        # Guard against tiny maps (win_size must not exceed the smaller
        # spatial dimension) -- shrink to the largest valid odd window.
        max_valid = min(a32.shape) if min(a32.shape) % 2 == 1 else min(a32.shape) - 1
        win = min(win, max_valid) if max_valid >= 3 else 3
        try:
            return float(_skimage_ssim(a32, b32, data_range=1.0, win_size=win))
        except Exception:
            # Last-resort fallback: treat as perfectly-dissimilar rather than
            # crashing the whole fusion on a single degenerate image.
            return 0.0

    def kernel_similarity(self, a: np.ndarray, b: np.ndarray,
                          kernel: Optional[str] = None) -> float:
        """Pairwise agreement under `kernel` (defaults to self.agreement_kernel).

        Public so the Section 18 kernel ablation can score one set of
        already-computed maps under every kernel without re-attributing.
        """
        k = kernel or self.agreement_kernel
        if k == "pearson":
            return self._pearson_similarity(a, b)
        if k == "topk_dice":
            return self._dice_similarity(a, b)
        if k == "ssim_cosine":
            return 0.5 * (self._ssim_similarity(a, b) + self._cosine_similarity(a, b))
        raise ValueError(f"unknown agreement kernel {k!r}")

    def _pairwise_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        return self.kernel_similarity(a, b, self.agreement_kernel)

    # ---- agreement + weighting -------------------------------------------

    def agreement_scores(self, maps: Dict[str, np.ndarray],
                         kernel: Optional[str] = None) -> Dict[str, float]:
        """a_k = mean similarity of method k with the other K-1 methods."""
        names = list(maps.keys())
        scores: Dict[str, float] = {}
        for name_i in names:
            sims = [self.kernel_similarity(maps[name_i], maps[name_j], kernel)
                    for name_j in names if name_j != name_i]
            scores[name_i] = float(np.mean(sims))
        return scores

    # back-compat alias for any external caller
    _agreement_scores = agreement_scores

    def weights_from_scores(self, scores: Dict[str, float],
                            temperature: Optional[float] = None,
                            rule: Optional[str] = None) -> Dict[str, float]:
        names = list(scores.keys())
        vals  = np.array([scores[n] for n in names], dtype=np.float64)
        rule  = rule or self.weight_rule
        if rule == "linear":
            c = np.clip(vals, 0.0, None)
            tot = c.sum()
            w = (c / tot) if tot > 1e-12 else np.full(len(c), 1.0 / len(c))
        else:
            tau = float(temperature if temperature is not None else self.temperature)
            v = vals / tau
            v = v - v.max()                       # numerical stability
            e = np.exp(v)
            w = e / (e.sum() + 1e-12)
        return {n: float(wi) for n, wi in zip(names, w)}

    def _softmax_weights(self, scores: Dict[str, float]) -> Dict[str, float]:
        return self.weights_from_scores(scores)

    # ---- fusion over PRE-COMPUTED maps (no attributor calls) -------------

    def fuse_maps(self, indiv: Dict[str, np.ndarray]
                  ) -> Tuple[np.ndarray, Dict[str, np.ndarray],
                             Dict[str, float], Dict[str, float]]:
        """Fuse maps that the caller already computed.

        Added in v6 so the evaluation loop can compute each base method
        EXACTLY ONCE. The old path computed GradCAM++/SmoothGrad/IG
        standalone for their own table rows and then fuse() recomputed all
        three internally -- roughly 52 extra forward/backward passes per
        image, about half the total GPU time of a run.

        It also removes a subtler asymmetry: fuse() clipped its constituents
        to >= 0 after resize_map (Lanczos interpolation can undershoot below
        zero even on a [0,1] input) while the standalone arm did not. Gini,
        Sparsity and Entropy for "GradCAM++" and for "the GradCAM++ inside
        AGF" were therefore computed on slightly different arrays. Routing
        both through this method makes them the same array by construction.

        Returns (fused_map, clipped_individual_maps, weights, agreement).
        """
        if not indiv:
            raise ValueError("fuse_maps() received no maps.")
        indiv = {name: np.clip(np.asarray(m, dtype=np.float32), 0.0, None)
                 for name, m in indiv.items()}
        shapes = {m.shape for m in indiv.values()}
        if len(shapes) != 1:
            raise ValueError(f"fuse_maps() got mismatched map shapes: {shapes}. "
                             "Resize every map to target_size first.")

        agreement = self.agreement_scores(indiv)
        weights   = self.weights_from_scores(agreement)

        fused = np.zeros_like(next(iter(indiv.values())), dtype=np.float64)
        for name, w in weights.items():
            fused += w * indiv[name].astype(np.float64)

        fused = np.clip(fused, 0.0, None)
        fused = fused / (fused.max() + 1e-8)
        fused = np.nan_to_num(fused, nan=0.0, posinf=0.0, neginf=0.0)
        return fused.astype(np.float32), indiv, weights, agreement

    # ---- main entry point --------------------------------------------------

    def fuse(self, tensor, class_idx: Optional[int] = None,
              target_size: Optional[Tuple[int, int]] = None
              ) -> Tuple[np.ndarray, int, Dict[str, np.ndarray],
                         Dict[str, float], Dict[str, float]]:
        """
        Returns (fused_map, resolved_class_idx, individual_maps,
                 weights, agreement_scores).
        """
        indiv: Dict[str, np.ndarray] = {}
        resolved_class = class_idx
        for name, method in self.methods.items():
            attr, pred = method(tensor, resolved_class)
            if resolved_class is None:
                resolved_class = pred
            if target_size is not None:
                attr = resize_map(attr, target_size)
            indiv[name] = attr
        fused, indiv, weights, agreement = self.fuse_maps(indiv)
        return fused, resolved_class, indiv, weights, agreement

    def freeze_noise(self, enable: bool = True) -> "AgreementGuidedFusion":
        """Forward noise freezing to any stochastic constituent.

        Required so SaliencyMetrics.sensitivity() can hold SmoothGrad's noise
        draw fixed across the (base, perturbed) pair when measuring the FUSED
        method, exactly as it does when measuring SmoothGrad standalone.
        Without this the AGF row of the robustness table carries Monte-Carlo
        variance that the GradCAM++ and IG rows do not.
        """
        for m in self.methods.values():
            if hasattr(m, "freeze_noise"):
                m.freeze_noise(enable)
        return self


# Short alias used in the narrative/figures.
AGF = AgreementGuidedFusion
print("xai_attribution final-1.0 (v7.1 numerics) loaded — methods: GradCAM++, SmoothGrad, IG; "
      "proposed fusion: AGF (AgreementGuidedFusion). "
      "kernel=%s; _normalize fails loudly on degenerate maps." % _AGF_DEFAULT_KERNEL)

Writing /kaggle/working/xai_attribution.py


In [11]:
# ── §8–9  Import the freshly-written module ─────────────────────────────
import importlib
sys.path.insert(0, "/kaggle/working")
import xai_attribution as _xai
importlib.reload(_xai)
# The two retired fusion variants are no longer importable: both classes were
# deleted from xai_attribution.py in v6.
from xai_attribution import (
    BaseAttributor, resize_map, _safe_entropy,
    GradCAMPlusPlus, SmoothGrad, IntegratedGradients,
    AgreementGuidedFusion, SaliencyMetrics,
)
print("Imported:", [c.__name__ for c in
      (GradCAMPlusPlus, SmoothGrad, IntegratedGradients,
       AgreementGuidedFusion, SaliencyMetrics)])
# Hard guard: if a stale xai_attribution.py is on the path (e.g. from an
# earlier session's /kaggle/working), the fixes below will silently not apply.
assert hasattr(AgreementGuidedFusion, "fuse_maps"), \
    "Stale xai_attribution.py imported — re-run the %%writefile cell above."
assert hasattr(SmoothGrad, "freeze_noise"), \
    "Stale xai_attribution.py imported — re-run the %%writefile cell above."
print("v7 module verified: fuse_maps + freeze_noise present.")

# ---- push the CONFIG kernel choice into the freshly-written module --------
#   The module is emitted by %%writefile and therefore cannot see notebook
#   globals at definition time. Setting the three module-level defaults here
#   means AGREEMENT_KERNEL in Section 2 is the ONE place the kernel is chosen;
#   every AgreementGuidedFusion(...) construction site inherits it.
#   `_xai` is the module object bound by the import above.
_xai._AGF_DEFAULT_KERNEL = AGREEMENT_KERNEL
_xai._AGF_DEFAULT_TAU    = AGREEMENT_TAU
_xai._AGF_DEFAULT_TOPK   = TOPK_FRAC
assert AgreementGuidedFusion({}).agreement_kernel == AGREEMENT_KERNEL, \
    "kernel override did not take effect"
assert AgreementGuidedFusion({}).temperature == AGREEMENT_TAU
for _k in KERNEL_ABLATION:
    assert _k in AgreementGuidedFusion.KERNELS, f"unknown kernel in KERNEL_ABLATION: {_k}"
assert AgreementGuidedFusion({}).topk_frac == TOPK_FRAC
assert AgreementGuidedFusion({}).weight_rule == AGF_WEIGHT_RULE
XAI_MODULE_SHA256 = hashlib.sha256(open(_xai.__file__, "rb").read()).hexdigest()
print(f"AGF agreement kernel = {AGREEMENT_KERNEL} (tau={AGREEMENT_TAU}, topk={TOPK_FRAC})")
print(f"xai_attribution.py sha256 = {XAI_MODULE_SHA256}")

xai_attribution final-1.0 (v7.1 numerics) loaded — methods: GradCAM++, SmoothGrad, IG; proposed fusion: AGF (AgreementGuidedFusion). kernel=topk_dice; _normalize fails loudly on degenerate maps.
xai_attribution final-1.0 (v7.1 numerics) loaded — methods: GradCAM++, SmoothGrad, IG; proposed fusion: AGF (AgreementGuidedFusion). kernel=topk_dice; _normalize fails loudly on degenerate maps.
Imported: ['GradCAMPlusPlus', 'SmoothGrad', 'IntegratedGradients', 'AgreementGuidedFusion', 'SaliencyMetrics']
v7 module verified: fuse_maps + freeze_noise present.
AGF agreement kernel = topk_dice (tau=1.0, topk=0.1)
xai_attribution.py sha256 = aa778d17494935c864236f8803edf7be9830d1549d08a2a43f6f9df32803b0e2


## §10 Sanity checks

Synthetic, data-free agreement-kernel control plus a weight-simplex/finite-map assertion for every kernel
and temperature. The self-check table is always written to `Statistics/`.

In [12]:
# ── §10  Sanity check: agreement-kernel self-check (synthetic, no GPU) ────
#   Runs BEFORE any attribution work. A kernel that cannot separate agreeing
#   from disagreeing maps silently degrades AGF into an arithmetic mean, and
#   the only symptom is a near-uniform weight table 12 hours later. This cell
#   fails loudly instead.
#
#   Three synthetic maps share a coarse spatial prior (as real attribution
#   maps do) and differ only in fine structure. In the DISAGREE case one map
#   puts its fine structure elsewhere; in the AGREE case all three match.
def _kernel_selfcheck(verbose=True):
    H = W = IMG_SIZE
    yy, xx = np.mgrid[0:H, 0:W]

    def _blob(cy, cx, s):
        m = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2.0 * s * s))
        return (m / m.max()).astype(np.float32)

    _common = _blob(H // 2, W // 2, H // 4)

    def _mm(cy, cx, s, seed):
        r = np.random.default_rng(seed)
        m = 0.72 * _common + 0.28 * _blob(cy, cx, s)
        m = np.clip(m + 0.03 * r.standard_normal((H, W)), 0, None)
        return (m / m.max()).astype(np.float32)

    disagree = {"gradcam": _mm(95, 95, 20, 1), "smoothgrad": _mm(99, 98, 22, 2),
                "ig": _mm(150, 155, 20, 3)}
    agree    = {"gradcam": _mm(95, 95, 20, 11), "smoothgrad": _mm(97, 96, 21, 12),
                "ig": _mm(96, 98, 20, 13)}

    rows = []
    for k in AgreementGuidedFusion.KERNELS:
        f = AgreementGuidedFusion({}, agreement_kernel=k)
        sd = f.agreement_scores(disagree)
        wd = np.array(list(f.weights_from_scores(sd).values()))
        wa = np.array(list(f.weights_from_scores(f.agreement_scores(agree)).values()))
        sv = np.array(list(sd.values()))
        rows.append({"kernel": k, "score_spread": float(sv.max() - sv.min()),
                     "w_std_disagree": float(wd.std()), "w_std_agree": float(wa.std()),
                     "ratio": float(wd.std() / (wa.std() + 1e-12)),
                     "outlier_last": bool(min(sd, key=sd.get) == "ig")})
    df = pd.DataFrame(rows)
    if verbose:
        print("AGREEMENT-KERNEL SELF-CHECK (synthetic shared-prior control)")
        print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    live = df[df.kernel == AGREEMENT_KERNEL].iloc[0]
    assert live["outlier_last"], \
        f"{AGREEMENT_KERNEL}: failed to rank the outlier method last"
    assert live["ratio"] > 3.0, (
        f"{AGREEMENT_KERNEL}: weight spread barely responds to disagreement "
        f"(ratio={live['ratio']:.2f}). The fusion is an arithmetic mean in "
        f"disguise -- pick a different AGREEMENT_KERNEL before spending GPU time.")
    # The ratio alone is not enough: a kernel can respond proportionally while
    # its ABSOLUTE separation stays negligible (pearson does exactly this --
    # ratio ~99x, but weights of 0.336/0.336/0.329). Require real separation.
    assert live["w_std_disagree"] > 0.01, (
        f"{AGREEMENT_KERNEL}: absolute weight separation under clear "
        f"disagreement is only {live['w_std_disagree']:.4f} (uniform = 0.0000, "
        f"usable > 0.01). The weighting cannot meaningfully re-rank the base "
        f"methods; do not claim agreement-guided weighting with this kernel.")

    # NaN safety on a degenerate all-zero map (the VGG16-BN GradCAM++ case)
    deg = dict(disagree); deg["gradcam"] = np.zeros((H, W), np.float32)
    for k in AgreementGuidedFusion.KERNELS:
        fu, _, w, _ = AgreementGuidedFusion({}, agreement_kernel=k).fuse_maps(deg)
        assert np.isfinite(fu).all(), f"{k}: non-finite fused map on a constant input"
        assert abs(sum(w.values()) - 1.0) < 1e-9, f"{k}: weights do not sum to 1"

    if verbose:
        print(f"\n[OK] {AGREEMENT_KERNEL}: outlier down-weighted; weight spread "
              f"{live['ratio']:.1f}x larger under disagreement than agreement; "
              f"degenerate maps handled without NaN.")
    return df

_KERNEL_SELFCHECK = _kernel_selfcheck()
# weight-simplex check on random non-negative maps, every kernel, several taus
_rng = np.random.default_rng(SEED)
for _k in AgreementGuidedFusion.KERNELS:
    for _tau in [AGREEMENT_TAU] + TEMP_GRID:
        _m = {n: _rng.random((32, 32)).astype(np.float32) for n in ("gradcam", "smoothgrad", "ig")}
        _f = AgreementGuidedFusion({}, agreement_kernel=_k, temperature=_tau)
        _fu, _, _w, _ = _f.fuse_maps(_m)
        assert abs(sum(_w.values()) - 1.0) < 1e-9 and min(_w.values()) > 0.0
        assert np.isfinite(_fu).all() and _fu.max() <= 1.0 + 1e-6 and _fu.min() >= 0.0
print("[OK] weights on the simplex, fused maps finite and in [0, 1] for all kernels/taus.")
_KERNEL_SELFCHECK_PATH = os.path.join(DIRS["Statistics"], "agf_kernel_selfcheck.csv")
_KERNEL_SELFCHECK.to_csv(_KERNEL_SELFCHECK_PATH, index=False)   # always persisted
mark_section("10_sanity_checks")

AGREEMENT-KERNEL SELF-CHECK (synthetic shared-prior control)
     kernel  score_spread  w_std_disagree  w_std_agree   ratio  outlier_last
  topk_dice        0.1126          0.0171       0.0002 75.6501          True
    pearson        0.0220          0.0034       0.0000 99.3128          True
ssim_cosine        0.0219          0.0034       0.0005  7.2402          True

[OK] topk_dice: outlier down-weighted; weight spread 75.7x larger under disagreement than agreement; degenerate maps handled without NaN.
[OK] weights on the simplex, fused maps finite and in [0, 1] for all kernels/taus.


## §11 Evaluation machinery, calibration, and the leakage audit

**Leakage audit (proved from the code below).** AGF weights come only from `fuse_maps` →
`agreement_scores(maps)` → `weights_from_scores` — three attribution maps of the *same* image and nothing
else. No label, no insertion/deletion score and no sensitivity value is ever passed into the weighting
path (`aopc_curve` and `SaliencyMetrics.sensitivity` are called only *after* a map exists, and their
outputs are never fed back). The explained class is the model's own clean prediction. So AGF-Full and
Equal Fusion carry no circularity: no test-set metric determines the weights that are then scored by that
metric.

**AGF-Fixed**, the only arm with a static weight vector, is calibrated on the **benchmark validation
split** (§5b) — `run_calibration()` below — so calibration data (A) and evaluation data (B) are disjoint
by construction.

Also in this section: one map post-processing definition (`postprocess_map`) shared by the faithfulness,
robustness and fusion paths, and the Equal Fusion arm built from the same clipped constituents as AGF.

In [13]:
# ── §11a  Reusable machinery (defined once, reused per backbone) ──────────

# ---- AOPC insertion/deletion curve (grey baseline, softmax) ----------------
_NORM_MEAN_T = torch.tensor(NORM_MEAN, dtype=torch.float32).view(1, 3, 1, 1)
_NORM_STD_T  = torch.tensor(NORM_STD,  dtype=torch.float32).view(1, 3, 1, 1)

def neutral_baseline(inp: torch.Tensor, pixel_value: float = AOPC_BASELINE_PIXEL):
    """Constant-pixel baseline expressed in NORMALISED space.

    The inputs reaching this function have already been through
    Normalize(mean=NORM_MEAN, std=NORM_STD), so a scalar fill is not a
    neutral image: torch.full_like(inp, 0.5) back-projects to RGB
    (0.600, 0.568, 0.519), a warm tan. Neutral grey has to be computed
    per channel as (pixel - mean) / std -> (0.066, 0.196, 0.418).
    """
    m = _NORM_MEAN_T.to(inp.device, inp.dtype)
    s = _NORM_STD_T.to(inp.device, inp.dtype)
    return ((pixel_value - m) / s).expand_as(inp).contiguous()

def rank_pixels_desc(flat, seed: int = AOPC_TIE_BREAK_SEED):
    """Rank flattened attribution values descending, with SEEDED RANDOM
    tie-breaking.

    WHY THE PREVIOUS VERSION WAS WRONG
    ----------------------------------
    It used `np.argsort(flat)[::-1]`. np.argsort is stable-ascending, so
    reversing it returns tied values in DESCENDING INDEX order -- a
    deterministic bottom-right-to-top-left raster scan. Consequences:

      * any map with large flat regions is scored partly on image geometry
        rather than on attribution quality;
      * a fully constant map is scored ENTIRELY on geometry. The reported
        VGG16-BN GradCAM++ AOPC-Ins of 0.6678 was the score of revealing
        pixels from the bottom-right corner inward. It measured nothing
        about GradCAM++ or about the model.

    lexsort with a seeded permutation as the secondary key removes the
    spatial prior while keeping the run reproducible. NOTE: np.lexsort takes
    the LAST key as primary, so `-flat` drives the ordering and the
    permutation only resolves exact ties.

    The seed is a fixed CONSTANT, not per-image and not per-method: the tie
    order must be identical across methods so the paired Wilcoxon compares
    like with like. A per-call seed would inject variance into the pairing
    rather than remove bias from it.
    """
    flat = np.asarray(flat, dtype=np.float64).ravel()
    rng = np.random.default_rng(seed)
    tiebreak = rng.permutation(flat.size)
    return np.lexsort((tiebreak, -flat))

def aopc_curve(model, tensor, attr_map, class_idx,
               n_steps=AOPC_N_STEPS, patch_pct=AOPC_PATCH_PCT, mode="insert"):
    """mode='insert': reveal top-k pixels from grey -> higher = better.
       mode='delete': mask top-k pixels with grey    -> lower  = better.
       Returns a list of n_steps softmax-confidence scores.

       Both modes share ONE ranking (most-important-first) so insertion and
       deletion are exact complements of each other.
    """
    inp    = tensor.unsqueeze(0).to(DEVICE)
    flat   = attr_map.flatten()
    n_pix  = len(flat)
    ranked = np.ascontiguousarray(rank_pixels_desc(flat))
    grey   = neutral_baseline(inp)
    scores = []
    perturbed = mask = None
    with torch.inference_mode():
        for step in range(1, n_steps + 1):
            k = int(step * patch_pct * n_pix)
            mask = torch.zeros(n_pix, dtype=torch.float32, device=DEVICE)
            mask[ranked[:k]] = 1.0
            mask = mask.reshape(attr_map.shape).unsqueeze(0).unsqueeze(0)
            perturbed = (grey * (1 - mask) + inp * mask) if mode == "insert" \
                        else (inp * (1 - mask) + grey * mask)
            s = float(torch.softmax(model(perturbed), dim=1)[0, class_idx].item())
            scores.append(s)
    del inp, grey, mask, perturbed
    return scores


# ---- per-map metrics (numpy, cheap; map assumed >=0, roughly [0,1]) --------
def map_entropy(m: np.ndarray) -> float:
    return _safe_entropy(np.clip(m, 0, None))

def map_sparsity(m: np.ndarray, top_frac: float = 0.10) -> float:
    """Fraction of total attribution mass concentrated in the top `top_frac`
    of pixels. Higher = sparser / more focused. Complements Gini."""
    a = np.abs(m).flatten().astype(np.float64)
    tot = a.sum()
    if tot < 1e-8:
        return 0.0
    k = max(1, int(top_frac * a.size))
    topk = np.sort(a)[-k:].sum()
    return float(topk / tot)

map_gini = SaliencyMetrics.gini


# ---- per-image scalar accumulator (NO arrays kept in lists) ---------------
@dataclass
class MethodAccumulator:
    ins_aucs:    List[float]       = field(default_factory=list)
    del_aucs:    List[float]       = field(default_factory=list)
    ins_curves:  List[List[float]] = field(default_factory=list)
    del_curves:  List[List[float]] = field(default_factory=list)
    gini:        List[float]       = field(default_factory=list)
    sparsity:    List[float]       = field(default_factory=list)
    entropy:     List[float]       = field(default_factory=list)
    runtime_ms:  List[float]       = field(default_factory=list)
    sens:        List[float]       = field(default_factory=list)   # robustness
    sens_idxs:   List[int]         = field(default_factory=list)   # image idx per sens value
    sample_idxs: List[int]         = field(default_factory=list)
    preds:       List[int]         = field(default_factory=list)
    labels:      List[int]         = field(default_factory=list)
    n_ok: int = 0


# ---- statistics: bootstrap CI + effect sizes ------------------------------
def bootstrap_ci(scores, n_boot=BOOTSTRAP_N, alpha=0.05, seed=SEED,
                 stat=np.mean):
    """Percentile bootstrap CI for `stat`.

    `stat` MUST match the point estimate you report alongside the interval.
    The previous version always resampled the mean while paired_compare
    reported a median, which is why the IG row's CI could sit entirely to
    one side of its own median_diff. Pass stat=np.median for paired
    differences, stat=np.mean for per-method score means.
    """
    rng = np.random.default_rng(seed)
    arr = np.asarray(scores, dtype=np.float64)
    if arr.size == 0:
        return (float("nan"), float("nan"))
    idx  = rng.integers(0, arr.size, size=(n_boot, arr.size))
    boot = stat(arr[idx], axis=1)
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    del boot, idx
    return float(lo), float(hi)

def cliffs_delta(a, b) -> float:
    """Cliff's delta in [-1,1]; >0 means a tends to exceed b. O(n log n)."""
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    na, nb = a.size, b.size
    if na == 0 or nb == 0:
        return 0.0
    b_sorted = np.sort(b)
    greater = np.searchsorted(b_sorted, a, side="left").sum()      # #(a>b)
    less    = (nb * na) - np.searchsorted(b_sorted, a, side="right").sum()  # #(a<b)
    return float((greater - less) / (na * nb))

def paired_rank_biserial(a, b) -> float:
    """Matched-pairs rank-biserial correlation for the Wilcoxon signed-rank
    test (Kerby 2014): r = (sum ranks favouring a - favouring b) / total.
    Range [-1,1]; +1 means a>b on every non-zero pair."""
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    d = a - b
    d = d[d != 0]
    if d.size == 0:
        return 0.0
    ranks = pd.Series(np.abs(d)).rank().to_numpy()
    r_pos = ranks[d > 0].sum()
    r_neg = ranks[d < 0].sum()
    tot = r_pos + r_neg
    return float((r_pos - r_neg) / tot) if tot > 0 else 0.0

def cliff_magnitude(delta: float) -> str:
    """Romano et al. (2006) thresholds -- valid for Cliff's delta ONLY."""
    d = abs(delta)
    return ("negligible" if d < 0.147 else "small" if d < 0.33
            else "medium" if d < 0.474 else "large")

def rb_magnitude(r: float) -> str:
    """Cohen's conventions for correlation-family effect sizes, applied to
    the matched-pairs rank-biserial r. State these thresholds in the paper.

    This is the effect size that belongs to the Wilcoxon signed-rank test.
    Cliff's delta is an UNPAIRED dominance statistic; it ignores the pairing
    the test relies on and can disagree sharply (e.g. rank-biserial -0.337,
    'moderate', vs delta -0.084, 'negligible', on the same comparison).
    """
    r = abs(r)
    return ("negligible" if r < 0.10 else "small" if r < 0.30
            else "medium" if r < 0.50 else "large")


# ---- inferential statistics ------------------------------------------------
# FINAL: the per-backbone `paired_compare()` of v7.1 was REMOVED. It tested only
# insertion and robustness (never deletion), never compared against Equal Fusion,
# applied Holm within a 3-test family per backbone+metric, and reported a paired
# MEDIAN next to a Wilcoxon p (which can disagree in sign on skewed data -- see the
# ResNet50 v7.1 row). All inference now happens ONCE, in §15, from the per-image
# CSVs, with pre-declared families (CONFIG: PRIMARY_* / SECONDARY_*).

# ---- map post-processing: ONE definition used by every loop ---------------
def postprocess_map(m: np.ndarray) -> np.ndarray:
    """CONFIG MAP_POSTPROC: Lanczos resize to IMG_SIZE x IMG_SIZE, clip >= 0.
    Identical to what fuse_maps() does to its constituents, so the standalone
    rows, the fusion inputs and the robustness maps are the same arrays."""
    return np.clip(resize_map(np.asarray(m, dtype=np.float32), (IMG_SIZE, IMG_SIZE)),
                   0.0, None).astype(np.float32)

EQUAL_WEIGHTS = {"gradcam": 1.0 / 3, "smoothgrad": 1.0 / 3, "ig": 1.0 / 3}

# ---- attributor factory (fresh per image; hooks never accumulate) ---------
def make_base_generators(model, target_layer):
    gc_gen = GradCAMPlusPlus(model, target_layer)
    sg_gen = SmoothGrad(model, n_samples=SG_N_SAMPLES, noise_std=SG_NOISE_STD,
                        seed=SEED, noise_mode=SG_NOISE_MODE,
                        noise_frac=SG_NOISE_FRAC, seed_mode=SG_SEED_MODE)
    ig_gen = IntegratedGradients(model, n_steps=IG_N_STEPS)
    return gc_gen, sg_gen, ig_gen

def fusion_dict(gc_gen, sg_gen, ig_gen):
    return {"gradcam": gc_gen, "smoothgrad": sg_gen, "ig": ig_gen}


class _MethodCallable:
    """Callable attributor wrapper that also forwards freeze_noise().

    SaliencyMetrics.sensitivity() freezes any stochastic estimator across
    the (base, perturbed) pair so the metric measures input sensitivity and
    not Monte-Carlo variance. That only works if the object it is handed
    exposes freeze_noise(). make_method_callable used to return bare
    closures, so the freeze silently did nothing for SmoothGrad and AGF --
    the two stochastic rows -- while GradCAM++ and IG were deterministic.
    The robustness column was therefore not measuring the same quantity
    across rows.
    """
    __slots__ = ("_fn", "_stochastic", "_cleanup")

    def __init__(self, fn, stochastic=(), cleanup=None):
        self._fn = fn
        self._stochastic = list(stochastic)
        self._cleanup = cleanup

    def __call__(self, t, c=None):
        return self._fn(t, c)

    def freeze_noise(self, enable: bool = True):
        for g in self._stochastic:
            if hasattr(g, "freeze_noise"):
                g.freeze_noise(enable)
        return self

    def close(self):
        if self._cleanup is not None:
            self._cleanup()


def make_method_callable(key, model, target_layer):
    """Return a fresh attributor callable f(tensor, class_idx)->(map, pred)
    for robustness/sensitivity.

    GAF and a reliability-weighted variant branches removed: both classes are gone from
    xai_attribution.py (retired from the experiment, ~250 lines of
    unreachable code). Re-adding them means restoring the classes first.

    Generators are now built ONCE per callable rather than per call, so a
    single sensitivity() evaluation reuses one SmoothGrad instance and its
    frozen noise cache survives across the base and perturbed calls. The
    GradCAM++ branch still rebuilds per call because its hooks must not
    accumulate; it is deterministic, so nothing is frozen there.
    """
    # FINAL (REQUIRES RERUN of robustness): every callable returns the
    # POST-PROCESSED 224x224 map (postprocess_map). v7.1 returned GradCAM++ at its
    # native 7x7 resolution here while IG/SmoothGrad/AGF were 224x224 and the
    # faithfulness loop used the resized maps -- the robustness column therefore
    # measured different objects for different rows. For IG and SmoothGrad the
    # post-processing is a no-op (already 224x224 and >= 0), so their values are
    # unchanged; GradCAM++ changes; Equal Fusion is new.
    if key == "gradcam_pp":
        def _fn(t, c=None):
            gc = GradCAMPlusPlus(model, target_layer)
            try:
                m, p = gc.generate(t, c)
                return postprocess_map(m), p
            finally: gc.remove()
        return _MethodCallable(_fn)

    if key == "ig":
        ig = IntegratedGradients(model, n_steps=IG_N_STEPS)
        def _fn(t, c=None):
            m, p = ig.generate(t, c)
            return postprocess_map(m), p
        return _MethodCallable(_fn)

    if key == "smoothgrad":
        sg = SmoothGrad(model, n_samples=SG_N_SAMPLES, noise_std=SG_NOISE_STD,
                        seed=SEED, noise_mode=SG_NOISE_MODE,
                        noise_frac=SG_NOISE_FRAC, seed_mode=SG_SEED_MODE)
        def _fn(t, c=None):
            m, p = sg.generate(t, c)
            return postprocess_map(m), p
        return _MethodCallable(_fn, stochastic=[sg])

    if key in ("agf", "equal"):
        gc, sg, ig = make_base_generators(model, target_layer)
        fuser = AgreementGuidedFusion(fusion_dict(gc, sg, ig))
        def _fn(t, c=None, _key=key):
            m, p, indiv, _, _ = fuser.fuse(t, class_idx=c,
                                           target_size=(IMG_SIZE, IMG_SIZE))
            if _key == "equal":
                m = weighted_arith(indiv, EQUAL_WEIGHTS)
            return m, p
        return _MethodCallable(_fn, stochastic=[sg], cleanup=gc.remove)

    raise KeyError(f"Unknown method key {key!r}. Valid keys: {ORDER}")


# ---- AGF ablation fusers (ONE fuse per image, no recomputation) ------------
def _renorm(m):
    m = np.clip(m, 0.0, None)
    return (m / (m.max() + 1e-8)).astype(np.float32)

def weighted_arith(indiv: Dict[str, np.ndarray], weights: Dict[str, float]) -> np.ndarray:
    fused = np.zeros_like(next(iter(indiv.values())), dtype=np.float64)
    for name, w in weights.items():
        fused += w * indiv[name].astype(np.float64)
    return _renorm(fused)

def agf_ablation_maps(gc_gen, sg_gen, ig_gen, tensor, pred, fixed_weights=None):
    """One AGF call -> the three ablation variants:
       agf_full : per-image agreement-weighted softmax weights (PROPOSED)
       agf_equal: uniform weights = 1/K   (agreement weighting REMOVED)
       agf_fixed: static weights calibrated on the VALIDATION split (§11b);
                  if None, falls back to equal (logged by the caller)
    Returns (maps_dict, agreement_weights_dict).

    Signature and return type are unchanged so Section 16 keeps working.
    """
    fused_full, _, indiv, w_agree, _ = AgreementGuidedFusion(
        fusion_dict(gc_gen, sg_gen, ig_gen)
    ).fuse(tensor, class_idx=pred, target_size=(IMG_SIZE, IMG_SIZE))
    names = list(indiv.keys())
    eq = {n: 1.0 / len(names) for n in names}
    if fixed_weights is None:
        fixed_weights = eq
    fx = {n: float(fixed_weights.get(n, eq[n])) for n in names}
    s = sum(fx.values()) or 1.0
    fx = {n: v / s for n, v in fx.items()}
    maps = {
        "agf_full":  fused_full,
        "agf_equal": weighted_arith(indiv, eq),
        "agf_fixed": weighted_arith(indiv, fx),
    }
    return maps, w_agree

print("Reusable machinery ready.")

Reusable machinery ready.


In [14]:
# ── §11c  Per-model evaluation loops (reused for every backbone) ──────────
# FINAL changes (all marked [FINAL]):
#   * Equal Fusion is computed in the main loop from the SAME clipped maps as AGF
#     (central baseline; REQUIRES RERUN -- new arm, other arms unaffected).
#   * AGF weight rows carry img_idx/pred/label/confidence explicitly (no positional
#     alignment); ablation rows carry the img_idx of every surviving image.
#   * run_calibration(): AGF-Fixed weights from the VALIDATION split.
#   * Every image is loaded through load_image_tensor() (benchmark preprocessing).

def run_main_loop(eval_model, target_layer, eval_idxs):
    """AOPC-Ins/Del + Gini + Sparsity + Entropy + Runtime for every method in
    ORDER. Also returns AGF per-image agreement weights and a disagreement
    score.

    TWO FIXES vs the previous version
    ---------------------------------
    (1) NO DOUBLE COMPUTATION. The old loop generated GradCAM++, SmoothGrad
        and IG standalone for their own table rows, and then called
        AgreementGuidedFusion(fd).fuse(...), which recomputed all three
        internally. That is ~52 extra forward/backward passes per image and
        was roughly half the 3h55m total runtime. We now compute each base
        method exactly once and hand the finished maps to fuse_maps().

    (2) SYMMETRIC PREPROCESSING. fuse() clipped its constituents to >= 0
        after resize_map (Lanczos can undershoot below zero even on a [0,1]
        input); the standalone arm did not clip. So "GradCAM++" and "the
        GradCAM++ inside AGF" were different arrays, and Gini / Sparsity /
        Entropy were computed on the unclipped version while the fusion used
        the clipped one. Both now come from the same fuse_maps() output.

    RUNTIME SEMANTICS. rt_agf is now base cost + fusion overhead measured
    directly, i.e. the true end-to-end cost of producing an AGF map. This is
    the honest number and it is large (AGF must run all three base methods);
    report it as such rather than timing the fusion arithmetic alone.
    """
    acc = {k: MethodAccumulator() for k in ORDER}
    agf_weight_rows, disagreement = [], []
    seen = {cn: 0 for cn in class_names}

    for pos, img_idx in enumerate(tqdm(eval_idxs, desc="Main eval", leave=False)):
        path, label = sample_paths[img_idx], sample_labels[img_idx]
        gc_gen = sg_gen = ig_gen = None
        try:
            tensor = load_image_tensor(path)
            gc_gen, sg_gen, ig_gen = make_base_generators(eval_model, target_layer)
            fd = fusion_dict(gc_gen, sg_gen, ig_gen)
            with torch.inference_mode(False):
                # --- each base method computed ONCE, timed individually ---
                t0 = time.perf_counter(); gc_map, pred = gc_gen.generate(tensor)
                rt_gc = (time.perf_counter() - t0) * 1e3
                t0 = time.perf_counter(); sg_map, _ = sg_gen.generate(tensor, class_idx=pred)
                rt_sg = (time.perf_counter() - t0) * 1e3
                t0 = time.perf_counter(); ig_map, _ = ig_gen.generate(tensor, class_idx=pred)
                rt_ig = (time.perf_counter() - t0) * 1e3

                # --- resize to a common grid, then fuse the FINISHED maps ---
                t0 = time.perf_counter()
                _pre = {"gradcam":    resize_map(gc_map, (IMG_SIZE, IMG_SIZE)),
                        "smoothgrad": resize_map(sg_map, (IMG_SIZE, IMG_SIZE)),
                        "ig":         resize_map(ig_map, (IMG_SIZE, IMG_SIZE))}
                agf_map, indiv, agf_w, agf_agree = \
                    AgreementGuidedFusion(fd).fuse_maps(_pre)
                rt_fuse = (time.perf_counter() - t0) * 1e3
                # [FINAL] Equal Fusion: same clipped constituents, w = 1/3 each
                t0 = time.perf_counter()
                equal_map = weighted_arith(indiv, EQUAL_WEIGHTS)
                rt_eq_fuse = (time.perf_counter() - t0) * 1e3
                with torch.inference_mode():
                    pred_conf = float(torch.softmax(
                        eval_model(tensor.unsqueeze(0).to(DEVICE)).float(), dim=1)[0, pred].item())
            # end-to-end cost of an AGF map = all three base methods + fusion
            rt_agf = rt_gc + rt_sg + rt_ig + rt_fuse
            rt_equal = rt_gc + rt_sg + rt_ig + rt_eq_fuse   # [FINAL] same base cost

            seen[class_names[label]] += 1
            # indiv holds the CLIPPED maps that AGF actually consumed, so the
            # standalone rows and the fusion constituents are identical arrays.
            _all_maps = {
                "gradcam_pp": indiv["gradcam"],
                "ig":         indiv["ig"],
                "smoothgrad": indiv["smoothgrad"],
                "equal":      equal_map,
                "agf":        agf_map,
            }
            _all_rts = {"gradcam_pp": rt_gc, "ig": rt_ig, "smoothgrad": rt_sg,
                        "equal": rt_equal, "agf": rt_agf}
            missing = [k for k in ORDER if k not in _all_maps]
            if missing:
                raise KeyError(f"ORDER contains keys this loop cannot produce: "
                               f"{missing}. the retired fusion variants were removed in v6.")
            maps = {k: _all_maps[k] for k in ORDER}
            rts  = {k: _all_rts[k]  for k in ORDER}
            for key, amap in maps.items():
                ins = aopc_curve(eval_model, tensor, amap, pred, mode="insert")
                del_ = aopc_curve(eval_model, tensor, amap, pred, mode="delete")
                a = acc[key]
                a.ins_aucs.append(float(np.mean(ins)))
                a.del_aucs.append(float(np.mean(del_)))
                a.ins_curves.append(ins); a.del_curves.append(del_)
                a.gini.append(map_gini(amap))
                a.sparsity.append(map_sparsity(amap))
                a.entropy.append(map_entropy(amap))
                a.runtime_ms.append(float(rts[key]))
                a.sample_idxs.append(img_idx)
                a.preds.append(int(pred)); a.labels.append(int(label))
                a.n_ok += 1
            agf_weight_rows.append({"img_idx": int(img_idx), "pred": int(pred), "label": int(label),
                                    "pred_conf": pred_conf,
                                    **{f"w_{n}": float(w) for n, w in agf_w.items()},
                                    **{f"a_{n}": float(a) for n, a in agf_agree.items()}})
            disagreement.append((img_idx, 1.0 - float(np.mean(list(agf_agree.values())))))
            del maps, _all_maps, _all_rts, _pre, indiv
            del gc_map, sg_map, ig_map, agf_map, equal_map, tensor
        except Exception as e:
            # _normalize now RAISES on a degenerate map instead of returning
            # zeros, so a broken target layer lands here loudly and the image
            # is skipped rather than contributing a fabricated data point.
            log(f"  [SKIP main] idx={img_idx}: {type(e).__name__}: {e}")
        finally:
            if gc_gen is not None: gc_gen.remove()
            del gc_gen, sg_gen, ig_gen
            if pos % 20 == 0:
                free_gpu()
    return acc, agf_weight_rows, disagreement


def run_sens_loop(eval_model, target_layer, gini_idxs, acc):
    """Fill acc[key].sens with scale-invariant max-sensitivity (robustness).

    Callables are built ONCE per image and closed afterwards, so a single
    sensitivity() evaluation reuses one SmoothGrad instance and its frozen
    noise cache spans the base and perturbed calls (see _MethodCallable).
    """
    metrics_eng = SaliencyMetrics(eval_model)
    for pos, img_idx in enumerate(tqdm(gini_idxs, desc="Robustness", leave=False)):
        tensor = load_image_tensor(sample_paths[img_idx])
        _vals = {}
        _callables = {}
        try:
            with torch.inference_mode(False):
                _gc = GradCAMPlusPlus(eval_model, target_layer)
                try:    pred = _gc.generate(tensor)[1]
                finally: _gc.remove()
                for key in ORDER:
                    fn = make_method_callable(key, eval_model, target_layer)
                    _callables[key] = fn
                    s = metrics_eng.sensitivity(
                        tensor, fn, pred, radius=SENS_RADIUS,
                        n_samples=SENS_SAMPLES, seed=SEED,
                        freeze_stochastic=SENS_FREEZE_SG_NOISE)
                    _vals[key] = float(s); del s
        except Exception as e:
            log(f"  [SKIP sens] idx={img_idx}: {type(e).__name__}: {e}")
        finally:
            for fn in _callables.values():
                try: fn.close()
                except Exception: pass
            # append all-or-nothing so every method's sens array stays the
            # same length and index-aligned. A partial failure previously left
            # some methods one entry short, silently misaligning all later pairs.
            for key in ORDER:
                acc[key].sens.append(_vals.get(key, float("nan")))
                acc[key].sens_idxs.append(int(img_idx))
            del tensor, _callables
            if pos % 10 == 0:
                free_gpu()
    return acc


def run_agf_ablation(eval_model, target_layer, ablation_idxs, fixed_weights):
    """AGF ablation: full vs equal-weights vs fixed-weights. AOPC-Ins + Del.

    [FINAL] `fixed_weights` comes from run_calibration() on the benchmark
    VALIDATION split, so no test image contributes to the AGF-Fixed vector.
    Returns abl with an explicit "idx" list of every image that survived.
    """
    keys = ["agf_full", "agf_equal", "agf_fixed"]
    abl = {k: {"ins": [], "del": []} for k in keys}
    abl["idx"] = []
    for pos, img_idx in enumerate(tqdm(ablation_idxs, desc="AGF ablation", leave=False)):
        tensor = load_image_tensor(sample_paths[img_idx])
        gc_gen = sg_gen = ig_gen = None
        try:
            gc_gen, sg_gen, ig_gen = make_base_generators(eval_model, target_layer)
            with torch.inference_mode(False):
                pred = gc_gen.generate(tensor)[1]
                maps, _ = agf_ablation_maps(gc_gen, sg_gen, ig_gen, tensor, pred,
                                            fixed_weights=fixed_weights)
            _vals = {}
            for k in keys:
                ins = aopc_curve(eval_model, tensor, maps[k], pred, mode="insert")
                dl  = aopc_curve(eval_model, tensor, maps[k], pred, mode="delete")
                _vals[k] = (float(np.mean(ins)), float(np.mean(dl)))
            for k in keys:                       # all-or-nothing append keeps arms aligned
                abl[k]["ins"].append(_vals[k][0]); abl[k]["del"].append(_vals[k][1])
            abl["idx"].append(int(img_idx))
            del maps
        except Exception as e:
            log(f"  [SKIP abl] idx={img_idx}: {type(e).__name__}: {e}")
        finally:
            if gc_gen is not None: gc_gen.remove()
            del gc_gen, sg_gen, ig_gen, tensor
            free_gpu()
    return abl

def run_calibration(eval_model, target_layer, spec):
    """[FINAL] AGF weights on the CALIBRATION split (benchmark validation images).

    Target class = the model's clean prediction (no label is used). Returns the
    mean weight vector (-> AGF-Fixed) and writes the per-image weights to
    Statistics/calibration/<key>_calibration_weights.csv plus a JSON vector.
    No faithfulness or robustness metric is computed here."""
    rows = []
    for pos in tqdm(CALIB_POSITIONS, desc="Calibration (val split)", leave=False):
        gc_gen = sg_gen = ig_gen = None
        try:
            x = load_image_tensor(VAL_PATHS[pos])
            gc_gen, sg_gen, ig_gen = make_base_generators(eval_model, target_layer)
            with torch.inference_mode(False):
                _, pred, _, w, a = AgreementGuidedFusion(
                    fusion_dict(gc_gen, sg_gen, ig_gen)).fuse(
                        x, class_idx=None, target_size=(IMG_SIZE, IMG_SIZE))
            rows.append({"backbone": spec["display"], "split": "benchmark_val",
                         "val_position": int(pos), "image_id": VAL_IDS[pos],
                         "label": int(VAL_LABELS[pos]), "pred": int(pred),
                         **{f"w_{n}": float(v) for n, v in w.items()},
                         **{f"a_{n}": float(v) for n, v in a.items()}})
            del x
        except Exception as e:
            log(f"  [SKIP calib] val_pos={pos}: {type(e).__name__}: {e}")
        finally:
            if gc_gen is not None: gc_gen.remove()
            del gc_gen, sg_gen, ig_gen
    if not rows:
        raise RuntimeError(f"[calib] no calibration image survived for {spec['display']}")
    cdf = pd.DataFrame(rows)
    wcols = [c for c in cdf.columns if c.startswith("w_")]
    fixed = {c[2:]: float(cdf[c].mean()) for c in wcols}
    s = sum(fixed.values()); fixed = {k: v / s for k, v in fixed.items()}
    cdir = os.path.join(DIRS["Statistics"], "calibration")
    cdf.to_csv(os.path.join(cdir, f"{spec['key']}_calibration_weights.csv"), index=False)
    with open(os.path.join(cdir, f"{spec['key']}_fixed_weights.json"), "w") as f:
        json.dump({"backbone": spec["display"], "n_images": len(cdf), "source": "benchmark_val_split",
                   "calib_seed": CALIB_SEED, "kernel": AGREEMENT_KERNEL, "tau": AGREEMENT_TAU,
                   "topk_frac": TOPK_FRAC, "fixed_weights": fixed,
                   "config_version": CONFIG_VERSION}, f, indent=2)
    log(f"[calib] {spec['display']}: n={len(cdf)} fixed weights = "
        + ", ".join(f"{k}={v:.4f}" for k, v in fixed.items()))
    return fixed


def load_fixed_weights(key):
    p = os.path.join(DIRS["Statistics"], "calibration", f"{key}_fixed_weights.json")
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} missing: run §12 (calibration) for this backbone first")
    with open(p) as f:
        return json.load(f)["fixed_weights"]

print("Evaluation loops ready.")

Evaluation loops ready.


In [15]:
# ── §11d  Per-image persistence (raw data for ALL statistics in §15) ──────
#   PURE ADDITION. Nothing in this cell touches the evaluation protocol:
#   no attributor is called, no RNG is consumed, no accumulator is mutated.
#   It only SERIALISES the per-image scalars that Section 9 already computed
#   and that Section 12 was previously reducing to means and discarding.
#
#   WHY THIS EXISTS
#   ---------------
#   The v7 run persisted aggregates only (Results/*_summary.json holds means
#   plus three stat rows). Every per-image array -- ins_aucs, del_aucs, sens,
#   gini, sparsity, entropy, runtime, the AGF weight vectors and the AOPC
#   curves -- died with the kernel. Any reviewer request that needs the raw
#   pairs (sign test, skewness, alternate correction, TOST, paired scatter,
#   per-class breakdown, curve overlays) therefore costs a full ~2h45m
#   re-run. This cell makes all of that answerable from disk.
#
#   OUTPUT LOCATION. Written directly into Statistics/per_image/, exactly as
#   Sections 17/18 write their raw sweeps. That path bypasses save_table()'s
#   PUBLICATION_ONLY whitelist by design -- these are the raw research
#   artifacts, not curated publication tables, so they must never be
#   silently dropped by a whitelist miss.
# ---------------------------------------------------------------------------
PER_IMAGE_DIR = os.path.join(DIRS["Statistics"], "per_image")
os.makedirs(PER_IMAGE_DIR, exist_ok=True)


def _pi_write(df: pd.DataFrame, name: str) -> str:
    """Write one per-image table to Statistics/per_image/ (no whitelist)."""
    p = os.path.join(PER_IMAGE_DIR, f"{name}.csv")
    df.to_csv(p, index=False)
    return p


def _pi_align(acc_obj, idx_attr: str, fields: dict):
    """Build index-aligned rows from one MethodAccumulator.

    `fields` maps output column name -> accumulator attribute name. Arrays
    that are shorter than the index list (possible only if an exception hit
    mid-image) are truncated to the common length rather than zipped
    blindly, so a partial failure can never silently offset a column.
    """
    idxs = list(getattr(acc_obj, idx_attr) or [])
    cols = {c: list(getattr(acc_obj, a) or []) for c, a in fields.items()}
    n = min([len(idxs)] + [len(v) for v in cols.values()]) if cols else len(idxs)
    rows = []
    for i in range(n):
        r = {"img_idx": int(idxs[i])}
        for c, v in cols.items():
            r[c] = v[i]
        rows.append(r)
    return rows, n, idxs


RUN_TAGS = {"config_version": CONFIG_VERSION, "split": "test", "agf_kernel": AGREEMENT_KERNEL,
            "agf_tau": AGREEMENT_TAU, "agf_topk_frac": TOPK_FRAC, "seed": SEED}

def _tag(df, spec, subset_seed):
    for k, v in RUN_TAGS.items():
        df[k] = v
    df["subset_seed"] = subset_seed
    if "img_idx" in df.columns:
        df["image_id"] = [sample_ids[int(i)] if int(i) >= 0 else "" for i in df["img_idx"]]
    return df

def dump_per_image_records(spec, acc, agf_weight_rows=None, disagreement=None,
                           abl=None, abl_idxs=None):
    """Serialise every per-image quantity Section 9/12 produced for ONE backbone.

    Emits, under Statistics/per_image/:
      <key>_per_image.csv         main-loop scalars, one row per (method, image)
      <key>_per_image_sens.csv    max-sensitivity, one row per (method, image)
      <key>_agf_weights.csv       AGF fusion weights + disagreement, per image
      <key>_aopc_curves.csv       full insertion/deletion curves, per step
      <key>_ablation_per_image.csv  AGF-Full/Equal/Fixed AOPC, per image

    Returns a dict of written paths. Never raises: a persistence failure must
    not take down a completed 2h evaluation. Failures are logged loudly.
    """
    key, disp = spec["key"], spec["display"]
    written = {}
    try:
        # ---- 1. main-loop per-image scalars -------------------------------
        rows = []
        for m in ORDER:
            a = acc[m]
            recs, n, _ = _pi_align(a, "sample_idxs", {
                "aopc_ins": "ins_aucs", "aopc_del_raw": "del_aucs",
                "gini": "gini", "sparsity": "sparsity", "entropy": "entropy",
                "runtime_ms": "runtime_ms", "pred": "preds", "label": "labels",
            })
            if n < len(a.sample_idxs or []):
                log(f"[per-image][WARN] {disp}/{m}: truncated to n={n} "
                    f"(index list had {len(a.sample_idxs)}) -- a mid-image "
                    f"failure left ragged arrays; columns realigned, not zipped.")
            for r in recs:
                r["backbone"] = disp
                r["method_key"] = m
                r["method"] = DISP[m]
                r["aopc_del_comp"] = 1.0 - float(r["aopc_del_raw"])
                r["label_name"] = class_names[int(r["label"])]
                r["pred_name"] = class_names[int(r["pred"])]
                r["correct"] = bool(int(r["pred"]) == int(r["label"]))
                rows.append(r)
        if rows:
            df = pd.DataFrame(rows)[[
                "backbone", "method_key", "method", "img_idx",
                "label", "label_name", "pred", "pred_name", "correct",
                "aopc_ins", "aopc_del_raw", "aopc_del_comp",
                "gini", "sparsity", "entropy", "runtime_ms"]]
            written["main"] = _pi_write(_tag(df, spec, EVAL_SEED), f"{key}_per_image")

        # ---- 2. sensitivity (its own, smaller index set) -------------------
        srows = []
        for m in ORDER:
            a = acc[m]
            recs, _, _ = _pi_align(a, "sens_idxs", {"sensitivity": "sens"})
            for r in recs:
                r["backbone"] = disp
                r["method_key"] = m
                r["method"] = DISP[m]
                r["label"] = int(sample_labels[r["img_idx"]])
                r["label_name"] = class_names[r["label"]]
                srows.append(r)
        if srows:
            sdf = pd.DataFrame(srows)[[
                "backbone", "method_key", "method", "img_idx",
                "label", "label_name", "sensitivity"]]
            sdf["sens_samples"] = SENS_SAMPLES; sdf["sens_radius"] = SENS_RADIUS
            written["sens"] = _pi_write(_tag(sdf, spec, SENS_SEED), f"{key}_per_image_sens")

        # ---- 3. AGF per-image weights (rows carry img_idx explicitly) -------
        if agf_weight_rows:
            dis_map = {int(i): float(d) for i, d in (disagreement or [])}
            wdf = pd.DataFrame(agf_weight_rows)
            wcols = [c for c in wdf.columns if c.startswith("w_")]
            wdf.insert(0, "backbone", disp)
            wdf["label_name"] = [class_names[int(v)] for v in wdf["label"]]
            wdf["wspread_std"] = wdf[wcols].std(axis=1, ddof=0)
            wdf["wspread_range"] = wdf[wcols].max(axis=1) - wdf[wcols].min(axis=1)
            wdf["wsum"] = wdf[wcols].sum(axis=1)
            wdf["disagreement"] = [dis_map.get(int(i), float("nan")) for i in wdf["img_idx"]]
            assert np.allclose(wdf["wsum"], 1.0, atol=1e-6), "AGF weights do not sum to 1"
            written["weights"] = _pi_write(_tag(wdf, spec, EVAL_SEED), f"{key}_agf_weights")

        # ---- 4. full AOPC curves (n_steps points per image per mode) -------
        crows = []
        for m in ORDER:
            a = acc[m]
            idxs = list(a.sample_idxs or [])
            for mode, curves in (("insert", a.ins_curves), ("delete", a.del_curves)):
                for i in range(min(len(idxs), len(curves or []))):
                    for step, val in enumerate(curves[i], start=1):
                        crows.append({
                            "backbone": disp, "method_key": m, "method": DISP[m],
                            "img_idx": int(idxs[i]), "mode": mode, "step": step,
                            "frac_pixels": round(step * AOPC_PATCH_PCT, 6),
                            "score": float(val)})
        if crows:
            written["curves"] = _pi_write(_tag(pd.DataFrame(crows), spec, EVAL_SEED),
                                          f"{key}_aopc_curves")

        # ---- 5. AGF ablation per image (Full / Equal / Fixed) --------------
        if abl:
            _ai = list(abl.get("idx", []))
            arows = []
            for vk in ("agf_full", "agf_equal", "agf_fixed"):
                ins = list(abl[vk]["ins"]); dls = list(abl[vk]["del"])
                assert len(ins) == len(dls) == len(_ai), "ablation arms misaligned"
                for i, idx in enumerate(_ai):
                    arows.append({
                        "backbone": disp, "variant": vk, "img_idx": int(idx),
                        "label": int(sample_labels[idx]),
                        "label_name": class_names[int(sample_labels[idx])],
                        "aopc_ins": float(ins[i]), "aopc_del_raw": float(dls[i]),
                        "aopc_del_comp": 1.0 - float(dls[i]),
                        "fixed_weights_source": "benchmark_val_split"})
            if arows:
                written["ablation"] = _pi_write(_tag(pd.DataFrame(arows), spec, ABL_SEED),
                                                f"{key}_ablation_per_image")

        log(f"[per-image] {disp}: wrote {len(written)} file(s) to "
            f"Statistics/per_image/ -> {sorted(written)}")
    except Exception as e:
        # Persistence must never destroy a finished evaluation.
        log(f"[per-image][FAIL] {spec.get('display','?')}: "
            f"{type(e).__name__}: {e}")
        traceback.print_exc()
    return written


print(f"Per-image persistence ready -> {PER_IMAGE_DIR}")

Per-image persistence ready -> /kaggle/working/Statistics/per_image


## §13–14 Per-backbone descriptive tables and figures

Means, standard deviations, per-class means, runtime and the AGF/Equal/Fixed ablation means. Significance
tests are computed in §15. Classification figures in §14 reuse the benchmark's TTA predictions and are
not publication outputs.

In [16]:
# ── §13  Per-backbone descriptive tables (CSV + LaTeX + Excel sheet) ──────
# [FINAL] DESCRIPTIVE ONLY. Inferential statistics are computed once, in §15,
# from the per-image CSVs (the v7.1 per-backbone paired_compare tables are gone).
def _ms(vals):
    v = np.asarray(vals, dtype=np.float64)
    v = v[~np.isnan(v)]                       # drop excluded/degenerate images
    return (float(v.mean()), float(v.std())) if v.size else (float("nan"), float("nan"))

def _n_valid(vals):
    """How many images actually contributed. Robustness excludes NaN rows, and
    an excluded set is NOT random (it is exactly the images where a method
    collapsed), so the surviving n belongs in the table, not in a footnote."""
    v = np.asarray(vals, dtype=np.float64)
    return int(np.count_nonzero(~np.isnan(v)))

def build_model_tables(key_disp, acc, abl):
    """Return {name: DataFrame}. Saves every table and registers Excel sheets."""
    tag = key_disp.replace(" ", "").replace("-", "")
    out = {}

    # ---- Table: XAI metric comparison -------------------------------------
    _ENT_MAX = float(np.log(IMG_SIZE * IMG_SIZE))    # ln(50176) = 10.823
    rows = []
    for k in ORDER:
        a = acc[k]
        ins_m, ins_s = _ms(a.ins_aucs)
        del_raw_m, del_raw_s = _ms(a.del_aucs)
        del_comp_m, del_comp_s = _ms([1.0 - x for x in a.del_aucs])   # higher=better
        sens_m, sens_s = _ms(a.sens)
        spar_m, spar_s = _ms(a.sparsity)
        gini_m, gini_s = _ms(a.gini)
        ent_m, ent_s = _ms(a.entropy)
        rt_m, rt_s = _ms(a.runtime_ms)
        # point estimate below is a MEAN, so the CI must resample the mean
        ci_i = bootstrap_ci(a.ins_aucs, stat=np.mean)
        rows.append({
            "Method": DISP[k],
            "n_images": int(a.n_ok),
            "AOPC_Ins": ins_m, "AOPC_Ins_std": ins_s,
            "AOPC_Ins_CI_lo": ci_i[0], "AOPC_Ins_CI_hi": ci_i[1],
            # FIX: this column stores 1 - del_auc, i.e. HIGHER IS BETTER. It
            # used to be named "AOPC_Del" and was then printed in Section 20's
            # S1 under a header of "AOPC-Del(down-arrow)" -- an inverted sign,
            # which made the WORST method read as best in that column. The
            # name now states the direction so the LaTeX cannot drift again.
            "AOPC_Del_comp": del_comp_m, "AOPC_Del_comp_std": del_comp_s,
            # raw deletion AUC kept alongside it: lower IS better here
            "AOPC_Del_raw": del_raw_m, "AOPC_Del_raw_std": del_raw_s,
            "Robustness_Sens": sens_m, "Robustness_Sens_std": sens_s,
            "Robustness_n": _n_valid(a.sens),
            "Sparsity": spar_m, "Sparsity_std": spar_s,
            "Gini": gini_m, "Gini_std": gini_s,
            "Entropy": ent_m, "Entropy_std": ent_s,
            # Raw Shannon entropy saturates: ln(224*224) = 10.823 and every
            # working method lands at 10.1-10.7, i.e. 93-99% of maximum, so it
            # has no discriminative power at this resolution. Report the
            # normalised form (and say in the text that it is saturated).
            "Entropy_norm": ent_m / _ENT_MAX if np.isfinite(ent_m) else np.nan,
            "Runtime_ms": rt_m, "Runtime_ms_std": rt_s,
        })
    df_metrics = pd.DataFrame(rows)
    out["xai_metrics"] = df_metrics

    # (inferential statistics: see §15 -> Statistics/stats_primary.csv, stats_secondary.csv)

    # ---- Table: AGF weighting ablation -----------------------------------
    abl_disp = {"agf_full": "AGF-Full (Proposed)",
                "agf_equal": "AGF-Equal weights (uniform)",
                "agf_fixed": "AGF-Fixed weights (validation-calibrated)"}
    abl_rows = []
    for k in ["agf_full", "agf_equal", "agf_fixed"]:
        im, is_ = _ms(abl[k]["ins"])
        dm, ds_ = _ms([1.0 - x for x in abl[k]["del"]])
        abl_rows.append({"Variant": abl_disp[k], "n_images": len(abl[k]["ins"]),
                         "AOPC_Ins": im, "AOPC_Ins_std": is_,
                         "AOPC_Del_comp": dm, "AOPC_Del_comp_std": ds_})
    out["ablation"] = pd.DataFrame(abl_rows)

    # ---- Table: per-class AOPC-Ins ----------------------------------------
    pc_rows = []
    for k in ORDER:
        a = acc[k]
        row = {"Method": DISP[k]}
        for ci, cn in enumerate(class_names):
            vals = [v for v, lb in zip(a.ins_aucs, a.labels) if lb == ci]
            row[cn] = float(np.mean(vals)) if vals else float("nan")
            row[f"n_{cn}"] = len(vals)
        pc_rows.append(row)
    out["per_class"] = pd.DataFrame(pc_rows)

    # ---- Table: runtime ----------------------------------------------------
    rt_rows = []
    for k in ORDER:
        m, s = _ms(acc[k].runtime_ms)
        rt_rows.append({"Method": DISP[k], "Runtime_ms_mean": m, "Runtime_ms_std": s})
    df_rt = pd.DataFrame(rt_rows)
    # Cost ratio vs the cheapest method: AGF must run all three base methods,
    # so this is typically ~30x. Confront it in the text; a reviewer will.
    _rt_min = df_rt["Runtime_ms_mean"].min()
    df_rt["Cost_vs_cheapest"] = df_rt["Runtime_ms_mean"] / _rt_min if _rt_min else np.nan
    out["runtime"] = df_rt

    # ---- persist ----------------------------------------------------------
    for name, df in out.items():
        save_table(df, f"{tag}_{name}", subdir=key_disp)
        register_excel(f"{key_disp[:12]}_{name}", df)
    log(f"[{key_disp}] tables written: {list(out.keys())}")
    return out

print("Table builder ready.")

Table builder ready.


In [17]:
# ── §14  Per-backbone figures (PNG + PDF + SVG @ 300 DPI) ──────────────
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
                             precision_recall_curve, average_precision_score)
from sklearn.preprocessing import label_binarize

def _tensor_to_rgb(t):
    mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
    img = t.permute(1, 2, 0).cpu().numpy()
    return np.clip(img * std + mean, 0, 1)

def _overlay(t, amap, alpha=0.5):
    base = _tensor_to_rgb(t)
    u8 = (np.clip(amap, 0, 1) * 255).astype(np.uint8)
    hm = cv2.applyColorMap(u8, cv2.COLORMAP_JET)
    hm = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB) / 255.0
    hm = cv2.resize(hm, (IMG_SIZE, IMG_SIZE))
    return np.clip(alpha * hm + (1 - alpha) * base, 0, 1).astype(np.float32)

def _metric_means(acc):
    M = {}
    for k in ORDER:
        a = acc[k]
        M[k] = {
            "AOPC-Ins": np.mean(a.ins_aucs) if a.ins_aucs else np.nan,
            "AOPC-Del": np.mean([1 - x for x in a.del_aucs]) if a.del_aucs else np.nan,
            "Gini": np.mean(a.gini) if a.gini else np.nan,
            "Sparsity": np.mean(a.sparsity) if a.sparsity else np.nan,
            "Robustness": float(np.nanmean(a.sens)) if a.sens and not np.all(np.isnan(a.sens)) else np.nan,   # lower better
            "Entropy": np.mean(a.entropy) if a.entropy else np.nan,
        }
    return M

def _try(fn, name):
    try:
        fn()
    except Exception as e:
        log(f"  [FIG FAIL] {name}: {type(e).__name__}: {e}")


def build_model_figures(spec, eval_model, target_layer, acc, abl,
                        agf_weight_rows, disagreement, preds_bundle):
    disp = spec["display"]; sub = disp
    y_true, y_pred, y_prob = preds_bundle
    x_axis = np.linspace(AOPC_PATCH_PCT, AOPC_N_STEPS * AOPC_PATCH_PCT, AOPC_N_STEPS)

    # 1) AOPC insertion/deletion curves
    def _f_aopc():
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        for ax, attr, ttl in [(axes[0], "ins_curves", "Insertion (higher = better)"),
                              (axes[1], "del_curves", "Deletion (lower = better)")]:
            for k in ORDER:
                arr = np.array(getattr(acc[k], attr))
                if arr.size == 0: continue
                m, s = arr.mean(0), arr.std(0)
                ax.plot(x_axis, m, color=MCOLOR[k], lw=2, marker="o", ms=3, label=DISP[k])
                ax.fill_between(x_axis, m - s, m + s, color=MCOLOR[k], alpha=0.15)
            ax.set_title(ttl, fontweight="bold"); ax.set_xlabel("Pixels perturbed")
            ax.set_ylabel("Class confidence"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
            ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        fig.suptitle(f"AOPC Curves — {disp}", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "aopc_curves", sub)
    _try(_f_aopc, "aopc_curves")

    # 2) AOPC comparison bar (Ins + Del)
    def _f_aopc_bar():
        ins = [np.mean(acc[k].ins_aucs) for k in ORDER]
        dl  = [np.mean([1 - x for x in acc[k].del_aucs]) for k in ORDER]
        xp = np.arange(len(ORDER)); w = 0.38
        fig, ax = plt.subplots(figsize=(9, 4.5))
        ax.bar(xp - w/2, ins, w, label="AOPC-Ins", color="#457B9D", edgecolor="black", lw=.5)
        ax.bar(xp + w/2, dl,  w, label="AOPC-Del", color="#E9C46A", edgecolor="black", lw=.5)
        ax.set_xticks(xp); ax.set_xticklabels([DISP[k] for k in ORDER], rotation=15, ha="right")
        ax.set_ylabel("AOPC (higher = better)"); ax.legend(); ax.grid(axis="y", alpha=.3)
        ax.set_title(f"AOPC Comparison — {disp}", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "aopc_comparison_bar", sub)
    _try(_f_aopc_bar, "aopc_comparison_bar")

    # 3) Boxplot + 4) Violin of per-image AOPC-Ins
    def _f_box():
        data = [acc[k].ins_aucs for k in ORDER]
        fig, ax = plt.subplots(figsize=(9, 4.5))
        bp = ax.boxplot(data, labels=[DISP[k] for k in ORDER], patch_artist=True)
        for patch, k in zip(bp["boxes"], ORDER): patch.set_facecolor(MCOLOR[k])
        ax.set_ylabel("AOPC-Ins"); ax.grid(axis="y", alpha=.3)
        ax.set_xticklabels([DISP[k] for k in ORDER], rotation=15, ha="right")
        ax.set_title(f"AOPC-Ins Distribution (Boxplot) — {disp}", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "boxplot_aopc_ins", sub)
    _try(_f_box, "boxplot_aopc_ins")

    def _f_violin():
        data = [acc[k].ins_aucs for k in ORDER]
        fig, ax = plt.subplots(figsize=(9, 4.5))
        vp = ax.violinplot(data, showmeans=True)
        for i, b in enumerate(vp["bodies"]): b.set_facecolor(MCOLOR[ORDER[i]]); b.set_alpha(.6)
        ax.set_xticks(range(1, len(ORDER) + 1))
        ax.set_xticklabels([DISP[k] for k in ORDER], rotation=15, ha="right")
        ax.set_ylabel("AOPC-Ins"); ax.grid(axis="y", alpha=.3)
        ax.set_title(f"AOPC-Ins Distribution (Violin) — {disp}", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "violin_aopc_ins", sub)
    _try(_f_violin, "violin_aopc_ins")

    # 5)-7) REMOVED in v6: radar chart, metric heatmap, metric correlation.
    #   All three normalised 6 metrics across only 4 methods and rendered
    #   them as if they were a multivariate result. The correlation matrix
    #   in particular computed Pearson r over FOUR data points per pair,
    #   which is not a statistic. All three also inverted 'Entropy' as
    #   lower-is-better, but entropy here is saturated at 93-99% of ln(N)
    #   and carries no signal to invert. They were AGF-era diagnostics and
    #   none of them appeared in KEEP_FIGURES, so every run generated and
    #   then silently discarded them.

    # 8) Runtime bar
    def _f_runtime():
        m = [np.mean(acc[k].runtime_ms) for k in ORDER]
        s = [np.std(acc[k].runtime_ms) for k in ORDER]
        fig, ax = plt.subplots(figsize=(9, 4.5))
        ax.bar(range(len(ORDER)), m, yerr=s, color=[MCOLOR[k] for k in ORDER],
               capsize=4, edgecolor="black", lw=.5)
        ax.set_xticks(range(len(ORDER))); ax.set_xticklabels([DISP[k] for k in ORDER],
                                                             rotation=15, ha="right")
        ax.set_ylabel("Runtime per image (ms)"); ax.grid(axis="y", alpha=.3)
        ax.set_title(f"Runtime Comparison — {disp}", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "runtime_bar", sub)
    _try(_f_runtime, "runtime_bar")

    # 9) AGF fusion-weight distribution
    def _f_weights():
        if not agf_weight_rows: return
        names = [n for n in agf_weight_rows[0].keys() if n.startswith("w_")]
        data = [[r[n] for r in agf_weight_rows] for n in names]
        fig, ax = plt.subplots(figsize=(8, 4.5))
        vp = ax.violinplot(data, showmeans=True)
        ax.set_xticks(range(1, len(names) + 1)); ax.set_xticklabels([n[2:] for n in names])
        ax.axhline(1 / 3, ls="--", color="grey", lw=0.8)
        ax.set_ylabel("AGF agreement weight"); ax.grid(axis="y", alpha=.3)
        ax.set_title(f"AGF Agreement-Weight Distribution — {disp}", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "agf_weight_distribution", sub)
    _try(_f_weights, "agf_weight_distribution")

    # 10) Ablation bar (AGF variants)
    def _f_ablation():
        keys = ["agf_full", "agf_equal", "agf_fixed"]
        lbls = ["AGF-Full\n(Proposed)", "AGF-Equal", "AGF-Fixed\n(val-calibrated)"]
        cols = ["#F4A261", "#B0B0B0", "#457B9D"]
        ins_m = [np.mean(abl[k]["ins"]) for k in keys]
        ins_s = [np.std(abl[k]["ins"]) for k in keys]
        del_m = [np.mean([1 - x for x in abl[k]["del"]]) for k in keys]
        fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
        axes[0].bar(range(3), ins_m, yerr=ins_s, color=cols, capsize=4, edgecolor="black", lw=.5)
        axes[0].set_title("AOPC-Ins (higher = better)", fontweight="bold"); axes[0].set_ylabel("AOPC-Ins")
        axes[1].bar(range(3), del_m, color=cols, edgecolor="black", lw=.5)
        axes[1].set_title("AOPC-Del (higher = better)", fontweight="bold"); axes[1].set_ylabel("AOPC-Del")
        for ax in axes:
            ax.set_xticks(range(3)); ax.set_xticklabels(lbls, fontsize=8); ax.grid(axis="y", alpha=.3)
        fig.suptitle(f"AGF Weighting Ablation — {disp}", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "ablation_bar", sub)
    _try(_f_ablation, "ablation_bar")

    # 11) Per-image improvement: AGF vs best baseline (AOPC-Ins)
    def _f_perimg():
        p_map = {i: v for i, v in zip(acc[PROPOSED].sample_idxs, acc[PROPOSED].ins_aucs)}
        base_best = {}
        for i in p_map:
            vals = []
            for bk in BASE_KEYS:          # [FINAL] individual methods only
                om = {j: v for j, v in zip(acc[bk].sample_idxs, acc[bk].ins_aucs)}
                if i in om: vals.append(om[i])
            if vals: base_best[i] = max(vals)
        shared = sorted(set(p_map) & set(base_best))
        imp = np.array([p_map[i] - base_best[i] for i in shared])
        imp.sort()
        fig, ax = plt.subplots(figsize=(9, 4.5))
        colors = ["#2A9D8F" if v >= 0 else "#E63946" for v in imp]
        ax.bar(range(len(imp)), imp, color=colors)
        ax.axhline(0, color="black", lw=.8)
        ax.set_xlabel("Images (sorted)"); ax.set_ylabel("AGF AOPC-Ins − per-image best individual method")
        ax.set_title(f"Per-Image Improvement of AGF — {disp} "
                     f"({100*np.mean(imp>=0):.0f}% wins)", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "per_image_improvement", sub)
    _try(_f_perimg, "per_image_improvement")

    # 12) REMOVED in v6: disagreement-vs-AGF-advantage scatter.
    #   It plotted AGF's AOPC advantage over DISAGREE_REF against
    #   inter-method disagreement. DISAGREE_REF was originally GAF; when GAF
    #   was retired the fallback silently became GradCAM++ — which on
    #   VGG16-BN meant correlating against an identically-zero map, and on
    #   ResNet50 against a map that collapsed on ~75% of images. The figure
    #   was uninterpretable and its config key is now deleted.

    # 13) Qualitative comparison grid (1 correct image / class), AGF highlighted
    def _f_grid():
        """Per-backbone class x method overlay grid.

        THREE FIXES vs the previous version
        -----------------------------------
        (1) THE CRASH. The old code ended with `for sp in axes[r][5].spines`
            -- a hardcoded column 5 left over from the six-column
            AGF/the retired fusion variants era. With GAF and a reliability-weighted variant retired, ncols is 5
            (Original + 4 methods), so column index 5 is out of bounds and
            this raised `IndexError: index 5 is out of bounds for axis 0
            with size 5` on ALL THREE backbones. Every run of this notebook
            produced zero qualitative figures. For a medical-imaging venue
            that is disqualifying on its own. The highlight column is now
            derived from ORDER, so it cannot drift again if the method list
            changes.
        (2) NO DOUBLE COMPUTATION. It generated the three base maps and then
            called AGF.fuse(), which recomputed all three. Now it computes
            each once and fuses the finished maps, matching Section 9.
        (3) HOOK LEAK. GradCAM++ hooks were only removed on the happy path
            via a bare `finally: gc_gen.remove()` inside the loop; the
            generators created for a row that raised earlier were never
            cleaned up. Cleanup is now unconditional per row.

        the retired fusion variants branches removed; the classes no longer exist.
        """
        repr_idx = {}
        for i, (p, lb) in enumerate(zip(sample_paths, sample_labels)):
            if lb in repr_idx: continue
            t = load_image_tensor(p)
            with torch.inference_mode():
                pr = int(eval_model(t.unsqueeze(0).to(DEVICE)).argmax(1).item())
            if pr == lb: repr_idx[lb] = i
            if len(repr_idx) == num_classes: break
        rows_k = list(ORDER)
        col_titles = ["Original"] + [DISP[k] + (" *" if k == PROPOSED else "")
                                     for k in rows_k]
        ncols = len(col_titles)
        nrows = len(repr_idx)
        # column index of the proposed method, +1 for the "Original" column.
        # Derived, never hardcoded: this is what broke before.
        hl_col = rows_k.index(PROPOSED) + 1 if PROPOSED in rows_k else None
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.2, nrows * 2.4),
                                 squeeze=False)
        fig.patch.set_facecolor("#0f0f0f")
        for r, cls_idx in enumerate(sorted(repr_idx)):
            t = load_image_tensor(sample_paths[repr_idx[cls_idx]])
            gc_gen = sg_gen = ig_gen = None
            try:
                gc_gen, sg_gen, ig_gen = make_base_generators(eval_model, target_layer)
                fd = fusion_dict(gc_gen, sg_gen, ig_gen)
                with torch.inference_mode(False):
                    gc_m, pred = gc_gen.generate(t)
                    sg_m, _ = sg_gen.generate(t, class_idx=pred)
                    ig_m, _ = ig_gen.generate(t, class_idx=pred)
                    _pre = {"gradcam":    resize_map(gc_m, (IMG_SIZE, IMG_SIZE)),
                            "smoothgrad": resize_map(sg_m, (IMG_SIZE, IMG_SIZE)),
                            "ig":         resize_map(ig_m, (IMG_SIZE, IMG_SIZE))}
                    agf_m, indiv, _w, _a = \
                        AgreementGuidedFusion(fd).fuse_maps(_pre)
            finally:
                if gc_gen is not None: gc_gen.remove()
                del gc_gen, sg_gen, ig_gen
            mp = {"gradcam_pp": indiv["gradcam"], "ig": indiv["ig"],
                  "smoothgrad": indiv["smoothgrad"],
                  "equal": weighted_arith(indiv, EQUAL_WEIGHTS), "agf": agf_m}
            panel = [None] + [mp[k] for k in rows_k]
            for c in range(ncols):
                ax = axes[r][c]; ax.set_facecolor("#0f0f0f")
                ax.imshow(_tensor_to_rgb(t) if c == 0 else _overlay(t, panel[c]))
                if r == 0:
                    ax.set_title(col_titles[c], fontsize=9, fontweight="bold", color="white")
                if c == 0:
                    ax.set_ylabel(class_names[cls_idx].title(), fontsize=9,
                                  fontweight="bold", color="#F4A261")
                ax.set_xticks([]); ax.set_yticks([])
            if hl_col is not None:
                for sp in axes[r][hl_col].spines.values():
                    sp.set_edgecolor("#FFD700"); sp.set_linewidth(2.5)
            del t, mp, panel, _pre, indiv; free_gpu()
        fig.suptitle(f"XAI Attribution Maps — {disp} (AGF = Proposed ★)",
                     color="white", fontweight="bold", y=1.005)
        plt.tight_layout(); plt.subplots_adjust(hspace=.05, wspace=.05)
        savefig_all(fig, "qualitative_grid", sub)
    _try(_f_grid, "qualitative_grid")

    # 14) Confusion matrix / 15) ROC / 16) PR  (classification)
    #     [FINAL] These use the benchmark's TTA x5 predictions when the npz is attached
    #     (get_predictions); they duplicate benchmarking.ipynb figures and are NOT
    #     in the publication whitelist.
    def _f_confusion():
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(5.5, 5))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(num_classes)); ax.set_xticklabels(class_names, rotation=30, ha="right")
        ax.set_yticks(range(num_classes)); ax.set_yticklabels(class_names)
        for i in range(num_classes):
            for j in range(num_classes):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
        fig.colorbar(im, ax=ax); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"Confusion Matrix — {disp}", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "confusion_matrix", sub)
    _try(_f_confusion, "confusion_matrix")

    def _f_roc():
        yb = label_binarize(y_true, classes=list(range(num_classes)))
        fig, ax = plt.subplots(figsize=(6, 5))
        for c in range(num_classes):
            fpr, tpr, _ = roc_curve(yb[:, c], y_prob[:, c])
            ax.plot(fpr, tpr, lw=1.6, label=f"{class_names[c]} (AUC={auc(fpr,tpr):.3f})")
        ax.plot([0, 1], [0, 1], "k--", lw=.8)
        ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(fontsize=8)
        ax.set_title(f"ROC (one-vs-rest) — {disp}", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "roc_curve", sub)
    _try(_f_roc, "roc_curve")

    def _f_pr():
        yb = label_binarize(y_true, classes=list(range(num_classes)))
        fig, ax = plt.subplots(figsize=(6, 5))
        for c in range(num_classes):
            pr, rc, _ = precision_recall_curve(yb[:, c], y_prob[:, c])
            ap = average_precision_score(yb[:, c], y_prob[:, c])
            ax.plot(rc, pr, lw=1.6, label=f"{class_names[c]} (AP={ap:.3f})")
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend(fontsize=8)
        ax.set_title(f"Precision-Recall — {disp}", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "pr_curve", sub)
    _try(_f_pr, "pr_curve")

    log(f"[{disp}] figures written to Figures/{disp}/")

print("Figure builder ready.")

Figure builder ready.


## §12 Main evaluation — every backbone, one on the GPU at a time

Per backbone: load → verify target layer → **calibrate on the validation split** → main faithfulness loop
(`EVAL_N`) → robustness loop (`GINI_SENS_N`) → Full/Equal/Fixed ablation (`ABLATION_N`) → descriptive
tables and figures → **per-image CSVs** → summary JSON (with subset seeds and index hashes) → free GPU.
No statistics are computed here.

In [18]:
# ── §12  Per-backbone orchestration + automatic loop ───────────────────────
#   [FINAL] Changes vs v7.1 (all REQUIRE RERUN of this section):
#     * benchmark-identical preprocessing (§4), Equal Fusion arm, post-processed
#       robustness maps, AGF-Fixed weights from the validation split (§11c);
#     * index lists and their hashes are written into each summary JSON;
#     * no inferential statistics here (moved to §15).
def _classification_metrics(spec, meta, preds_bundle):
    """Prefer benchmarking JSON; fall back to checkpoint meta or recomputed."""
    if BENCH_JSON and "models" in BENCH_JSON:
        for _n, _r in BENCH_JSON["models"].items():
            if _r.get("backbone") == meta.get("backbone_name") or _n == spec["display"]:
                m = _r.get("metrics", {})
                if m: return {k: m.get(k) for k in ("accuracy", "f1", "mean_auc", "mcc")}
    y_true, y_pred, y_prob = preds_bundle
    from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "f1": float(f1_score(y_true, y_pred, average="macro")),
            "mean_auc": meta.get("best_auc"),
            "mcc": float(matthews_corrcoef(y_true, y_pred))}

def evaluate_one_model(spec):
    disp = spec["display"]
    log("=" * 70); log(f"▶ EVALUATING {disp}"); log("=" * 70)
    seed_everything(SEED)
    model, eval_model, in_features, backbone_name, meta = load_model(spec)
    target_layer = get_target_layer(model, backbone_name)
    # Probe the resolved layer BEFORE spending 2h of GPU on it. This asserts
    # spatial extent >= 7x7 and >= 99% non-negative activations, i.e. exactly
    # the two conditions whose violation produced identically-zero GradCAM++
    # maps on VGG16-BN (1x1 classifier conv) and collapsed maps on ~75% of
    # ResNet50 images (pre-BatchNorm signed activations). It raises rather
    # than warns: a silent degenerate map is what contaminated the last run.
    _tl_shape = assert_spatial_target(eval_model, target_layer,
                                      IMG_SIZE, DEVICE, label=disp)
    log(f"[{disp}] backbone={backbone_name} | target={target_layer.__class__.__name__} "
        f"{_tl_shape} | VRAM={torch.cuda.memory_allocated()/1e9:.2f}GB")

    eval_idxs = stratified_sample(EVAL_N, EVAL_SEED)
    gini_idxs = stratified_sample(GINI_SENS_N, SENS_SEED)
    abl_idxs  = stratified_sample(ABLATION_N, ABL_SEED)

    # [FINAL] calibration (split A, validation images) BEFORE any test metric
    fixed_w = run_calibration(eval_model, target_layer, spec)

    acc, agf_weight_rows, disagreement = run_main_loop(eval_model, target_layer, eval_idxs)
    acc = run_sens_loop(eval_model, target_layer, gini_idxs, acc)
    abl = run_agf_ablation(eval_model, target_layer, abl_idxs, fixed_w)
    main_mean_w = None
    if agf_weight_rows:
        _wk = [n for n in agf_weight_rows[0] if n.startswith("w_")]
        main_mean_w = {n[2:]: float(np.mean([r[n] for r in agf_weight_rows])) for n in _wk}

    preds_bundle = get_predictions(spec, eval_model)
    tables = build_model_tables(disp, acc, abl)
    build_model_figures(spec, eval_model, target_layer, acc, abl,
                        agf_weight_rows, disagreement, preds_bundle)

    # ---- NEW (v7.1): persist every per-image scalar before `acc` is freed --
    # Additive only. Runs AFTER all tables/figures/summaries are built, so it
    # cannot influence any reported quantity; it merely writes to disk what
    # the lines below were about to discard. See Section 9b.
    dump_per_image_records(spec, acc, agf_weight_rows, disagreement,
                           abl=abl, abl_idxs=abl_idxs)

    # ---- compact, tensor-free summary for cross-architecture analysis -----
    def _ihash(ix): return hashlib.sha256(",".join(map(str, ix)).encode()).hexdigest()[:16]
    summary = {"display": disp, "backbone": backbone_name, "config_version": CONFIG_VERSION,
               "checkpoint": meta.get("ckpt_path"),
               "target_layer": f"backbone.{TARGET_LAYERS[backbone_name]}", "target_shape": list(_tl_shape),
               "classification_source": "benchmark_results.json (TTA x5) if attached",
               "classification": _classification_metrics(spec, meta, preds_bundle),
               "agf_fixed_weights_calibration": fixed_w,
               "agf_mean_weights_eval_descriptive": main_mean_w,
               "subsets": {"eval": {"seed": EVAL_SEED, "n": len(eval_idxs), "sha": _ihash(eval_idxs)},
                           "sens": {"seed": SENS_SEED, "n": len(gini_idxs), "sha": _ihash(gini_idxs)},
                           "ablation": {"seed": ABL_SEED, "n": len(abl_idxs), "sha": _ihash(abl_idxs)},
                           "calibration": {"seed": CALIB_SEED, "n": len(CALIB_POSITIONS),
                                           "sha": _ihash(CALIB_POSITIONS)}},
               "n_ok": {k: int(acc[k].n_ok) for k in ORDER},
               "metrics": {}, "ablation": {},
               "note": "descriptive means only; inference in Statistics/stats_primary.csv"}
    for k in ORDER:
        a = acc[k]
        summary["metrics"][k] = {
            "AOPC_Ins": float(np.mean(a.ins_aucs)) if a.ins_aucs else None,
            # renamed: this stores 1 - del_auc, i.e. HIGHER IS BETTER. Under
            # the old name "AOPC_Del" it was printed in Section 20 under a
            # down-arrow header, inverting the reported ranking.
            "AOPC_Del_comp": float(np.mean([1 - x for x in a.del_aucs])) if a.del_aucs else None,
            "AOPC_Del_raw": float(np.mean(a.del_aucs)) if a.del_aucs else None,
            "Robustness_n": int(np.isfinite(np.asarray(a.sens, dtype=float)).sum()) if a.sens else 0,
            "Robustness": (float(np.nanmean(a.sens)) if a.sens and not np.all(np.isnan(a.sens)) else None),
            "Sparsity": float(np.mean(a.sparsity)) if a.sparsity else None,
            "Gini": float(np.mean(a.gini)) if a.gini else None,
            "Entropy": float(np.mean(a.entropy)) if a.entropy else None,
            "Runtime_ms": float(np.mean(a.runtime_ms)) if a.runtime_ms else None,
        }
    for k in ["agf_full", "agf_equal", "agf_fixed"]:
        summary["ablation"][k] = {
            "AOPC_Ins": float(np.mean(abl[k]["ins"])) if abl[k]["ins"] else None,
            "AOPC_Del_comp": float(np.mean([1 - x for x in abl[k]["del"]])) if abl[k]["del"] else None}

    with open(os.path.join(DIRS["Results"], f"{spec['key']}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2, default=float)

    # ---- release everything before the next backbone ----------------------
    del model, eval_model, target_layer, acc, agf_weight_rows, disagreement, _tl_shape, abl_idxs
    del preds_bundle, tables, abl
    plt.close("all"); free_gpu()
    log(f"[{disp}] done. VRAM now {torch.cuda.memory_allocated()/1e9:.2f}GB")
    return summary


ALL_RESULTS = {}
for spec in MODELS:
    if spec["key"] not in RUN_MODELS:
        log(f"(skip {spec['display']} — not in RUN_MODELS)"); continue
    try:
        ALL_RESULTS[spec["key"]] = evaluate_one_model(spec)
    except Exception as e:
        log(f"[FATAL] {spec['display']}: {type(e).__name__}: {e}")
        traceback.print_exc()
    plt.close("all"); free_gpu()

log(f"\nEvaluated backbones: {list(ALL_RESULTS.keys())}")
_missing_bb = [k for k in RUN_MODELS if k not in ALL_RESULTS]
if _missing_bb:
    log(f"[WARN] backbones that FAILED in §12: {_missing_bb} -- downstream tables will be incomplete")
mark_section("12_main_evaluation", "completed" if not _missing_bb else f"partial (failed: {_missing_bb})")

[13:24:46] ======================================================================
[13:24:46] ▶ EVALUATING VGG16-BN
[13:24:46] ======================================================================
[13:24:48] [target-layer] [VGG16-BN] Sequential -> (1, 512, 7, 7) | non-negative 100.0% | OK
[13:24:48] [VGG16-BN] backbone=vgg16_bn | target=Sequential (1, 512, 7, 7) | VRAM=0.59GB


Calibration (val split):   0%|          | 0/100 [00:00<?, ?it/s]

[13:28:33] [calib] VGG16-BN: n=100 fixed weights = gradcam=0.3413, smoothgrad=0.3302, ig=0.3284


Main eval:   0%|          | 0/200 [00:00<?, ?it/s]

Robustness:   0%|          | 0/50 [00:00<?, ?it/s]

AGF ablation:   0%|          | 0/100 [00:00<?, ?it/s]

... (9 outputs left out)

Main eval:   0%|          | 0/200 [00:00<?, ?it/s]

Robustness:   0%|          | 0/50 [00:00<?, ?it/s]

AGF ablation:   0%|          | 0/100 [00:00<?, ?it/s]

[17:06:34] [ResNet50] tables written: ['xai_metrics', 'ablation', 'per_class', 'runtime']
[17:06:57] [ResNet50] figures written to Figures/ResNet50/
[17:06:57] [per-image] ResNet50: wrote 5 file(s) to Statistics/per_image/ -> ['ablation', 'curves', 'main', 'sens', 'weights']
[17:06:57] [ResNet50] done. VRAM now 0.07GB
[17:06:58] 
Evaluated backbones: ['vgg16', 'densenet121', 'resnet50']


## §15 Statistical analysis — computed once, from the raw per-image CSVs

**Procedure (pre-declared in §1):**

* Unit of analysis: one test image; samples are paired (same image, same fixed class, same tie-break
  permutation, same perturbations).
* Test: two-sided Wilcoxon signed-rank (`zero_method="wilcox"`).
* **Primary family** — AGF vs **Equal Fusion** × {insertion, deletion (complement), robustness} ×
  3 backbones = 9 tests, Holm-corrected together.
* **Secondary family** — AGF vs each individual method (Grad-CAM++, IG, SmoothGrad) × 3 metrics ×
  3 backbones = 27 tests, Holm-corrected together.
* Reported for every test: *n*, mean of each method, **mean of paired differences** (identical to the
  difference of means on the same *n* images) with a 95 % bootstrap CI, **median of paired differences**
  with its own 95 % bootstrap CI, matched-pairs rank-biserial *r* (the effect size that belongs to the
  Wilcoxon test; its sign is the direction the test detects), counts of images favouring each side,
  raw and Holm p, and a `sign_concordant` flag.
* Direction is stated only for Holm-significant tests and is taken from the rank-biserial sign combined
  with the metric direction (↑ insertion/deletion-complement, ↓ sensitivity).
* Every result is reported **per metric and per backbone**; no combined verdict is produced.

Consistency check: AGF and Equal Fusion values of images that appear in both the main subset and the
Full/Equal/Fixed ablation subset must be identical (same deterministic computation); mismatches are logged.

In [19]:
# ── §15  Statistical analysis from per-image CSVs (single inference engine) ─
STATS_DIR = DIRS["Statistics"]

def _stable_seed(*parts):
    return zlib.crc32("|".join(map(str, parts)).encode()) & 0x7FFFFFFF

def load_metric_matrix(key):
    """Return {metric: DataFrame[index=img_idx, columns=method_key]} for one backbone."""
    pi = os.path.join(PER_IMAGE_DIR, f"{key}_per_image.csv")
    ps = os.path.join(PER_IMAGE_DIR, f"{key}_per_image_sens.csv")
    if not (os.path.exists(pi) and os.path.exists(ps)):
        raise FileNotFoundError(f"per-image CSVs missing for {key}: {pi} / {ps}")
    main, sens = pd.read_csv(pi), pd.read_csv(ps)
    for df, name in ((main, pi), (sens, ps)):
        if "config_version" in df.columns:
            bad = set(df["config_version"].astype(str)) - {CONFIG_VERSION}
            assert not bad, f"{name} was produced under config {bad}, not {CONFIG_VERSION}"
        else:
            raise AssertionError(f"{name} has no config_version column: produced by an older version")
    out = {}
    for m in ("aopc_ins", "aopc_del_comp", "aopc_del_raw"):
        out[m] = main.pivot_table(index="img_idx", columns="method_key", values=m, aggfunc="first")
    out["sensitivity"] = sens.pivot_table(index="img_idx", columns="method_key",
                                          values="sensitivity", aggfunc="first")
    out["_labels"] = main.drop_duplicates("img_idx").set_index("img_idx")["label"]
    return out


def paired_test(a, b, direction, seed):
    """Paired two-sided Wilcoxon + fully labelled location summaries. a = AGF."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    a, b = a[ok], b[ok]
    n = int(a.size)
    row = {"n": n, "n_excluded_nonfinite": int((~ok).sum())}
    if n < 5:
        row.update({"p_raw": np.nan, "test_note": "n<5, not tested"})
        return row
    d = a - b
    row.update({
        "mean_agf": float(a.mean()), "mean_ref": float(b.mean()),
        "mean_paired_diff": float(d.mean()),
        "median_agf": float(np.median(a)), "median_ref": float(np.median(b)),
        "median_paired_diff": float(np.median(d)),
        "n_agf_better": int(((d * direction) > 0).sum()),
        "n_ref_better": int(((d * direction) < 0).sum()),
        "n_tied": int((d == 0).sum()),
        "rank_biserial": paired_rank_biserial(a, b),
    })
    row["mean_diff_CI_lo"], row["mean_diff_CI_hi"] = bootstrap_ci(d, seed=seed, stat=np.mean)
    row["median_diff_CI_lo"], row["median_diff_CI_hi"] = bootstrap_ci(d, seed=seed + 1, stat=np.median)
    if np.all(d == 0):
        row.update({"W": np.nan, "p_raw": 1.0, "test_note": "all paired differences are zero"})
    else:
        W, p = wilcoxon(a, b, zero_method=STATS_ZERO_METHOD, alternative=STATS_ALTERNATIVE)
        row.update({"W": float(W), "p_raw": float(p), "test_note": ""})
    signs = {np.sign(row["mean_paired_diff"]), np.sign(row["median_paired_diff"]),
             np.sign(row["rank_biserial"])} - {0.0}
    row["sign_concordant"] = bool(len(signs) <= 1)
    return row


def run_family(family, ref_keys, metrics, backbones):
    rows = []
    for key in backbones:
        mats = load_metric_matrix(key)
        disp = next(s["display"] for s in MODELS if s["key"] == key)
        for metric in metrics:
            M = mats[metric]
            for ref in ref_keys:
                if PROPOSED not in M.columns or ref not in M.columns:
                    log(f"[stats][WARN] {disp}/{metric}: column {ref} or {PROPOSED} missing"); continue
                seed = _stable_seed(family, key, metric, ref)
                r = paired_test(M[PROPOSED].values, M[ref].values, METRIC_DIRECTION[metric], seed)
                rows.append({"family": family, "backbone": disp, "backbone_key": key,
                             "metric": metric, "metric_label": METRIC_LABEL[metric],
                             "direction": "higher_better" if METRIC_DIRECTION[metric] > 0 else "lower_better",
                             "comparison": f"{DISP[PROPOSED]} vs {DISP[ref]}", "reference_key": ref,
                             "test": f"{STATS_TEST} ({STATS_ALTERNATIVE})", "correction": MULTIPLE_CORRECTION,
                             "bootstrap_n": BOOTSTRAP_N, "bootstrap_seed": seed, **r})
    df = pd.DataFrame(rows)
    tested = df["p_raw"].notna() if len(df) else pd.Series(dtype=bool)
    df["family_size"] = int(tested.sum())
    df["p_holm"] = np.nan; df["significant"] = False
    if tested.any():
        rej, padj, _, _ = multipletests(df.loc[tested, "p_raw"].values, alpha=ALPHA,
                                        method=MULTIPLE_CORRECTION)
        df.loc[tested, "p_holm"] = padj
        df.loc[tested, "significant"] = rej
    df["effect_magnitude_rb"] = df["rank_biserial"].apply(
        lambda r: rb_magnitude(r) if pd.notna(r) else "")

    def _verdict(r):
        if pd.isna(r["p_raw"]):
            return "not tested"
        if not r["significant"]:
            return "not statistically significant"
        agf_better = (r["rank_biserial"] * (1 if r["direction"] == "higher_better" else -1)) > 0
        return "significant: AGF better" if agf_better else "significant: AGF worse"
    df["verdict"] = df.apply(_verdict, axis=1)
    return df


def _sentence(r):
    ref = DISP[r["reference_key"]]
    metric = {"aopc_ins": "insertion AOPC", "aopc_del_comp": "deletion (1 − AOPC-Del)",
              "sensitivity": "max-sensitivity"}[r["metric"]]
    if r["verdict"] == "not tested":
        return f"{r['backbone']}: {metric}, AGF vs {ref}: not tested (n={r['n']})."
    core = (f"mean paired Δ = {r['mean_paired_diff']:+.4f} [{r['mean_diff_CI_lo']:+.4f}, "
            f"{r['mean_diff_CI_hi']:+.4f}], median paired Δ = {r['median_paired_diff']:+.4f} "
            f"[{r['median_diff_CI_lo']:+.4f}, {r['median_diff_CI_hi']:+.4f}], r_rb = {r['rank_biserial']:+.3f}, "
            f"n = {r['n']}, p_raw = {r['p_raw']:.3g}, p_Holm = {r['p_holm']:.3g} "
            f"(family of {r['family_size']})")
    if r["verdict"] == "significant: AGF better":
        head = f"AGF showed a statistically significant improvement in {metric} over {ref}"
    elif r["verdict"] == "significant: AGF worse":
        head = f"AGF was statistically significantly worse than {ref} in {metric}"
    else:
        head = f"the {metric} difference between AGF and {ref} was not statistically significant"
    _ci_excl = ((r["mean_diff_CI_lo"] > 0) or (r["mean_diff_CI_hi"] < 0) or
                (r["median_diff_CI_lo"] > 0) or (r["median_diff_CI_hi"] < 0))
    tail = ("" if (r["sign_concordant"] or not (r["significant"] or _ci_excl)) else
            " [mean, median and rank-based directions disagree; describe the distribution, not a single number]")
    return f"{r['backbone']}: {head}; {core}.{tail}"


_bb_done = [k for k in RUN_MODELS if os.path.exists(os.path.join(PER_IMAGE_DIR, f"{k}_per_image.csv"))]
STATS_PRIMARY   = run_family("primary",   [PRIMARY_REFERENCE],  PRIMARY_METRICS, _bb_done)
STATS_SECONDARY = run_family("secondary", SECONDARY_BASELINES,  PRIMARY_METRICS, _bb_done)
STATS_PRIMARY.to_csv(os.path.join(STATS_DIR, "stats_primary.csv"), index=False)
STATS_SECONDARY.to_csv(os.path.join(STATS_DIR, "stats_secondary.csv"), index=False)
save_table(STATS_PRIMARY, "stats_primary", float_fmt="%.5g")
save_table(STATS_SECONDARY, "stats_secondary", float_fmt="%.5g")
register_excel("stats_primary", STATS_PRIMARY); register_excel("stats_secondary", STATS_SECONDARY)

print("=" * 100); print("PRIMARY FAMILY — AGF vs Equal Fusion (Holm over all rows)"); print("=" * 100)
for m in PRIMARY_METRICS:
    print(f"\n[{METRIC_LABEL[m]}]")
    for _, r in STATS_PRIMARY[STATS_PRIMARY.metric == m].iterrows():
        print("  - " + _sentence(r))
print("\n" + "=" * 100); print("SECONDARY FAMILY — AGF vs individual methods (Holm over all rows)"); print("=" * 100)
for m in PRIMARY_METRICS:
    print(f"\n[{METRIC_LABEL[m]}]")
    for _, r in STATS_SECONDARY[STATS_SECONDARY.metric == m].iterrows():
        print("  - " + _sentence(r))
with open(os.path.join(STATS_DIR, "stats_sentences.txt"), "w") as f:
    for df_ in (STATS_PRIMARY, STATS_SECONDARY):
        for _, r in df_.iterrows():
            f.write(f"[{r['family']}] " + _sentence(r) + "\n")

# ---- consistency: main-loop values == ablation-loop values on shared images --
_cons_rows = []
for key in _bb_done:
    pa = os.path.join(PER_IMAGE_DIR, f"{key}_ablation_per_image.csv")
    if not os.path.exists(pa):
        continue
    ab = pd.read_csv(pa); mats = load_metric_matrix(key)
    for arm, mk in (("agf_full", PROPOSED), ("agf_equal", "equal")):
        sub = ab[ab.variant == arm].set_index("img_idx")
        shared = sorted(set(sub.index) & set(mats["aopc_ins"].index))
        if not shared:
            continue
        dev = float(np.max(np.abs(sub.loc[shared, "aopc_ins"].values - mats["aopc_ins"].loc[shared, mk].values)))
        _cons_rows.append({"backbone_key": key, "arm": arm, "n_shared": len(shared), "max_abs_dev_ins": dev})
        if dev > 1e-5:
            log(f"[stats][WARN] {key}/{arm}: main vs ablation insertion differ by {dev:.2e} on shared images")
CONSISTENCY_DF = pd.DataFrame(_cons_rows)
CONSISTENCY_DF.to_csv(os.path.join(STATS_DIR, "consistency_main_vs_ablation.csv"), index=False)
print("\nMain-vs-ablation determinism check:\n", CONSISTENCY_DF.to_string(index=False))

# ---- §18 per-class analysis (descriptive only, no tests) -----------------
_pc = []
for key in _bb_done:
    mats = load_metric_matrix(key); labels = mats["_labels"]
    disp = next(s["display"] for s in MODELS if s["key"] == key)
    for metric in ("aopc_ins", "aopc_del_comp"):
        M = mats[metric]
        for c, cn in enumerate(class_names):
            idx = labels.index[labels == c]
            Mc = M.loc[M.index.intersection(idx)]
            row = {"backbone": disp, "metric": metric, "class": cn, "n": len(Mc)}
            for mk in ORDER:
                row[f"mean_{mk}"] = float(Mc[mk].mean()) if mk in Mc else np.nan
            row["mean_paired_diff_agf_minus_equal"] = float((Mc[PROPOSED] - Mc["equal"]).mean())
            _pc.append(row)
PER_CLASS_DF = pd.DataFrame(_pc)
save_table(PER_CLASS_DF, "per_class_descriptive", float_fmt="%.4f")
PER_CLASS_DF.to_csv(os.path.join(STATS_DIR, "per_class_descriptive.csv"), index=False)

# ---- §17 input: metric means per backbone x method (descriptive) ---------
_cm = []
for key in _bb_done:
    mats = load_metric_matrix(key)
    disp = next(s["display"] for s in MODELS if s["key"] == key)
    for mk in ORDER:
        r = {"backbone": disp, "method_key": mk, "method": DISP[mk]}
        for metric in ("aopc_ins", "aopc_del_comp", "aopc_del_raw", "sensitivity"):
            col = mats[metric][mk] if mk in mats[metric] else pd.Series(dtype=float)
            r[f"{metric}_mean"] = float(col.mean()); r[f"{metric}_std"] = float(col.std(ddof=0))
            r[f"{metric}_n"] = int(col.notna().sum())
        _cm.append(r)
CROSS_MEANS_DF = pd.DataFrame(_cm)
save_table(CROSS_MEANS_DF, "cross_metric_means", float_fmt="%.4f")
CROSS_MEANS_DF.to_csv(os.path.join(STATS_DIR, "cross_metric_means.csv"), index=False)

# ---- §19 weight diagnostics on the main evaluation subset (descriptive) ---
_wd = []
_wfig = {}
for key in _bb_done:
    pw = os.path.join(PER_IMAGE_DIR, f"{key}_agf_weights.csv")
    if not os.path.exists(pw):
        continue
    w = pd.read_csv(pw); disp = next(s["display"] for s in MODELS if s["key"] == key)
    wcols = [f"w_{n}" for n in ("gradcam", "smoothgrad", "ig")]
    _wfig[disp] = w[wcols]
    for c in wcols:
        _wd.append({"backbone": disp, "method": c[2:], "n": len(w), "weight_mean": w[c].mean(),
                    "weight_std_across_images": w[c].std(ddof=0), "weight_min": w[c].min(),
                    "weight_max": w[c].max(),
                    "mean_within_image_spread_std": w["wspread_std"].mean(),
                    "frac_images_spread_gt_thresh": float((w["wspread_std"] > WEIGHT_SPREAD_THRESH).mean()),
                    "spread_threshold": WEIGHT_SPREAD_THRESH})
WEIGHT_DIAG_MAIN = pd.DataFrame(_wd)
save_table(WEIGHT_DIAG_MAIN, "weight_diagnostic_main", float_fmt="%.4f")
WEIGHT_DIAG_MAIN.to_csv(os.path.join(STATS_DIR, "weight_diagnostic_main.csv"), index=False)
print("\nAGF weight diagnostics (main subset):\n", WEIGHT_DIAG_MAIN.to_string(index=False))
if _wfig:
    fig, axes = plt.subplots(1, len(_wfig), figsize=(4.2 * len(_wfig), 3.6), squeeze=False)
    for ax, (disp, wd) in zip(axes[0], _wfig.items()):
        ax.boxplot([wd[c].values for c in wd.columns], showmeans=True)
        ax.set_xticks(range(1, len(wd.columns) + 1)); ax.set_xticklabels([c[2:] for c in wd.columns])
        ax.axhline(1 / 3, ls="--", color="grey", lw=0.8); ax.set_title(disp); ax.set_ylabel("AGF weight")
        ax.grid(axis="y", alpha=.3)
    fig.suptitle(f"AGF weights on the evaluation subset (kernel={AGREEMENT_KERNEL}, τ={AGREEMENT_TAU:g})")
    plt.tight_layout(); savefig_all(fig, "weight_distribution_main")
mark_section("15_statistics")

PRIMARY FAMILY — AGF vs Equal Fusion (Holm over all rows)

[Insertion (AOPC-Ins, higher better)]
  - VGG16-BN: AGF showed a statistically significant improvement in insertion AOPC over Equal Fusion; mean paired Δ = +0.0010 [+0.0003, +0.0018], median paired Δ = +0.0004 [+0.0001, +0.0009], r_rb = +0.240, n = 200, p_raw = 0.00326, p_Holm = 0.0228 (family of 9).
  - DenseNet121: the insertion AOPC difference between AGF and Equal Fusion was not statistically significant; mean paired Δ = +0.0003 [-0.0014, +0.0022], median paired Δ = -0.0008 [-0.0018, +0.0005], r_rb = +0.002, n = 200, p_raw = 0.983, p_Holm = 1 (family of 9).
  - ResNet50: the insertion AOPC difference between AGF and Equal Fusion was not statistically significant; mean paired Δ = +0.0007 [-0.0011, +0.0026], median paired Δ = -0.0008 [-0.0024, +0.0011], r_rb = +0.012, n = 200, p_raw = 0.884, p_Holm = 1 (family of 9).

[Deletion (1 - AOPC-Del, higher better)]
  - VGG16-BN: the deletion (1 − AOPC-Del) difference between AGF and

## §17 Cross-architecture analysis

Descriptive means per method and backbone for all three metrics, plus the §15 inferential results
arranged by backbone. No "best method" flag is produced. Mean ranks are
still produced but are labelled descriptive and are not used for any claim.
Architecture-dependence is described by comparing the per-backbone verdicts; no interaction test is run
(it was not part of the experimental design).

In [20]:
# ── §17  Cross-architecture aggregation, tables & figures ──────────────────
_run_keys = [k for k in RUN_MODELS if k in ALL_RESULTS]
if len(_run_keys) == 0:
    log("[cross-arch] no models evaluated — nothing to aggregate.")
else:
    _bbd = [ALL_RESULTS[k]["display"] for k in _run_keys]
    # ---- classification (benchmark source of truth) ------------------------
    df_class = pd.DataFrame([
        {"Backbone": ALL_RESULTS[k]["display"],
         **{m: ALL_RESULTS[k]["classification"].get(m) for m in ("accuracy", "f1", "mean_auc", "mcc")}}
        for k in _run_keys])
    save_table(df_class, "cross_classification_results", index=False)
    register_excel("cross_classification", df_class)

    # ---- descriptive means, method x backbone, one table per metric --------
    KEEP_TABLES.add("cross_xai_aopc_del_comp")
    def _pivot(metric):
        pv = (CROSS_MEANS_DF.pivot_table(index="method", columns="backbone", values=f"{metric}_mean")
              .reindex([DISP[m] for m in ORDER])[_bbd])
        pv["Mean_over_backbones"] = pv[_bbd].mean(axis=1)
        return pv.reset_index().rename(columns={"method": "Method"})
    df_xai_out = _pivot("aopc_ins");       save_table(df_xai_out, "cross_xai_aopc_ins", index=False)
    df_del_out = _pivot("aopc_del_comp");  save_table(df_del_out, "cross_xai_aopc_del_comp", index=False)
    df_robx_out = _pivot("sensitivity");   save_table(df_robx_out, "cross_robustness_sens", index=False)
    for nm, d in (("cross_xai_aopc_ins", df_xai_out), ("cross_xai_aopc_del_comp", df_del_out),
                  ("cross_robustness_sens", df_robx_out)):
        register_excel(nm, d)

    # ---- descriptive mean rank (NOT used for claims) ------------------------
    _rank_rows = []
    for metric in ("aopc_ins", "aopc_del_comp", "sensitivity"):
        pv = CROSS_MEANS_DF.pivot_table(index="method", columns="backbone", values=f"{metric}_mean")[_bbd]
        rk = pv.rank(ascending=(METRIC_DIRECTION[metric] < 0), axis=0).mean(axis=1)
        for mth, v in rk.items():
            _rank_rows.append({"metric": metric, "Method": mth, "MeanRank_descriptive": float(v)})
    df_rank_all = pd.DataFrame(_rank_rows)
    df_rank = (df_rank_all[df_rank_all.metric == "aopc_ins"][["Method", "MeanRank_descriptive"]]
               .sort_values("MeanRank_descriptive").rename(columns={"MeanRank_descriptive": "MeanRank"}))
    save_table(df_rank_all, "cross_overall_ranking", index=False)
    register_excel("cross_overall_ranking", df_rank_all)

    # ---- inferential results arranged by backbone (from §15) ----------------
    _cols = ["family", "backbone", "metric", "comparison", "n", "mean_agf", "mean_ref",
             "mean_paired_diff", "mean_diff_CI_lo", "mean_diff_CI_hi", "median_paired_diff",
             "median_diff_CI_lo", "median_diff_CI_hi", "rank_biserial", "p_raw", "p_holm",
             "family_size", "significant", "verdict", "sign_concordant"]
    df_gen = pd.concat([STATS_PRIMARY, STATS_SECONDARY], ignore_index=True)[_cols]
    save_table(df_gen, "cross_agf_generalization", index=False, float_fmt="%.5g")
    register_excel("cross_agf_generalization", df_gen)
    df_robstat = df_gen[df_gen.metric == "sensitivity"]
    save_table(df_robstat, "cross_robustness_stats", index=False, float_fmt="%.5g")

    print("\n" + "=" * 90); print("CROSS-ARCHITECTURE VERDICT MATRIX (per metric, per backbone; no pooling)")
    print("=" * 90)
    _vm = df_gen.pivot_table(index=["family", "comparison", "metric"], columns="backbone",
                             values="verdict", aggfunc="first")[_bbd]
    print(_vm.to_string())
    _vm.to_csv(os.path.join(DIRS["Statistics"], "cross_verdict_matrix.csv"))

    # ---- Figure: grouped means per metric ----------------------------------
    def _f_cross_bar():
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
        for ax, (metric, ttl) in zip(axes, [("aopc_ins", "Insertion AOPC ↑"),
                                            ("aopc_del_comp", "Deletion 1−AOPC ↑"),
                                            ("sensitivity", "Max-sensitivity ↓")]):
            nb = len(_bbd); xp = np.arange(len(ORDER)); w = 0.8 / max(nb, 1)
            for bi, bb in enumerate(_bbd):
                sub = CROSS_MEANS_DF[CROSS_MEANS_DF.backbone == bb].set_index("method_key")
                ax.bar(xp + bi * w, [sub.loc[m, f"{metric}_mean"] for m in ORDER], w, label=bb,
                       edgecolor="black", lw=.4)
            ax.set_xticks(xp + w * (nb - 1) / 2)
            ax.set_xticklabels([DISP[m] for m in ORDER], rotation=20, ha="right")
            ax.set_title(ttl, fontweight="bold"); ax.grid(axis="y", alpha=.3)
        axes[0].legend(fontsize=8)
        fig.suptitle("Descriptive means by method and backbone", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "cross_arch_metric_means")
        # legacy single-metric figure kept for continuity
        fig, ax = plt.subplots(figsize=(11, 5)); nb = len(_bbd); xp = np.arange(len(ORDER)); w = 0.8 / nb
        for bi, bb in enumerate(_bbd):
            sub = CROSS_MEANS_DF[CROSS_MEANS_DF.backbone == bb].set_index("method_key")
            ax.bar(xp + bi * w, [sub.loc[m, "aopc_ins_mean"] for m in ORDER], w, label=bb, edgecolor="black", lw=.4)
        ax.set_xticks(xp + w * (nb - 1) / 2); ax.set_xticklabels([DISP[m] for m in ORDER], rotation=15, ha="right")
        ax.set_ylabel("AOPC-Ins (mean, higher = better)"); ax.legend(); ax.grid(axis="y", alpha=.3)
        ax.set_title("Cross-Architecture AOPC-Ins by Method (descriptive)", fontweight="bold")
        plt.tight_layout(); savefig_all(fig, "cross_arch_aopc_ins")
    _try(_f_cross_bar, "cross_arch_metric_means")

    # ---- Figure: primary family forest plot (AGF − Equal Fusion) ------------
    def _f_forest():
        d = STATS_PRIMARY.dropna(subset=["mean_paired_diff"]).reset_index(drop=True)
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
        for ax, metric in zip(axes, PRIMARY_METRICS):
            s = d[d.metric == metric].reset_index(drop=True)
            yp = np.arange(len(s))[::-1]
            for i, r in s.iterrows():
                col = ("#2A9D8F" if r["verdict"] == "significant: AGF better" else
                       "#E63946" if r["verdict"] == "significant: AGF worse" else "#8D99AE")
                ax.plot([r["mean_diff_CI_lo"], r["mean_diff_CI_hi"]], [yp[i], yp[i]], color=col, lw=2.4)
                ax.scatter([r["mean_paired_diff"]], [yp[i]], s=60, color=col, edgecolor="black", zorder=3)
                ax.text(r["mean_diff_CI_hi"], yp[i] + 0.18, f"p$_{{Holm}}$={r['p_holm']:.3g}",
                        fontsize=7, ha="right")
            ax.axvline(0, color="black", ls="--", lw=1)
            ax.set_ylim(-0.5, len(s) - 0.2)
            ax.set_yticks(yp); ax.set_yticklabels(s["backbone"].tolist())
            ax.set_title(METRIC_LABEL[metric], fontsize=9, fontweight="bold")
            ax.set_xlabel("mean paired Δ (AGF − Equal Fusion), 95% bootstrap CI"); ax.grid(axis="x", alpha=.3)
        fig.suptitle(f"Primary family: AGF vs Equal Fusion (Wilcoxon, Holm over {int(d['family_size'].iloc[0])} tests; "
                     "green = AGF significantly better, red = worse, grey = n.s.)", fontsize=9)
        plt.tight_layout(); savefig_all(fig, "primary_agf_vs_equal_forest")
    _try(_f_forest, "primary_agf_vs_equal_forest")

    def _f_rank():
        fig, ax = plt.subplots(figsize=(9, 4.5))
        colors = ["#F4A261" if m == DISP[PROPOSED] else "#9AA0A6" for m in df_rank["Method"]]
        ax.barh(df_rank["Method"], df_rank["MeanRank"], color=colors, edgecolor="black", lw=.5)
        ax.invert_yaxis(); ax.set_xlabel("Mean rank of mean AOPC-Ins across backbones (1 = best) — descriptive")
        ax.set_title("Descriptive Method Ranking (AOPC-Ins means; not a significance statement)", fontweight="bold")
        ax.grid(axis="x", alpha=.3)
        plt.tight_layout(); savefig_all(fig, "cross_overall_ranking")
    _try(_f_rank, "cross_overall_ranking")
    mark_section("17_cross_architecture")


CROSS-ARCHITECTURE VERDICT MATRIX (per metric, per backbone; no pooling)
backbone                                                                     VGG16-BN                    DenseNet121                       ResNet50
family    comparison                     metric                                                                                                    
primary   AGF (Proposed) vs Equal Fusion aopc_del_comp  not statistically significant  not statistically significant  not statistically significant
                                         aopc_ins             significant: AGF better  not statistically significant  not statistically significant
                                         sensitivity          significant: AGF better        significant: AGF better  not statistically significant
secondary AGF (Proposed) vs GradCAM++    aopc_del_comp  not statistically significant  not statistically significant         significant: AGF worse
                                      

## §16 Ablations

* **§16a Robustness ablation** — AGF-Full vs Equal vs Fixed under input perturbation. Fixed weights come
  from the validation split, SmoothGrad noise is frozen across each (base, perturbed) pair, and the
  perturbation seed is derived from the image index.
* **§16b Corruption probe** (single backbone) — does the weighting activate when the base methods are
  pushed apart? Target class pinned on the clean image; Holm across all conditions; reproducible noise
  seeds.
* **§16c Weighting, kernel and temperature ablation** — reports weight spread, the three agreement kernels
  and τ ∈ {1, 0.25, 0.1, 0.05} against uniform weights, on every backbone. These are *reported*; τ and the
  kernel are fixed a priori and are never selected from these numbers.

In [21]:
# ── §16a  Robustness ablation: AGF-Full vs AGF-Equal vs AGF-Fixed ─────────
# [FINAL] REQUIRES RERUN. Changes vs v7.1:
#   (1) AGF-Fixed weights are the §11c CALIBRATION vector (validation split),
#       no longer the mean over the same 24 test images (pass 1 removed);
#   (2) SmoothGrad noise is FROZEN across each (base, perturbed) pair, exactly as
#       in the main robustness metric (v7.1 drew fresh noise per perturbation here);
#   (3) maps are post-processed identically to the main loop (224x224, clip>=0);
#   (4) knobs come from §1; verdict text is neutral and per comparison.
# AGF-Full vs Equal robustness at n=GINI_SENS_N is already in the PRIMARY family
# (§15); this cell's unique contribution is the AGF-Fixed arm.
from scipy.stats import wilcoxon as _wilcoxon
from statsmodels.stats.multitest import multipletests as _multipletests

# ---- knobs: from §1 ----------------------------------------------------------
ROBUST_ABL_MODELS  = list(RUN_MODELS)
ROBUST_ABL_RADIUS  = SENS_RADIUS
_ABL_VARIANTS = ["agf_full", "agf_equal", "agf_fixed"]
_ABL_DISP = {"agf_full": "AGF-Full (agreement-weighted)",
             "agf_equal": "AGF-Equal (uniform)",
             "agf_fixed": "AGF-Fixed (validation-calibrated)"}


def robustness_ablation_one(eval_model, target_layer, idxs, radius, n_samples, fixed_w):
    """Per-image relative max-sensitivity for the three AGF variants.
    All variants share the SAME base maps, perturbations and frozen SmoothGrad
    noise, so the three columns are exactly paired."""
    sens = {v: [] for v in _ABL_VARIANTS}
    kept = []
    for pos, img_idx in enumerate(tqdm(idxs, desc="robustness ablation", leave=False)):
        t = load_image_tensor(sample_paths[img_idx])
        gc = sg = ig = None
        try:
            gc, sg, ig = make_base_generators(eval_model, target_layer)
            torch.manual_seed(stable_seed("robust_abl", img_idx))     # perturbation draw
            with torch.inference_mode(False):
                pred = gc.generate(t)[1]
                sg.freeze_noise(SENS_FREEZE_SG_NOISE)               # base call defines the draw
                base, _ = agf_ablation_maps(gc, sg, ig, t, pred, fixed_w)
                norms = {v: float(np.linalg.norm(base[v])) + 1e-8 for v in _ABL_VARIANTS}
                maxrel = {v: 0.0 for v in _ABL_VARIANTS}
                for _ in range(n_samples):
                    noise = torch.randn_like(t) * radius
                    pert, _ = agf_ablation_maps(gc, sg, ig, t + noise, pred, fixed_w)
                    for v in _ABL_VARIANTS:
                        d = float(np.linalg.norm(pert[v] - base[v])) / norms[v]
                        if not np.isnan(d) and d > maxrel[v]:
                            maxrel[v] = d
                sg.freeze_noise(False)
            for v in _ABL_VARIANTS:
                sens[v].append(maxrel[v])
            kept.append(int(img_idx))
        except Exception as e:
            log(f"  [SKIP rob] idx={img_idx}: {type(e).__name__}: {e}")
        finally:
            if gc is not None: gc.remove()
            del gc, sg, ig, t
            if pos % 5 == 0:
                free_gpu()
    return sens, kept


# ---- run over the requested backbones --------------------------------------
ROBUST_ABL, _rob_stat_rows, _rob_mean_rows = {}, [], []
for spec in MODELS:
    if spec["key"] not in ROBUST_ABL_MODELS:
        continue
    log("=" * 70); log(f"▶ ROBUSTNESS ABLATION — {spec['display']}"); log("=" * 70)
    seed_everything(SEED)
    model, eval_model, _inf, backbone_name, _meta = load_model(spec)
    target_layer = get_target_layer(model, backbone_name)
    idxs = stratified_sample(ROBUST_ABL_N, ROBUST_ABL_SEED)
    fixed_w = load_fixed_weights(spec["key"])
    sens, kept = robustness_ablation_one(
        eval_model, target_layer, idxs, ROBUST_ABL_RADIUS, ROBUST_ABL_SAMPLES, fixed_w)
    ROBUST_ABL[spec["key"]] = {"display": spec["display"], "sens": sens, "fixed_w": fixed_w, "idx": kept}
    pd.DataFrame({"img_idx": kept, "image_id": [sample_ids[i] for i in kept],
                  **{f"sens_{v}": sens[v] for v in _ABL_VARIANTS},
                  "config_version": CONFIG_VERSION, "subset_seed": ROBUST_ABL_SEED,
                  "samples": ROBUST_ABL_SAMPLES, "radius": ROBUST_ABL_RADIUS}).to_csv(
        os.path.join(PER_IMAGE_DIR, f"{spec['key']}_robustness_ablation_per_image.csv"), index=False)

    for v in _ABL_VARIANTS:
        arr = np.asarray(sens[v], dtype=float)
        _rob_mean_rows.append({"Backbone": spec["display"], "Variant": _ABL_DISP[v],
                               "Sensitivity_mean": float(arr.mean()) if arr.size else float("nan"),
                               "Sensitivity_std": float(arr.std()) if arr.size else float("nan"),
                               "n": int(arr.size)})
    # Paired tests: does AGF-Full differ in robustness (sensitivity, LOWER is
    # better) from each control? Two-sided -- a one-sided "less" reports the
    # case where the control wins as p~1.0 / significant=False, which reads as
    # "no difference" rather than "agreement weighting hurt".
    for bi, (base_v, label) in enumerate([("agf_equal", "AGF-Full vs AGF-Equal"),
                                          ("agf_fixed", "AGF-Full vs AGF-Fixed")]):
        a = np.asarray(sens["agf_full"], dtype=float)
        b = np.asarray(sens[base_v], dtype=float)
        # all variants are appended in lockstep inside robustness_ablation_one,
        # so lengths must match; truncating silently would misalign the pairs
        if a.size != b.size:
            log(f"[WARN] robustness pair length mismatch for {label}: "
                f"{a.size} vs {b.size} -- truncating to the shorter run")
        n = min(a.size, b.size); a, b = a[:n], b[:n]; d = a - b
        try:
            W, p = _wilcoxon(a, b, alternative=STATS_ALTERNATIVE)
        except ValueError:
            W, p = 0.0, 1.0
        rb = paired_rank_biserial(a, b)
        ci_lo, ci_hi = bootstrap_ci(d, seed=SEED + 1000 * bi, stat=np.median)
        _rob_stat_rows.append({
            "Backbone": spec["display"], "comparison": label, "n": int(n),
            "median_full": float(np.median(a)) if n else float("nan"),
            "median_control": float(np.median(b)) if n else float("nan"),
            "median_diff": float(np.median(d)) if n else float("nan"),
            "diff_CI_lo": ci_lo, "diff_CI_hi": ci_hi,
            "rank_biserial": rb, "cliffs_delta": cliffs_delta(a, b),
            "p_raw": float(p), "W": float(W),
            "mean_diff": float(np.mean(d)) if n else float("nan"),
            "more_robust": bool(rb < 0),   # direction of the signed-rank statistic (lower = better)
            "effect_magnitude": rb_magnitude(rb),
            "cliffs_magnitude": cliff_magnitude(cliffs_delta(a, b)),
        })
    del model, eval_model, target_layer
    plt.close("all"); free_gpu()

# ---- Holm correction across the whole family of tests ---------------------
df_rob = pd.DataFrame(_rob_stat_rows)
if not df_rob.empty:
    rej, p_adj, _, _ = _multipletests(df_rob["p_raw"].values, alpha=0.05, method="holm")
    df_rob["p_holm"] = p_adj
    # `significant` now means the two variants DIFFER; direction lives in
    # `favours`. `significant_more_robust` is the old (directional) flag, kept
    # so the AGF-Full-wins claim can still be read off a single column.
    df_rob["significant"] = rej
    df_rob["favours"] = np.where(~rej, "-",
                        np.where(df_rob["more_robust"], "AGF-Full", "control"))
    df_rob["significant_more_robust"] = rej & df_rob["more_robust"]
df_rob_means = pd.DataFrame(_rob_mean_rows)

# ---- persist ---------------------------------------------------------------
if not df_rob.empty:
    save_table(df_rob, "robustness_ablation_stats"); register_excel("robustness_ablation", df_rob)
    save_table(df_rob_means, "robustness_ablation_means"); register_excel("robustness_abl_means", df_rob_means)
    with open(os.path.join(DIRS["Statistics"], "robustness_ablation.json"), "w") as f:
        json.dump({"stats": _rob_stat_rows, "means": _rob_mean_rows,
                   "config": {"n": ROBUST_ABL_N, "samples": ROBUST_ABL_SAMPLES,
                              "radius": ROBUST_ABL_RADIUS, "seed": ROBUST_ABL_SEED,
                              "fixed_weights_source": "benchmark_val_split",
                              "sg_noise_frozen": SENS_FREEZE_SG_NOISE,
                              "config_version": CONFIG_VERSION}}, f, indent=2, default=float)

# ---- figure: per-variant sensitivity distribution per backbone ------------
def _fig_robustness_ablation():
    keys = [k for k in ROBUST_ABL_MODELS if k in ROBUST_ABL]
    if not keys: return
    fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 4.5), squeeze=False)
    cols = ["#F4A261", "#B0B0B0", "#457B9D"]
    for ax, k in zip(axes[0], keys):
        sens = ROBUST_ABL[k]["sens"]
        data = [sens[v] for v in _ABL_VARIANTS]
        bp = ax.boxplot(data, labels=["Full", "Equal", "Fixed"], patch_artist=True, showmeans=True)
        for patch, c in zip(bp["boxes"], cols): patch.set_facecolor(c)
        ax.set_title(ROBUST_ABL[k]["display"], fontweight="bold")
        ax.set_ylabel("Max-sensitivity (↓ more robust)"); ax.grid(axis="y", alpha=.3)
    fig.suptitle("Robustness ablation — AGF-Full vs Equal vs Fixed", fontweight="bold")
    plt.tight_layout(); savefig_all(fig, "robustness_ablation")
try:
    _fig_robustness_ablation()
except Exception as e:
    log(f"[FIG FAIL] robustness_ablation: {type(e).__name__}: {e}")

# ---- verdict ---------------------------------------------------------------
print("\n" + "=" * 78); print("ROBUSTNESS ABLATION — mean max-sensitivity (lower = more robust)")
print("=" * 78)
if not df_rob_means.empty:
    print(df_rob_means.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\n" + "=" * 78)
print("PAIRED TEST (two-sided: does AGF-Full differ from the control?)")
print("=" * 78)
if not df_rob.empty:
    cols = ["Backbone", "comparison", "median_diff", "diff_CI_lo", "diff_CI_hi",
            "rank_biserial", "effect_magnitude", "p_holm", "significant", "favours"]
    cols = [c for c in cols if c in df_rob.columns]
    print(df_rob[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("\nPer comparison (no pooled verdict):")
    for _, r in df_rob.iterrows():
        if r["significant"]:
            msg = ("significant: AGF-Full MORE robust than the control" if r["more_robust"]
                   else "significant: AGF-Full LESS robust than the control")
        else:
            msg = "not statistically significant"
        print(f"  {r['Backbone']:<12} {r['comparison']:<22} mean Δ={r['mean_diff']:+.4f} "
              f"median Δ={r['median_diff']:+.4f} [{r['diff_CI_lo']:+.4f}, {r['diff_CI_hi']:+.4f}] "
              f"r_rb={r['rank_biserial']:+.3f} n={int(r['n'])} p_holm={r['p_holm']:.4g} -> {msg}")
log("Robustness ablation complete. Tables in Tables/, figure in Figures/, JSON in Statistics/.")
mark_section("16a_robustness_ablation")

[17:07:04] ======================================================================
[17:07:04] ▶ ROBUSTNESS ABLATION — VGG16-BN
[17:07:04] ======================================================================


robustness ablation:   0%|          | 0/24 [00:00<?, ?it/s]

[17:11:38] ======================================================================
[17:11:38] ▶ ROBUSTNESS ABLATION — DenseNet121
[17:11:38] ======================================================================


robustness ablation:   0%|          | 0/24 [00:00<?, ?it/s]

[17:16:51] ======================================================================
[17:16:51] ▶ ROBUSTNESS ABLATION — ResNet50
[17:16:51] ======================================================================


robustness ablation:   0%|          | 0/24 [00:00<?, ?it/s]


ROBUSTNESS ABLATION — mean max-sensitivity (lower = more robust)
   Backbone                           Variant  Sensitivity_mean  Sensitivity_std  n
   VGG16-BN     AGF-Full (agreement-weighted)            0.1165           0.0327 24
   VGG16-BN               AGF-Equal (uniform)            0.1185           0.0321 24
   VGG16-BN AGF-Fixed (validation-calibrated)            0.1172           0.0318 24
DenseNet121     AGF-Full (agreement-weighted)            0.1506           0.0449 24
DenseNet121               AGF-Equal (uniform)            0.1557           0.0420 24
DenseNet121 AGF-Fixed (validation-calibrated)            0.1523           0.0415 24
   ResNet50     AGF-Full (agreement-weighted)            0.3990           0.0761 24
   ResNet50               AGF-Equal (uniform)            0.3991           0.0768 24
   ResNet50 AGF-Fixed (validation-calibrated)            0.3998           0.0775 24

PAIRED TEST (two-sided: does AGF-Full differ from the control?)
   Backbone            compar

In [22]:
# ── §16b  Corruption probe (single backbone: PROBE_BACKBONE) ───────────────
# [FINAL] REQUIRES RERUN of the gaussian_noise rows: v7.1 seeded the corruption
# noise with Python's hash((kind, level)), which is salted per interpreter
# process (PYTHONHASHSEED set at runtime has no effect), so those rows were not
# reproducible across sessions. Now stable_seed(...). Other corruptions consume no
# RNG and are unaffected except through the §4 preprocessing fix.
#   REBUILT in v6. The previous version was invalid as run, for three
#   independent reasons — see the notes on _corrupt(), the sweep, and the
#   statistics below. Do not compare its numbers to these.
import numpy as np, torch, os
import pandas as pd
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

# ---- probe config: PROBE_BACKBONE, N_CORRUPT, CORRUPTIONS, SEVERITY from §1 --
PROBE_METHODS = [
    "agf_full", "agf_equal", "gradcam", "smoothgrad", "ig"]

# ---- corruption operators (operate on the normalised 3xHxW tensor) --------
def _corrupt(tensor, kind, level):
    t = tensor.clone()
    if kind == "clean":
        return t
    if kind == "gaussian_noise":
        return t + torch.randn_like(t) * float(level)
    if kind == "gaussian_blur":
        k = max(3, int(2 * round(3 * level) + 1))      # odd kernel from sigma
        return TF.gaussian_blur(t.unsqueeze(0), kernel_size=[k, k],
                                sigma=[float(level), float(level)]).squeeze(0)
    if kind == "brightness":
        return t + float(level)
    if kind == "contrast":
        mean = t.mean(dim=(1, 2), keepdim=True)
        return (t - mean) * float(level) + mean
    raise KeyError(kind)

# ---- load the probe backbone ----------------------------------------------
_spec = next(s for s in MODELS if s["key"] == PROBE_BACKBONE)
log("=" * 70); log(f"CORRUPTION PROBE — {_spec['display']}"); log("=" * 70)
_model, _eval_model, _infeat, _bbname, _meta = load_model(_spec)
_tlayer = get_target_layer(_eval_model, _bbname)
assert_spatial_target(_eval_model, _tlayer, IMG_SIZE, DEVICE, label=_spec["display"])
_idxs = stratified_sample(N_CORRUPT, seed=CORRUPT_SEED)
log(f"[probe] {len(_idxs)} stratified images | corruptions={CORRUPTIONS}")

# ---- FIX 2 — PIN THE TARGET CLASS ON THE CLEAN IMAGE ----------------------
#   The old sweep called fuse(..., class_idx=None), which RE-RESOLVES the
#   predicted class on the CORRUPTED image. So AOPC at severity 4 explained a
#   different class than at severity 0, and the two were then plotted on the
#   same curve and Wilcoxon-tested against each other. That is what produced
#   the tell-tale non-monotonic reversal: gaussian_noise AOPC-Ins ran
#   0.752 -> 0.726 -> 0.694 -> 0.596 -> 0.822, i.e. HIGHER at maximum
#   corruption than clean. The model collapses to a confident wrong class on
#   heavy noise and the metric rewards explaining that class well.
#   The clean-image prediction is now resolved once per image and held fixed
#   across the entire severity sweep, so the curves are comparable.
_clean_pred = {}
for img_idx in tqdm(_idxs, desc="clean predictions", leave=False):
    _t = load_image_tensor(sample_paths[img_idx])
    with torch.inference_mode():
        _clean_pred[int(img_idx)] = int(
            _eval_model(_t.unsqueeze(0).to(DEVICE)).argmax(1).item())
    del _t
free_gpu()
log(f"[probe] target classes pinned on the clean image for {len(_clean_pred)} images")

# ---- main sweep -----------------------------------------------------------
_CONDITIONS = [("clean", 0, 0.0)] + [
    (kind, i + 1, lv) for kind in CORRUPTIONS
    for i, lv in enumerate(SEVERITY[kind])]

rows = []
for kind, lv_i, lv in _CONDITIONS:
    torch.manual_seed(stable_seed("corruption", kind, lv_i, SEED))
    for img_idx in tqdm(_idxs, desc=f"{kind} sev{lv_i}", leave=False):
        gc_gen = sg_gen = ig_gen = None
        try:
            clean = load_image_tensor(sample_paths[img_idx])
            x = _corrupt(clean, kind, lv)
            pred = _clean_pred[int(img_idx)]          # PINNED, never re-resolved
            gc_gen, sg_gen, ig_gen = make_base_generators(_eval_model, _tlayer)
            fd = fusion_dict(gc_gen, sg_gen, ig_gen)
            with torch.inference_mode(False):
                gc_m, _ = gc_gen.generate(x, class_idx=pred)
                sg_m, _ = sg_gen.generate(x, class_idx=pred)
                ig_m, _ = ig_gen.generate(x, class_idx=pred)
                _pre = {"gradcam":    resize_map(gc_m, (IMG_SIZE, IMG_SIZE)),
                        "smoothgrad": resize_map(sg_m, (IMG_SIZE, IMG_SIZE)),
                        "ig":         resize_map(ig_m, (IMG_SIZE, IMG_SIZE))}
                fused_full, indiv, _w, _a = \
                    AgreementGuidedFusion(fd).fuse_maps(_pre)
            eq = {k: 1.0 / len(indiv) for k in indiv}
            maps = {"agf_full":   fused_full,
                    "agf_equal":  weighted_arith(indiv, eq),
                    "gradcam":    indiv["gradcam"],
                    "smoothgrad": indiv["smoothgrad"],
                    "ig":         indiv["ig"]}
            for m in PROBE_METHODS:
                ins = aopc_curve(_eval_model, x, maps[m], pred, mode="insert")
                rows.append({"corruption": kind, "sev_i": lv_i, "sev": float(lv),
                             "img": int(img_idx), "method": m,
                             "aopc_ins": float(np.mean(ins))})   # curve -> scalar
            del maps, indiv, _pre, gc_m, sg_m, ig_m, clean, x
        except Exception as e:
            log(f"  [probe] skip img {img_idx} ({kind} sev{lv_i}): "
                f"{type(e).__name__}: {e}")
        finally:
            if gc_gen is not None: gc_gen.remove()
            del gc_gen, sg_gen, ig_gen
            free_gpu()

df = pd.DataFrame(rows)
df["image_id"] = [sample_ids[i] for i in df["img"]]
df["config_version"] = CONFIG_VERSION; df["subset_seed"] = CORRUPT_SEED
df.to_csv(os.path.join(DIRS["Statistics"], f"corruption_probe_{PROBE_BACKBONE}.csv"),
          index=False)
log(f"[probe] {len(df)} raw rows saved to Statistics/")

# ---- aggregate + the key test ---------------------------------------------
def _paired(a_vals, b_vals):
    a = np.asarray(a_vals, float); b = np.asarray(b_vals, float)
    mask = ~(np.isnan(a) | np.isnan(b)); a, b = a[mask], b[mask]
    if len(a) < 5 or np.allclose(a, b):
        return dict(n=len(a),
                    median_diff=float(np.median(a - b)) if len(a) else float("nan"),
                    p=1.0, favours="-")
    try:
        _w, p = wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
    except Exception:
        p = 1.0
    md = float(np.median(a - b))
    return dict(n=int(len(a)), median_diff=md, p=float(p),
                favours=("AGF-Full" if md > 0 else "AGF-Equal" if md < 0 else "-"))

summary, weighting_rows = [], []
for kind, lv_i, lv in _CONDITIONS:
    sub = df[(df.corruption == kind) & (df.sev_i == lv_i)]
    if sub.empty:
        continue
    piv = sub.pivot_table(index="img", columns="method", values="aopc_ins")
    for m in PROBE_METHODS:
        summary.append({"corruption": kind, "sev_i": int(lv_i), "sev": float(lv),
                        "method": m,
                        "aopc_ins_mean": float(piv[m].mean()) if m in piv else np.nan,
                        "aopc_ins_std":  float(piv[m].std())  if m in piv else np.nan})
    if {"agf_full", "agf_equal"}.issubset(piv.columns):
        t = _paired(piv["agf_full"], piv["agf_equal"])
        t.update({"corruption": kind, "sev_i": int(lv_i), "sev": float(lv)})
        weighting_rows.append(t)

# ---- FIX 3 — MULTIPLE-TESTING CORRECTION ----------------------------------
#   The old probe ran 20 Wilcoxon tests and reported raw p-values with an
#   "uncorrected, exploratory" footnote. Its single p = 0.034 would not
#   survive Holm across the family (adjusted ~0.68). Correcting inside the
#   cell means the printed table cannot be read as 20 independent findings.
if weighting_rows:
    _raw = [r["p"] for r in weighting_rows]
    _rej, _adj, _, _ = multipletests(_raw, alpha=0.05, method="holm")
    for r, pa, rj in zip(weighting_rows, _adj, _rej):
        r["p_holm"] = float(pa)
        r["significant"] = bool(rj)
        if not rj:
            r["favours"] = "-"          # no direction claim on a null result

sum_df   = pd.DataFrame(summary)
adapt_df = pd.DataFrame(weighting_rows, columns=["corruption", "sev_i", "sev",
                        "n", "median_diff", "p", "p_holm", "significant", "favours"])
save_table(sum_df,   f"corruption_summary_{PROBE_BACKBONE}",   subdir=None, float_fmt="%.5f")
save_table(adapt_df, f"corruption_weighting_{PROBE_BACKBONE}", subdir=None, float_fmt="%.5g")

print("\n" + "=" * 86)
print(f"AGREEMENT WEIGHTING UNDER CORRUPTION — {_spec['display']}  (AGF-Full vs AGF-Equal)")
print(f"target class pinned on the clean image | Holm-corrected over "
      f"{len(weighting_rows)} comparisons")
print("=" * 86)
print(f"{'corruption':<16}{'sev':>5}{'median_diff':>14}{'p_raw':>10}{'p_Holm':>10}  favours")
for r in weighting_rows:
    star = "  *" if (r.get("significant") and r["favours"] == "AGF-Full") else ""
    print(f"{r['corruption']:<16}{r['sev_i']:>5}{r['median_diff']:>14.5f}"
          f"{r['p']:>10.4g}{r.get('p_holm', float('nan')):>10.4g}  {r['favours']}{star}")
print("\n  * = AGF-Full significantly differs from AGF-Equal in favour of AGF-Full after Holm")
print("      (Holm family = all conditions of this probe, single backbone; exploratory).")

# ---- figure ---------------------------------------------------------------
_COL = {"agf_full": "#c0392b", "agf_equal": "#e67e22", "gradcam": "#7f8c8d",
        "smoothgrad": "#2980b9", "ig": "#27ae60"}
_LAB = {"agf_full": "AGF-Full (agreement-weighted)", "agf_equal": "AGF-Equal (uniform)",
        "gradcam": "GradCAM++", "smoothgrad": "SmoothGrad", "ig": "IG"}
fig, axes = plt.subplots(1, len(CORRUPTIONS),
                         figsize=(5 * len(CORRUPTIONS), 4.2), sharey=True)
if len(CORRUPTIONS) == 1: axes = [axes]
_clean = sum_df[sum_df.corruption == "clean"]
for ax, kind in zip(axes, CORRUPTIONS):
    s = sum_df[sum_df.corruption == kind].sort_values("sev_i")
    for m in PROBE_METHODS:
        sm = s[s.method == m]
        # prepend the SHARED clean anchor at severity 0 rather than
        # re-measuring it per corruption type
        c0 = _clean[_clean.method == m]
        xs = np.concatenate([[0], sm.sev_i.to_numpy()])
        ys = np.concatenate([c0.aopc_ins_mean.to_numpy()[:1]
                             if len(c0) else [np.nan],
                             sm.aopc_ins_mean.to_numpy()])
        lw = 2.6 if m.startswith("agf") else 1.4
        ls = "-" if m == "agf_full" else ("--" if m == "agf_equal" else ":")
        ax.plot(xs, ys, ls, color=_COL[m], lw=lw, marker="o", ms=4, label=_LAB[m])
    ax.set_title(kind.replace("_", " ")); ax.set_xlabel("severity (0 = clean)")
    ax.grid(alpha=.3)
axes[0].set_ylabel("AOPC-Insertion (higher = more faithful)")
axes[-1].legend(fontsize=8, loc="best")
fig.suptitle(f"Faithfulness under input corruption — {_spec['display']} "
             f"(target class pinned on the clean image)", y=1.02)
fig.tight_layout()
savefig_all(fig, f"corruption_probe_{PROBE_BACKBONE}")
log(f"[probe] done — figure + summary/agreement weighting tables written for {_spec['display']}.")
del _model, _eval_model, _tlayer
free_gpu()
mark_section("16b_corruption_probe")

[17:19:03] ======================================================================
[17:19:03] CORRUPTION PROBE — DenseNet121
[17:19:03] ======================================================================
[17:19:04] [target-layer] [DenseNet121] Sequential -> (1, 1024, 7, 7) | non-negative 100.0% | OK
[17:19:04] [probe] 40 stratified images | corruptions=['gaussian_noise', 'gaussian_blur', 'brightness', 'contrast']


clean predictions:   0%|          | 0/40 [00:00<?, ?it/s]

[17:19:05] [probe] target classes pinned on the clean image for 40 images


clean sev0:   0%|          | 0/40 [00:00<?, ?it/s]

gaussian_noise sev1:   0%|          | 0/40 [00:00<?, ?it/s]

gaussian_noise sev2:   0%|          | 0/40 [00:00<?, ?it/s]

... (11 outputs left out)

contrast sev2:   0%|          | 0/40 [00:00<?, ?it/s]

contrast sev3:   0%|          | 0/40 [00:00<?, ?it/s]

contrast sev4:   0%|          | 0/40 [00:00<?, ?it/s]

[18:01:47] [probe] 3400 raw rows saved to Statistics/

AGREEMENT WEIGHTING UNDER CORRUPTION — DenseNet121  (AGF-Full vs AGF-Equal)
target class pinned on the clean image | Holm-corrected over 17 comparisons
corruption        sev   median_diff     p_raw    p_Holm  favours
clean               0      -0.00051    0.7954         1  -
gaussian_noise      1      -0.00011    0.6463         1  -
gaussian_noise      2       0.00026    0.8264         1  -
gaussian_noise      3       0.00038    0.9206         1  -
gaussian_noise      4      -0.00011    0.5992         1  -
gaussian_blur       1       0.00093    0.8472         1  -
gaussian_blur       2       0.00143    0.5099         1  -
gaussian_blur       3       0.00089    0.7246         1  -
gaussian_blur       4       0.00052    0.6368         1  -
brightness          1      -0.00213    0.1281         1  -
brightness          2       0.00262    0.3403         1  -
brightness          3       0.00018    0.8576         1  -
brightness          

In [23]:
# ── §16c  Weighting diagnostic + kernel / temperature ablation ────────────
# [FINAL] Procedure unchanged; knobs now come from §1 (N_ABL, TEMP_GRID,
# WEIGHT_SPREAD_THRESH, WABL_SEED). Values change only through the §4
# preprocessing fix -> REQUIRES RERUN. These ablations REPORT alternative kernels
# and temperatures; they are never used to select tau or the kernel.
#   PATCHED (v7.1) — MULTI-BACKBONE. The per-backbone computation is
#   byte-for-byte the v7 procedure: same VARIANTS, same TEMPS derived from
#   AGREEMENT_TAU, same single Holm family, same stratified_sample(N_ABL,
#   seed=SEED+3) image set, same AgreementGuidedFusion / weighted_arith /
#   aopc_curve calls in the same order. Re-running with
#   ABL_RUN_BACKBONES=["densenet121"] reproduces the reported DenseNet121
#   tables exactly. What changed:
#
#   (A) LOOP, NOT A SINGLE BACKBONE. The body is now a function keyed on the
#       backbone, so the kernel / temperature / weight-spread analyses can be
#       replicated on VGG16-BN and ResNet50. This is the point of the patch:
#       "the agreement weighting is inert" currently rests on ONE backbone,
#       which is not a claim that survives review. The sweep is fully
#       deterministic (SG_SEED_MODE='content', fixed AOPC_TIE_BREAK_SEED, no
#       global RNG consumed), so backbone order does not affect any result.
#
#   (B) WHITELIST BUG FIXED — THIS WAS SILENTLY DESTROYING OUTPUT. Section 3's
#       KEEP_TABLES / KEEP_FIGURES hard-code the literal string
#       "..._densenet121". save_table()/savefig_all() therefore DISCARD every
#       Section 18 artifact for any other backbone without a word, so a VGG or
#       ResNet run would have appeared to succeed and written nothing. The same
#       bug already cost the DenseNet run two artifacts: the whitelist has no
#       entry for `agf_kernel_scores_densenet121` or for the
#       `kernel_ablation_densenet121` FIGURE, so both were produced and thrown
#       away. This cell registers its own output names for every backbone in
#       MODELS before writing anything, so it cannot be silently truncated
#       again regardless of Section 3's contents or cell execution order.
#
#   (C) CROSS-BACKBONE CONSOLIDATION. After the loop, the per-backbone tables
#       are re-read FROM DISK (not from memory) and merged into three cross
#       tables plus one figure. Reading from disk is deliberate: on Kaggle you
#       will run these backbones in separate sessions to stay inside quota, and
#       the consolidation must still see the earlier session's output that was
#       committed to the working directory.
#
#   COST: base attribution is computed ONCE per image; every variant is a
#   cheap re-weight plus one AOPC-Insertion sweep. The DenseNet121 run took
#   3m14s at N_ABL=40, so both remaining backbones cost roughly 7 minutes.
#   Fast wiring check: set N_ABL=8 first.
# ---------------------------------------------------------------------------
import numpy as np, torch, os, glob
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

# ── CONFIG ────────────────────────────────────────────────────────────────
#   DEFAULT = all three backbones, so "Save Version -> Save & Run All" works
#   end-to-end with NO manual cell edits. Do not trim this list for a committed
#   run: a commit starts from an EMPTY /kaggle/working, so any backbone omitted
#   here never gets its agf_*_<key>.csv written, and Section 20 -- which reads
#   agf_temperature_ablation_densenet121.csv and agf_kernel_ablation_densenet121.csv
#   by literal filename inside a try/except -- would silently DROP tables S4 and
#   S5 while still reporting success.
#
#   Trimming the list is only for INTERACTIVE quota staging, and then only when
#   the other backbones' tables are already present in /kaggle/working from
#   earlier in the SAME session (the consolidation step below re-reads them from
#   disk). Cost is ~3 min per backbone at N_ABL=40, so running all three is
#   cheaper than reasoning about which ones you can skip.
ABL_RUN_BACKBONES = [_s["key"] for _s in MODELS]   # ["vgg16","densenet121","resnet50"]
_METHOD_ORDER     = ["gradcam", "smoothgrad", "ig"]

# ---- (B) register EVERY output name this cell can emit, for EVERY backbone
#      Must run before the first save_table()/savefig_all() call.
_ABL_KEY2DISP = {s["key"]: s["display"] for s in MODELS}
for _bb in _ABL_KEY2DISP:
    KEEP_TABLES.update({f"agf_weight_diagnostic_{_bb}",
                        f"agf_temperature_ablation_{_bb}",
                        f"agf_kernel_ablation_{_bb}",
                        f"agf_kernel_scores_{_bb}"})
    KEEP_FIGURES.update({f"weighting_ablation_{_bb}", f"kernel_ablation_{_bb}"})
KEEP_TABLES.update({"agf_weight_diagnostic_cross",
                    "agf_temperature_ablation_cross",
                    "agf_kernel_ablation_cross"})
KEEP_FIGURES.update({"kernel_ablation_cross"})

# ---- (4) the ONE weight-spread threshold ----------------------------------
#   Applied to both spread quantities so the diagnostic text, the ablation
#   tables and the figure cannot disagree with each other. 0.02 is the v6
#   ssim_cosine ceiling (weight spread ~0.007) plus margin, and sits above the
#   <0.01 the all-agree synthetic control produces under topk_dice -- i.e.
#   below this line the weights are not distinguishable from uniform.

def _spread_verdict(x, thr=WEIGHT_SPREAD_THRESH):
    if not np.isfinite(x):
        return "undetermined"
    return "INERT (~uniform)" if x <= thr else "ACTIVE (weights move)"

# ---- (2) temperature grid derived from AGREEMENT_TAU ----------------------
_TEMP_GRID = list(TEMP_GRID)
TEMPS = sorted({float(AGREEMENT_TAU), *_TEMP_GRID}, reverse=True)

def _tname(t):
    """Variant key for temperature t. str(float) reproduces the v7 spelling
    ('softmax_t1.0', 'softmax_t0.25', 'softmax_t0.1', 'softmax_t0.05') so the
    Section 20 label maps keep resolving."""
    return f"softmax_t{float(t)}"

_AGF_FULL_VARIANT = _tname(AGREEMENT_TAU)

# ---- weighting recipes (all operate on per-method agreement scores) --------
def _softmax_w(scores, tau):
    s = np.asarray(scores, dtype=np.float64) / float(tau)
    s = s - s.max()
    e = np.exp(s)
    return e / (e.sum() + 1e-12)

def _linear_w(scores):
    c = np.clip(np.asarray(scores, dtype=np.float64), 0.0, None)
    tot = c.sum()
    return (c / tot) if tot > 1e-12 else np.full(len(c), 1.0 / len(c))

def _agreement(sim_pair):
    """sim_pair[(i,j)] symmetric -> per-method mean similarity to the OTHER two."""
    K = len(_METHOD_ORDER)
    out = []
    for i in range(K):
        vals = [sim_pair[(min(i, j), max(i, j))] for j in range(K) if j != i]
        out.append(float(np.mean(vals)))
    return np.array(out, dtype=np.float64)

assert AGREEMENT_KERNEL in KERNEL_ABLATION, \
    "AGREEMENT_KERNEL must appear in KERNEL_ABLATION -- the temperature " \
    "variants score against the live kernel."

# ---- (3) variants COMPUTED, with the duplicate collapsed ------------------
VARIANTS = {
    "equal":  lambda A: np.full(3, 1/3),
    "linear": lambda A: _linear_w(A[AGREEMENT_KERNEL]),
}
for _t in TEMPS:
    VARIANTS[_tname(_t)] = (lambda tt: (lambda A: _softmax_w(A[AGREEMENT_KERNEL], tt)))(_t)
for _k in KERNEL_ABLATION:
    if _k == AGREEMENT_KERNEL:
        continue                      # identical to _AGF_FULL_VARIANT -- aliased below
    VARIANTS[f"kernel_{_k}"] = (
        lambda kk: (lambda A: _softmax_w(A[kk], float(AGREEMENT_TAU))))(_k)

# ---- REPORTED tables: (row label, variant actually computed) --------------
TEMP_REPORT = ([("equal", "equal"), ("linear", "linear")]
               + [(_tname(t), _tname(t)) for t in TEMPS])
KERNEL_REPORT = ([("equal", "equal")]
                 + [(f"kernel_{k}",
                     _AGF_FULL_VARIANT if k == AGREEMENT_KERNEL else f"kernel_{k}")
                    for k in KERNEL_ABLATION])

if float(AGREEMENT_TAU) != 1.0:
    log(f"[abl][WARN] AGREEMENT_TAU={AGREEMENT_TAU}: the S4/S5 captions and "
        f"label maps in Section 20 hard-code tau=1. Update them before using "
        f"those .tex files in the paper.")

# ---- statistics (now parameterised on the frame, not a global) ------------
def _paired_tests(aopc_df, sources, ref="equal"):
    """ONE Holm family over the DISTINCT computed variants, per backbone.
    The live-kernel row and the AGF-Full temperature row are the SAME computed
    column, so the shared comparison enters the family once and receives ONE
    adjusted p that both S4 and S5 then report."""
    piv = aopc_df.pivot_table(index="img", columns="variant", values="aopc_ins")
    b = piv[ref].values if ref in piv.columns else np.array([np.nan])
    res, tested = {}, []
    for src in sources:
        a = piv[src].values if src in piv.columns else np.array([np.nan])
        if src == ref:
            res[src] = dict(n_pairs=int(np.sum(np.isfinite(a))), median_diff=0.0,
                            p_raw=1.0, tested=False)
            continue
        m = ~(np.isnan(a) | np.isnan(b)); aa, bb = a[m], b[m]
        ok = len(aa) >= 5 and not np.allclose(aa, bb)
        p = 1.0
        if ok:
            try:
                _, p = wilcoxon(aa, bb, zero_method="wilcox", alternative="two-sided")
            except Exception:
                p, ok = 1.0, False
        res[src] = dict(n_pairs=int(len(aa)),
                        median_diff=float(np.median(aa - bb)) if len(aa) else float("nan"),
                        p_raw=float(p), tested=bool(ok))
        if ok:
            tested.append(src)
    if tested:
        _rej, _adj, _, _ = multipletests([res[s]["p_raw"] for s in tested],
                                         alpha=0.05, method="holm")
        for s, r, pa in zip(tested, _rej, _adj):
            res[s]["p_holm"], res[s]["reject"] = float(pa), bool(r)
    for d in res.values():
        d.setdefault("p_holm", d["p_raw"]); d.setdefault("reject", False)
    return res, len(tested)

def _nanmean(v):
    v = np.asarray(v, dtype=float)
    return float(np.nanmean(v)) if np.isfinite(v).any() else float("nan")

def _summ(aopc_df, report_pairs, TESTS, HOLM_FAMILY):
    """Build ONE reported table. report_pairs: [(row_label, computed_variant)].
    Purely a lookup into TESTS -- no test is re-run and no p is re-corrected."""
    piv = aopc_df.pivot_table(index="img", columns="variant", values="aopc_ins")
    spr = aopc_df.pivot_table(index="img", columns="variant", values="w_spread")
    rows = []
    for label, src in report_pairs:
        a = piv[src].values if src in piv.columns else np.array([np.nan])
        wsp = _nanmean(spr[src].values) if src in spr.columns else float("nan")
        t = TESTS[src]
        rows.append({
            "variant": label,
            "aopc_ins_mean": _nanmean(a),
            "aopc_ins_std": float(np.nanstd(a)) if np.isfinite(a).any() else float("nan"),
            "w_spread_mean": wsp,
            "weights_active": bool(np.isfinite(wsp) and wsp > WEIGHT_SPREAD_THRESH),
            "n_pairs": t["n_pairs"],
            "vs_equal_median_diff": t["median_diff"],
            "vs_equal_p_raw": t["p_raw"],
            "vs_equal_p_holm": t["p_holm"],
            # Section 20 reads `vs_equal_p` into S4/S5 -> ADJUSTED value, so the
            # starred rows and the printed p agree.
            "vs_equal_p": t["p_holm"],
            "beats_equal": bool(t["reject"] and t["median_diff"] > 0),
            "family_size": int(HOLM_FAMILY),
        })
    return pd.DataFrame(rows)

def _print_abl(title, dfx):
    print("\n" + "=" * 92); print(title); print("=" * 92)
    print(f"{'variant':<16}{'AOPC-Ins':>10}{'w_spread':>10}{'active':>9}"
          f"{'Δ vs equal':>12}{'p_raw':>10}{'p_Holm':>10}  beats_equal")
    for _, r in dfx.iterrows():
        star = "  *" if r["beats_equal"] else ""
        print(f"{r['variant']:<16}{r['aopc_ins_mean']:>10.5f}{r['w_spread_mean']:>10.4f}"
              f"{str(bool(r['weights_active'])):>9}"
              f"{r['vs_equal_median_diff']:>12.5f}{r['vs_equal_p_raw']:>10.4g}"
              f"{r['vs_equal_p_holm']:>10.4g}  {bool(r['beats_equal'])}{star}")
    print(f"  Holm family size = {int(dfx['family_size'].iloc[0])}; "
          f"'active' = w_spread > {WEIGHT_SPREAD_THRESH}; "
          f"'*' = beats uniform at alpha=0.05 AFTER Holm.")


# ══ PER-BACKBONE SWEEP ════════════════════════════════════════════════════
def run_weighting_ablation(bb_key, n_abl=None):
    """Run the full Section 18 analysis for ONE backbone. Identical procedure
    to v7; only the backbone is a parameter. Returns (diag_df, temp_df, ker_df)."""
    n_abl = N_ABL if n_abl is None else int(n_abl)
    _spec = next(s for s in MODELS if s["key"] == bb_key)
    log("=" * 70); log(f"WEIGHTING ABLATION — {_spec['display']}"); log("=" * 70)
    _model = _eval_model = _tlayer = None
    try:
        _model, _eval_model, _infeat, _bbname, _meta = load_model(_spec)
        _tlayer = get_target_layer(_eval_model, _bbname)
        # (5) same guard Sections 12/17/19 run: a 1x1 / non-spatial target layer
        # makes GradCAM++ identically zero, which would flow silently into every
        # agreement score, every weight vector and every row of both tables.
        assert_spatial_target(_eval_model, _tlayer, IMG_SIZE, DEVICE,
                              label=_spec["display"])
        # SAME image set for every backbone -> the cross-backbone table is paired.
        _idxs = stratified_sample(n_abl, seed=WABL_SEED)
        log(f"[abl] {len(_idxs)} images | kernel={AGREEMENT_KERNEL} "
            f"| tau={AGREEMENT_TAU} | temps={TEMPS} | variants computed={len(VARIANTS)}")

        # ---- sweep --------------------------------------------------------
        aopc_rows, weight_rows, kernel_rows = [], [], []
        _n_skipped = 0
        for img_idx in tqdm(_idxs, desc=f"weighting ablation [{_spec['display']}]",
                            leave=False):
            gc_gen = sg_gen = ig_gen = None
            try:
                x = load_image_tensor(sample_paths[img_idx])
                gc_gen, sg_gen, ig_gen = make_base_generators(_eval_model, _tlayer)
                fd = fusion_dict(gc_gen, sg_gen, ig_gen)
                agf = AgreementGuidedFusion(fd)
                with torch.inference_mode(False):
                    _fused, pred, indiv, _w, _agr = agf.fuse(
                        x, class_idx=None, target_size=(IMG_SIZE, IMG_SIZE))
                maps = [indiv[m] for m in _METHOD_ORDER]

                # Score the SAME maps under every candidate kernel. Cheap: no
                # attributor is called again, only the similarity function.
                AGR = {}
                for _kern in KERNEL_ABLATION:
                    simk = {}
                    for i in range(3):
                        for j in range(i + 1, 3):
                            simk[(i, j)] = agf.kernel_similarity(maps[i], maps[j], _kern)
                    AGR[_kern] = _agreement(simk)
                    kernel_rows.append({"img": int(img_idx), "kernel": _kern,
                                        "score_spread": float(AGR[_kern].max() - AGR[_kern].min()),
                                        "score_min": float(AGR[_kern].min()),
                                        "score_max": float(AGR[_kern].max())})

                for vname, wfn in VARIANTS.items():
                    w = wfn(AGR)
                    wdict = {m: float(w[k]) for k, m in enumerate(_METHOD_ORDER)}
                    fused = weighted_arith(indiv, wdict)
                    ins = aopc_curve(_eval_model, x, fused, pred, mode="insert")
                    aopc_rows.append({"img": int(img_idx), "variant": vname,
                                      "aopc_ins": float(np.mean(ins)),
                                      "w_spread": float(np.std(w))})
                    if vname == _AGF_FULL_VARIANT:      # the proposed AGF weights
                        weight_rows.append({"img": int(img_idx), **wdict})
            except Exception as e:
                _n_skipped += 1
                log(f"  [abl] skip img {img_idx}: {type(e).__name__}: {e}")
            finally:
                # (5) GradCAM++ holds a forward AND a backward hook on the
                # target layer; release them or they accumulate and fire on
                # every subsequent AOPC forward pass.
                if gc_gen is not None:
                    try: gc_gen.remove()
                    except Exception: pass
                del gc_gen, sg_gen, ig_gen
                free_gpu()

        aopc_df = pd.DataFrame(aopc_rows)
        w_df    = pd.DataFrame(weight_rows)
        if aopc_df.empty or w_df.empty:
            raise RuntimeError(
                f"[abl] no image survived the sweep on {_spec['display']} "
                f"({_n_skipped}/{len(_idxs)} skipped). Refusing to emit a weight "
                "diagnostic -- v7 fell back to a hard-coded 1/3 here, which "
                "prints as 'weights are uniform', i.e. it reports total failure "
                "as a finding. Check the log lines above.")
        if _n_skipped:
            log(f"[abl][WARN] {_spec['display']}: {_n_skipped}/{len(_idxs)} "
                f"images skipped; n={len(w_df)} for the diagnostic.")
        aopc_df.to_csv(os.path.join(DIRS["Statistics"],
                                    f"weighting_ablation_raw_{bb_key}.csv"), index=False)
        w_df.to_csv(os.path.join(DIRS["Statistics"],
                                 f"weighting_ablation_weights_{bb_key}.csv"), index=False)

        # ---- (1) weight distribution diagnostic ---------------------------
        #   ACROSS-IMAGE std of one method's weight: "does this method's weight
        #   move from image to image?"  Distinct from w_spread below.
        diag = []
        for m in _METHOD_ORDER:
            col = w_df[m].values.astype(float)
            diag.append({"method": {"gradcam": "GradCAM++", "smoothgrad": "SmoothGrad",
                                    "ig": "IG"}[m],
                         "weight_mean": float(col.mean()), "weight_std": float(col.std()),
                         "weight_min": float(col.min()), "weight_max": float(col.max())})
        diag_df = pd.DataFrame(diag)
        diag_df["spread_threshold"] = WEIGHT_SPREAD_THRESH
        diag_df["weights_active"] = diag_df["weight_std"] > WEIGHT_SPREAD_THRESH
        save_table(diag_df, f"agf_weight_diagnostic_{bb_key}", float_fmt="%.4f")

        print("\n" + "=" * 66)
        print(f"AGF WEIGHT DISTRIBUTION (weighting diagnostic) — {_spec['display']}")
        print(f"  n={len(w_df)} images | kernel={AGREEMENT_KERNEL} | tau={AGREEMENT_TAU}")
        print("=" * 66)
        print(f"{'method':<12}{'mean':>8}{'std':>8}{'min':>8}{'max':>8}")
        for r in diag:
            print(f"{r['method']:<12}{r['weight_mean']:>8.4f}{r['weight_std']:>8.4f}"
                  f"{r['weight_min']:>8.4f}{r['weight_max']:>8.4f}")
        _meanstd = float(np.mean([r["weight_std"] for r in diag]))
        print(f"\n  mean ACROSS-IMAGE weight std = {_meanstd:.4f} "
              f"-> {_spread_verdict(_meanstd)}")

        # ---- paired tests: ONE Holm family per backbone -------------------
        TESTS, HOLM_FAMILY = _paired_tests(aopc_df, list(VARIANTS))
        temp_df = _summ(aopc_df, TEMP_REPORT, TESTS, HOLM_FAMILY)
        ker_df  = _summ(aopc_df, KERNEL_REPORT, TESTS, HOLM_FAMILY)

        # attach the raw agreement-score spread per kernel (the diagnostic that
        # explains WHY a kernel does or does not move the weights)
        _ks = pd.DataFrame(kernel_rows)
        if len(_ks):
            _kagg = _ks.groupby("kernel")["score_spread"].mean().to_dict()
            ker_df["score_spread_mean"] = [
                _kagg.get(str(v).replace("kernel_", ""), float("nan")) for v in ker_df["variant"]]
            save_table(_ks.groupby("kernel").agg(
                score_spread_mean=("score_spread", "mean"),
                score_spread_std=("score_spread", "std"),
                score_min=("score_min", "min"),
                score_max=("score_max", "max")).reset_index(),
                f"agf_kernel_scores_{bb_key}", float_fmt="%.5f")

        save_table(temp_df, f"agf_temperature_ablation_{bb_key}", float_fmt="%.5f")
        save_table(ker_df,  f"agf_kernel_ablation_{bb_key}",      float_fmt="%.5f")

        _print_abl(f"SOFTMAX-TEMPERATURE ABLATION — {_spec['display']}  (vs AGF-Equal, "
                   f"kernel={AGREEMENT_KERNEL} fixed; tau={AGREEMENT_TAU} is AGF-Full)", temp_df)
        _print_abl(f"AGREEMENT-KERNEL ABLATION — {_spec['display']}  (vs AGF-Equal, "
                   f"tau={AGREEMENT_TAU} throughout)", ker_df)
        if "score_spread_mean" in ker_df.columns:
            print("\n  raw agreement-score spread (max-min over the 3 methods):")
            for _, r in ker_df.iterrows():
                if str(r["variant"]).startswith("kernel_"):
                    _tag = ("  <- live kernel (= AGF-Full)"
                            if str(r["variant"])[7:] == AGREEMENT_KERNEL else "")
                    print(f"    {str(r['variant'])[7:]:<14}{r['score_spread_mean']:.4f}{_tag}")

        # ---- figures ------------------------------------------------------
        fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.4))
        def _vmean(dfx, v):
            s = dfx.loc[dfx.variant == v, "aopc_ins_mean"]
            return float(s.iloc[0]) if len(s) else np.nan
        tmeans = [_vmean(temp_df, _tname(t)) for t in TEMPS]
        axL.plot(range(len(TEMPS)), tmeans, "-o", color="#c0392b", lw=2.4,
                 label="softmax (agreement-weighted)")
        _full_pos = TEMPS.index(float(AGREEMENT_TAU))
        if np.isfinite(tmeans[_full_pos]):
            axL.plot([_full_pos], [tmeans[_full_pos]], "o", ms=13, mfc="none",
                     mec="#c0392b", mew=2.0, label=f"AGF-Full (τ={AGREEMENT_TAU:g})")
        eq_m, lin_m = _vmean(temp_df, "equal"), _vmean(temp_df, "linear")
        if np.isfinite(eq_m):
            axL.axhline(eq_m, ls="--", color="#7f8c8d", label="equal (uniform)")
        if np.isfinite(lin_m):
            axL.axhline(lin_m, ls=":", color="#2980b9", label="linear (no softmax)")
        axL.set_xticks(range(len(TEMPS))); axL.set_xticklabels([f"{t:g}" for t in TEMPS])
        axL.set_xlabel("softmax temperature τ (lower = sharper)")
        axL.set_ylabel("AOPC-Insertion"); axL.set_title("Faithfulness vs weight sharpness")
        axL.grid(alpha=.3); axL.legend(fontsize=8)
        _bx = [w_df[m].values.astype(float) for m in _METHOD_ORDER]
        axR.boxplot(_bx, showmeans=True)
        axR.set_xticks([1, 2, 3]); axR.set_xticklabels(["GradCAM++", "SmoothGrad", "IG"])
        axR.axhline(1/3, ls="--", color="#c0392b", alpha=.6, label="uniform (1/3)")
        axR.set_ylabel("AGF-Full weight"); axR.set_title("Agreement-weight distribution")
        axR.grid(axis="y", alpha=.3); axR.legend(fontsize=8)
        fig.suptitle(f"Agreement-weighting diagnostic — {_spec['display']}", y=1.02)
        fig.tight_layout()
        savefig_all(fig, f"weighting_ablation_{bb_key}")

        try:
            _kv = ker_df[ker_df["variant"].astype(str).str.startswith("kernel_")].copy()
            if len(_kv):
                _kv["kernel"] = _kv["variant"].astype(str).str[7:]
                fig2, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
                a1.bar(_kv["kernel"], _kv["score_spread_mean"], color="#2A9D8F")
                a1.set_ylabel("agreement-score spread (max-min)")
                a1.set_title("Dynamic range of the agreement signal")
                a1.grid(axis="y", alpha=.3)
                a2.bar(_kv["kernel"], _kv["w_spread_mean"], color="#F4A261")
                a2.axhline(WEIGHT_SPREAD_THRESH, color="#c0392b", ls="--", lw=1.2,
                           label=f"inert/active threshold ({WEIGHT_SPREAD_THRESH})")
                a2.set_ylabel("fusion-weight spread (std)")
                a2.set_title("What survives the softmax")
                a2.grid(axis="y", alpha=.3); a2.legend(fontsize=8)
                fig2.suptitle(f"Agreement-kernel ablation — {_spec['display']}", y=1.02)
                fig2.tight_layout()
                savefig_all(fig2, f"kernel_ablation_{bb_key}")
        except Exception as e:
            log(f"[FIG FAIL] kernel_ablation_{bb_key}: {type(e).__name__}: {e}")

        log(f"[abl] done — diagnostic + temperature/kernel tables + 2 figures "
            f"written for {_spec['display']} (Holm-corrected, tau={AGREEMENT_TAU}, "
            f"spread threshold={WEIGHT_SPREAD_THRESH}).")
        return diag_df, temp_df, ker_df
    finally:
        # Release the backbone before the next iteration / Section 19. Binding
        # to None is what actually drops the reference; `del locals()[...]` is
        # a no-op on a locals() snapshot and would leave the model resident.
        plt.close("all")
        _model = _eval_model = _tlayer = None
        free_gpu()


# ══ RUN ═══════════════════════════════════════════════════════════════════
ABL_RESULTS = {}
for _bb in ABL_RUN_BACKBONES:
    if _bb not in _ABL_KEY2DISP:
        log(f"[abl][SKIP] '{_bb}' is not a key in MODELS — skipping."); continue
    try:
        ABL_RESULTS[_bb] = run_weighting_ablation(_bb)
    except Exception as e:
        log(f"[abl][FATAL] {_bb}: {type(e).__name__}: {e}")
        traceback.print_exc()
    plt.close("all"); free_gpu()

# Back-compat alias: v7 exposed a module-level ABL_BACKBONE.
ABL_BACKBONE = ABL_RUN_BACKBONES[-1] if ABL_RUN_BACKBONES else None


# ══ (C) CROSS-BACKBONE CONSOLIDATION (reads from DISK) ════════════════════
#   Re-reads every per-backbone table present in Tables/, so a backbone run in
#   an EARLIER Kaggle session is still included. Emits nothing if fewer than
#   two backbones are on disk (a one-backbone "cross" table is misleading).
def _abl_consolidate():
    _TB = DIRS["Tables"]
    def _load(prefix):
        out = {}
        for p in sorted(glob.glob(os.path.join(_TB, f"{prefix}_*.csv"))):
            key = os.path.basename(p)[len(prefix) + 1:-4]
            if key == "cross" or key not in _ABL_KEY2DISP:
                continue
            try: out[key] = pd.read_csv(p)
            except Exception as e:
                log(f"[abl][cross] unreadable {os.path.basename(p)}: {e}")
        return out

    kers  = _load("agf_kernel_ablation")
    temps = _load("agf_temperature_ablation")
    diags = _load("agf_weight_diagnostic")
    order = [s["key"] for s in MODELS]           # canonical backbone order
    found = [k for k in order if k in kers]
    if len(found) < 2:
        log(f"[abl][cross] only {len(found)} backbone(s) on disk ({found}); "
            f"skipping cross-backbone tables. Run the remaining backbones, then "
            f"re-execute this cell with ABL_RUN_BACKBONES=[] to consolidate.")
        return None

    def _stack(d, extra=None):
        rows = []
        for k in order:
            if k not in d: continue
            df = d[k].copy()
            df.insert(0, "Backbone", _ABL_KEY2DISP[k])
            rows.append(df)
        return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    ker_x, temp_x, diag_x = _stack(kers), _stack(temps), _stack(diags)
    save_table(ker_x,  "agf_kernel_ablation_cross",      float_fmt="%.5f")
    save_table(temp_x, "agf_temperature_ablation_cross", float_fmt="%.5f")
    save_table(diag_x, "agf_weight_diagnostic_cross",    float_fmt="%.4f")

    # ---- headline verdict, computed from the data (never hard-coded) ------
    _live = f"kernel_{AGREEMENT_KERNEL}"
    _lk = ker_x[ker_x["variant"] == _live]
    n_bb = len(found)
    n_inert = int((~_lk["weights_active"].astype(bool)).sum())
    n_beat_k = int(ker_x["beats_equal"].astype(bool).sum())
    n_beat_t = int(temp_x["beats_equal"].astype(bool).sum())
    print("\n" + "=" * 92)
    print(f"CROSS-BACKBONE WEIGHTING VERDICT  (n={n_bb} backbones: "
          f"{', '.join(_ABL_KEY2DISP[k] for k in found)})")
    print("=" * 92)
    print(f"  live kernel = {AGREEMENT_KERNEL}, tau = {AGREEMENT_TAU}, "
          f"threshold = {WEIGHT_SPREAD_THRESH}")
    print(f"  weights INERT (w_spread <= threshold) on {n_inert}/{n_bb} backbones")
    print(f"  variants beating AGF-Equal after Holm: {n_beat_k} (kernel family), "
          f"{n_beat_t} (temperature family)")
    print("\n  live-kernel weight spread per backbone:")
    for _, r in _lk.iterrows():
        print(f"    {r['Backbone']:<14} score_spread="
              f"{r.get('score_spread_mean', float('nan')):.4f}  "
              f"w_spread={r['w_spread_mean']:.4f}  "
              f"active={bool(r['weights_active'])}  "
              f"Δ vs equal={r['vs_equal_median_diff']:+.5f}  "
              f"p_Holm={r['vs_equal_p_holm']:.3g}")
    print("\n  READ THIS BEFORE WRITING THE RESULTS SECTION. If the weights are")
    print("  inert on every backbone and nothing beats AGF-Equal, the paper must")
    print("  NOT claim the agreement weighting is the mechanism. Report AGF as")
    print("  consensus fusion with a demonstrated faithfulness floor, and present")
    print("  this table as the rigorous negative result that isolates it. Argue")
    print("  from the CI width (equivalence), not from p > 0.05 at n=%d." % N_ABL)

    # ---- cross figure: grouped w_spread by kernel x backbone --------------
    try:
        _kv = ker_x[ker_x["variant"].astype(str).str.startswith("kernel_")].copy()
        if len(_kv):
            _kv["kernel"] = _kv["variant"].astype(str).str[7:]
            kern_names = [k for k in KERNEL_ABLATION if k in set(_kv["kernel"])]
            bbs = [_ABL_KEY2DISP[k] for k in found]
            fig3, (b1, b2) = plt.subplots(1, 2, figsize=(12.5, 4.4))
            width = 0.8 / max(len(bbs), 1)
            xs = np.arange(len(kern_names))
            for bi, bb in enumerate(bbs):
                sub = _kv[_kv["Backbone"] == bb].set_index("kernel")
                ss = [float(sub["score_spread_mean"].get(k, np.nan)) for k in kern_names]
                ws = [float(sub["w_spread_mean"].get(k, np.nan)) for k in kern_names]
                b1.bar(xs + bi * width, ss, width, label=bb)
                b2.bar(xs + bi * width, ws, width, label=bb)
            off = width * (len(bbs) - 1) / 2
            for ax, ylab, title in ((b1, "agreement-score spread (max-min)",
                                     "Dynamic range of the agreement signal"),
                                    (b2, "fusion-weight spread (std)",
                                     "What survives the softmax")):
                ax.set_xticks(xs + off); ax.set_xticklabels(kern_names)
                ax.set_ylabel(ylab); ax.set_title(title)
                ax.grid(axis="y", alpha=.3); ax.legend(fontsize=8)
            b2.axhline(WEIGHT_SPREAD_THRESH, color="#c0392b", ls="--", lw=1.2,
                       label=f"inert/active threshold ({WEIGHT_SPREAD_THRESH})")
            b2.legend(fontsize=8)
            fig3.suptitle("Agreement-kernel ablation across backbones "
                          f"(τ={AGREEMENT_TAU:g}, n={N_ABL} images)", y=1.02)
            fig3.tight_layout()
            savefig_all(fig3, "kernel_ablation_cross")
    except Exception as e:
        log(f"[FIG FAIL] kernel_ablation_cross: {type(e).__name__}: {e}")

    log(f"[abl][cross] consolidated {n_bb} backbones -> "
        f"agf_kernel_ablation_cross / agf_temperature_ablation_cross / "
        f"agf_weight_diagnostic_cross (+ kernel_ablation_cross figure).")
    return ker_x, temp_x, diag_x

ABL_CROSS = _abl_consolidate()
plt.close("all"); free_gpu()
mark_section("16c_weighting_kernel_temperature_ablation")

[18:01:49] ======================================================================
[18:01:49] WEIGHTING ABLATION — VGG16-BN
[18:01:49] ======================================================================
[18:01:52] [target-layer] [VGG16-BN] Sequential -> (1, 512, 7, 7) | non-negative 100.0% | OK
[18:01:52] [abl] 40 images | kernel=topk_dice | tau=1.0 | temps=[1.0, 0.25, 0.1, 0.05] | variants computed=8


weighting ablation [VGG16-BN]:   0%|          | 0/40 [00:00<?, ?it/s]


AGF WEIGHT DISTRIBUTION (weighting diagnostic) — VGG16-BN
  n=40 images | kernel=topk_dice | tau=1.0
method          mean     std     min     max
GradCAM++     0.3420  0.0077  0.3243  0.3540
SmoothGrad    0.3288  0.0164  0.2881  0.3552
IG            0.3292  0.0176  0.2968  0.3631

  mean ACROSS-IMAGE weight std = 0.0139 -> INERT (~uniform)

SOFTMAX-TEMPERATURE ABLATION — VGG16-BN  (vs AGF-Equal, kernel=topk_dice fixed; tau=1.0 is AGF-Full)
variant           AOPC-Ins  w_spread   active  Δ vs equal     p_raw    p_Holm  beats_equal
equal              0.83884    0.0000    False     0.00000         1         1  False
linear             0.84064    0.0586     True     0.00025      0.59         1  False
softmax_t1.0       0.84037    0.0137    False     0.00057    0.1281    0.7686  False
softmax_t0.25      0.84020    0.0519     True    -0.00039    0.8472         1  False
softmax_t0.1       0.83801    0.1136     True    -0.00430    0.1499    0.7686  False
softmax_t0.05      0.83820    0.1841   

weighting ablation [DenseNet121]:   0%|          | 0/40 [00:00<?, ?it/s]


AGF WEIGHT DISTRIBUTION (weighting diagnostic) — DenseNet121
  n=40 images | kernel=topk_dice | tau=1.0
method          mean     std     min     max
GradCAM++     0.3454  0.0082  0.3293  0.3594
SmoothGrad    0.3411  0.0126  0.3134  0.3600
IG            0.3135  0.0198  0.2806  0.3527

  mean ACROSS-IMAGE weight std = 0.0136 -> INERT (~uniform)

SOFTMAX-TEMPERATURE ABLATION — DenseNet121  (vs AGF-Equal, kernel=topk_dice fixed; tau=1.0 is AGF-Full)
variant           AOPC-Ins  w_spread   active  Δ vs equal     p_raw    p_Holm  beats_equal
equal              0.78040    0.0000    False     0.00000         1         1  False
linear             0.78122    0.0597     True     0.00198    0.3011    0.4383  False
softmax_t1.0       0.78159    0.0177    False     0.00339   0.04976    0.2797  False
softmax_t0.25      0.78247    0.0649     True     0.00501    0.1461    0.4383  False
softmax_t0.1       0.78497    0.1319     True     0.00528   0.05481    0.2797  False
softmax_t0.05      0.78988    0.1

weighting ablation [ResNet50]:   0%|          | 0/40 [00:00<?, ?it/s]


AGF WEIGHT DISTRIBUTION (weighting diagnostic) — ResNet50
  n=40 images | kernel=topk_dice | tau=1.0
method          mean     std     min     max
GradCAM++     0.3299  0.0044  0.3174  0.3391
SmoothGrad    0.3348  0.0058  0.3176  0.3500
IG            0.3353  0.0065  0.3135  0.3474

  mean ACROSS-IMAGE weight std = 0.0056 -> INERT (~uniform)

SOFTMAX-TEMPERATURE ABLATION — ResNet50  (vs AGF-Equal, kernel=topk_dice fixed; tau=1.0 is AGF-Full)
variant           AOPC-Ins  w_spread   active  Δ vs equal     p_raw    p_Holm  beats_equal
equal              0.52468    0.0000    False     0.00000         1         1  False
linear             0.53676    0.0433     True     0.00903    0.0451    0.2255  False
softmax_t1.0       0.52682    0.0052    False     0.00230    0.5185         1  False
softmax_t0.25      0.52999    0.0208     True     0.00613    0.1659    0.6636  False
softmax_t0.1       0.53685    0.0510     True     0.01545    0.0259    0.1554  False
softmax_t0.05      0.55051    0.0974   

## §20 Qualitative figure

Rows = classes, columns = MRI | Grad-CAM++ | SmoothGrad | IG | Equal Fusion | AGF, first correctly
classified test image per class, predicted class explained. Maps are overlaid on the border-cropped image
the model actually sees.

In [24]:
# ── §20  Qualitative explanation grid (publication figure) ────────────────
# [FINAL] Equal Fusion column added; QUAL_BACKBONE from §1; benchmark preprocessing.
#   ONE clean figure for the paper/thesis: rows = the four tumor classes,
#   columns = [MRI | GradCAM++ | SmoothGrad | IG | AGF]. For each class we pick
#   the first CORRECTLY-classified test image and overlay each method's saliency
#   on the grayscale MRI.
#
#   v6 fixes:
#     (a) HOOK LEAK. make_base_generators() registers a forward AND a full
#         backward hook on the target layer, and this cell never called
#         gc.remove(). Four rows => eight live hooks on the same module, every
#         one of them firing on every subsequent forward pass for the rest of
#         the session. That is the most likely reason the cell started
#         (the log's last line is "[qual] representative images: ...") and
#         never finished. Cleanup is now unconditional via try/finally.
#     (b) TARGET CLASS. It passed class_idx=y, the TRUE label, while every
#         other experiment in this notebook explains the PREDICTED class. On a
#         misclassified image those differ, and a qualitative figure that
#         explains a different class than Table 1 measured is not evidence for
#         Table 1. Images are chosen to be correctly classified, so pred == y
#         here by construction — but it is now derived from the model, not
#         asserted by the label, and a mismatch is logged.
#     (c) NAME COLLISION. This cell defined `_overlay(ax, gray, heat)`, which
#         SHADOWED Section 11's `_overlay(t, amap, alpha)` for the rest of the
#         session. Re-running Section 12 after this cell crashed with a
#         signature error. Renamed to _overlay_gray.
#     (d) NO DOUBLE COMPUTATION: uses fuse_maps on the already-computed maps.
# ---------------------------------------------------------------------------
_METHODS = [("gradcam", "GradCAM++"), ("smoothgrad", "SmoothGrad"),
            ("ig", "IG"), ("equal", "Equal Fusion"), ("agf", "AGF (ours)")]

def _norm01(a):
    a = np.asarray(a, dtype=np.float32)
    lo, hi = float(a.min()), float(a.max())
    return (a - lo) / (hi - lo + 1e-8)

def _overlay_gray(ax, gray, heat=None):
    """Grayscale MRI with an optional jet saliency overlay.

    Renamed from _overlay to stop shadowing Section 11's RGB overlay, which
    has an incompatible signature (t, amap, alpha) and is called from
    build_model_figures().
    """
    ax.imshow(gray, cmap="gray", interpolation="nearest")
    if heat is not None:
        ax.imshow(_norm01(heat), cmap="jet", alpha=0.45, interpolation="bilinear")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.6); s.set_color("#333333")

log("=" * 70); log(f"QUALITATIVE GRID — {QUAL_BACKBONE}"); log("=" * 70)
_spec = next(s for s in MODELS if s["key"] == QUAL_BACKBONE)
_m, _em, _inf, _bb, _meta = load_model(_spec)
_tl = get_target_layer(_em, _bb)
assert_spatial_target(_em, _tl, IMG_SIZE, DEVICE, label=_spec["display"])

# pick the first correctly-classified test image for each class
_picks = {}
for idx in range(len(sample_paths)):
    y = int(sample_labels[idx])
    if y in _picks:
        continue
    xx = load_image_tensor(sample_paths[idx])
    with torch.inference_mode():
        pred = int(_em(xx.unsqueeze(0).to(DEVICE)).argmax(1).item())
    if pred == y:
        _picks[y] = idx
    del xx
    if len(_picks) == len(class_names):
        break
log(f"[qual] representative images: "
    + ", ".join(f"{class_names[y]}#{i}" for y, i in sorted(_picks.items())))
log("[qual] NOTE: selection is the FIRST correctly-classified image per class, "
    "not a random draw. State this in the caption.")

ncols = 1 + len(_METHODS)
nrows = len(_picks)
fig, axes = plt.subplots(nrows, ncols, figsize=(2.05 * ncols, 2.05 * nrows),
                         squeeze=False)          # always 2-D, avoids IndexError

for ri, (y, idx) in enumerate(sorted(_picks.items())):
    # background = the border-cropped input (the model sees this crop, + CLAHE) in grey, so overlays align
    # with the attribution grid (v7.1 used the raw, uncropped image).
    gray = np.array(RemoveBlackBorder()(Image.open(sample_paths[idx]).convert("RGB"))
                    .convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    x = load_image_tensor(sample_paths[idx])
    # NOTE: these MUST NOT be named `gc`/`sg`/`ig`. This loop runs at MODULE
    # level, so `gc = ...` rebinds the imported `gc` MODULE in globals() and the
    # `del` below then removes it entirely -- after which the very next
    # free_gpu() raises `NameError: name 'gc' is not defined` and kills the run
    # before Sections 20/14/15 ever execute. Section 16 uses the same names
    # safely only because its loop is inside a function, so `del` hits locals.
    _gcg = _sgg = _igg = None
    try:
        _gcg, _sgg, _igg = make_base_generators(_em, _tl)
        fd = fusion_dict(_gcg, _sgg, _igg)
        with torch.inference_mode(False):
            # explain the PREDICTED class, as every other experiment does
            gc_m, pred = _gcg.generate(x)
            if pred != y:
                log(f"  [qual] WARNING: class {class_names[y]} image #{idx} "
                    f"predicted as {class_names[pred]}; explaining the "
                    f"prediction, not the label.")
            sg_m, _ = _sgg.generate(x, class_idx=pred)
            ig_m, _ = _igg.generate(x, class_idx=pred)
            _pre = {"gradcam":    resize_map(gc_m, (IMG_SIZE, IMG_SIZE)),
                    "smoothgrad": resize_map(sg_m, (IMG_SIZE, IMG_SIZE)),
                    "ig":         resize_map(ig_m, (IMG_SIZE, IMG_SIZE))}
            fused, indiv, _w, _a = AgreementGuidedFusion(fd).fuse_maps(_pre)
    finally:
        if _gcg is not None: _gcg.remove()      # <- the leak that stalled this cell
        del _gcg, _sgg, _igg
    maps = {"gradcam": indiv["gradcam"], "smoothgrad": indiv["smoothgrad"],
            "ig": indiv["ig"], "equal": weighted_arith(indiv, EQUAL_WEIGHTS), "agf": fused}
    _overlay_gray(axes[ri, 0], gray)
    axes[ri, 0].set_ylabel(class_names[y], fontsize=11, fontweight="bold")
    if ri == 0:
        axes[ri, 0].set_title("MRI", fontsize=11)
    for ci, (mk, mlab) in enumerate(_METHODS, start=1):
        _overlay_gray(axes[ri, ci], gray, maps[mk])
        if ri == 0:
            axes[ri, ci].set_title(mlab, fontsize=11,
                                   fontweight=("bold" if mk == "agf" else "normal"))
    del maps, indiv, _pre, gray, x
    free_gpu()

fig.suptitle(f"Qualitative explanations on {_spec['display']} "
             f"(predicted class; overlays are min–max normalised)",
             y=1.005, fontsize=12)
fig.tight_layout()
savefig_all(fig, "qualitative_grid_publication")
log("[qual] saved qualitative_grid_publication.{png,pdf,svg} to Figures/")
del _m, _em, _tl
free_gpu()
mark_section("20_qualitative")

[18:08:48] ======================================================================
[18:08:48] QUALITATIVE GRID — densenet121
[18:08:48] ======================================================================
[18:08:48] [target-layer] [DenseNet121] Sequential -> (1, 1024, 7, 7) | non-negative 100.0% | OK
[18:08:49] [qual] representative images: glioma#0, meningioma#400, notumor#800, pituitary#1200
[18:08:49] [qual] NOTE: selection is the FIRST correctly-classified image per class, not a random draw. State this in the caption.
[18:09:11] [qual] saved qualitative_grid_publication.{png,pdf,svg} to Figures/


## §21 Publication tables (booktabs LaTeX) — generated only from saved CSVs

Every number in `Tables/publication/*.tex` is read from `Statistics/stats_*.csv`,
`Statistics/cross_metric_means.csv`, the per-backbone `*_xai_metrics.csv`, or the ablation CSVs. Captions
are **descriptive**: they state what is in the table and where inference lives, and they never assert a
win.

In [25]:
# ── §21  Publication tables (curated booktabs LaTeX) ───────────────────────
_TB = DIRS["Tables"]
_PUB = os.path.join(_TB, "publication"); os.makedirs(_PUB, exist_ok=True)
_METH = {DISP[k]: DISP[k].replace(" (Proposed)", " (ours)") for k in ORDER}
def _b(s): return r"\textbf{" + s + "}"
def _fx(x, nd=3):
    try:
        if pd.isna(x): return "--"
    except Exception: pass
    return f"{float(x):.{nd}f}"
def _pfmt(p):
    return "--" if pd.isna(p) else ("$<$0.001" if p < 1e-3 else f"{p:.3f}")
def _wrap(cap, lab, spec, head, rows, notes=None):
    t  = "\\begin{table}[htbp]\n\\centering\n\\small\n"
    t += f"\\caption{{{cap}}}\n\\label{{{lab}}}\n\\begin{{tabular}}{{{spec}}}\n\\toprule\n"
    t += head + " \\\\\n\\midrule\n" + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}\n"
    if notes: t += f"\\\\[2pt]\n\\footnotesize\\textit{{{notes}}}\n"
    return t + "\\end{table}\n"
_written = []
def _save(n, tex):
    with open(os.path.join(_PUB, n), "w") as f: f.write(tex)
    _written.append(n)
_bbd_all = [s["display"] for s in MODELS if s["key"] in ALL_RESULTS]
_MLAB = {"aopc_ins": "Insertion$\\uparrow$", "aopc_del_comp": "Deletion (1$-$AUC)$\\uparrow$",
         "sensitivity": "Max-sens.$\\downarrow$"}

# T1 classification (benchmark TTA x5 numbers, source of truth)
df = pd.read_csv(f"{_TB}/cross_classification_results.csv")
best = {c: df[c].max() for c in ["accuracy", "f1", "mean_auc", "mcc"]}
rows = [" & ".join([r["Backbone"]] + [(_b(_fx(r[c])) if abs(r[c] - best[c]) < 1e-9 else _fx(r[c]))
        for c in ["accuracy", "f1", "mean_auc", "mcc"]]) + " \\\\" for _, r in df.iterrows()]
_save("T1_classification.tex", _wrap(
    "Classification performance on the held-out test set (benchmarking notebook, TTA$\\times$5; "
    "best per column in bold).", "tab:classification", "lcccc",
    "Backbone & Accuracy & Macro-F1 & Mean AUC & MCC", rows))

# T2 descriptive means, all three metrics, all methods incl. Equal Fusion
cm = pd.read_csv(os.path.join(DIRS["Statistics"], "cross_metric_means.csv"))
rows = []
for metric in PRIMARY_METRICS:
    rows.append(f"\\multicolumn{{{1 + len(_bbd_all)}}}{{l}}{{\\textit{{{_MLAB[metric]}}}}} \\\\")
    pv = cm.pivot_table(index="method_key", columns="backbone", values=f"{metric}_mean")[_bbd_all]
    bestv = pv.max() if METRIC_DIRECTION[metric] > 0 else pv.min()
    for mk in ORDER:
        cells = [_METH[DISP[mk]]]
        for bb in _bbd_all:
            v = pv.loc[mk, bb]
            cells.append(_b(_fx(v)) if abs(v - bestv[bb]) < 1e-12 else _fx(v))
        rows.append(" & ".join(cells) + " \\\\")
_n_eval = int(cm["aopc_ins_n"].max()); _n_sens = int(cm["sensitivity_n"].max())
_save("T2_xai_means.tex", _wrap(
    f"Mean explanation scores per method and backbone (insertion/deletion: $n={_n_eval}$ images; "
    f"max-sensitivity: $n={_n_sens}$). Bold marks the best mean in a column and is descriptive only; "
    "paired tests are in Tables~\\ref{tab:primary} and \\ref{tab:secondary}.",
    "tab:xai_means", "l" + "c" * len(_bbd_all), "Method & " + " & ".join(_bbd_all), rows))

# T3a / T3b inferential tables (from §15)
def _stats_rows(sdf):
    rows = []
    for _, r in sdf.iterrows():
        dlt = (f"${r['mean_paired_diff']:+.4f}$ [${r['mean_diff_CI_lo']:+.4f}$, ${r['mean_diff_CI_hi']:+.4f}$]"
               if pd.notna(r.get("mean_paired_diff")) else "--")
        med = f"${r['median_paired_diff']:+.4f}$" if pd.notna(r.get("median_paired_diff")) else "--"
        verdict = {"significant: AGF better": "AGF better", "significant: AGF worse": "AGF worse",
                   "not statistically significant": "n.s.", "not tested": "--"}[r["verdict"]]
        rows.append(f"{r['backbone']} & {DISP[r['reference_key']]} & {_MLAB[r['metric']]} & {int(r['n'])} & "
                    f"{dlt} & {med} & ${r['rank_biserial']:+.3f}$ & {_pfmt(r['p_holm'])} & {verdict} \\\\")
    return rows
_head = ("Backbone & Reference & Metric & $n$ & Mean paired $\\Delta$ [95\\% CI] & Median $\\Delta$ & "
         "$r_{rb}$ & $p_{\\text{Holm}}$ & Result")
_note = ("$\\Delta$ = AGF $-$ reference on the same images (for max-sensitivity a negative $\\Delta$ favours AGF). "
         "Two-sided Wilcoxon signed-rank; $r_{rb}$ = matched-pairs rank-biserial; "
         "Result is stated only after Holm correction at $\\alpha=0.05$.")
_save("T3a_primary_agf_vs_equal.tex", _wrap(
    f"Primary comparison: AGF vs.\\ Equal Fusion, per backbone and metric "
    f"(Holm over {int(STATS_PRIMARY['family_size'].iloc[0])} tests).",
    "tab:primary", "lllcccccc", _head, _stats_rows(STATS_PRIMARY), notes=_note))
_save("T3b_secondary_agf_vs_individual.tex", _wrap(
    f"Secondary comparison: AGF vs.\\ individual attribution methods "
    f"(Holm over {int(STATS_SECONDARY['family_size'].iloc[0])} tests).",
    "tab:secondary", "lllcccccc", _head, _stats_rows(STATS_SECONDARY), notes=_note))

# T4 robustness ablation (Full / Equal / Fixed)
try:
    m = pd.read_csv(f"{_TB}/robustness_ablation_means.csv")
    st = pd.read_csv(f"{_TB}/robustness_ablation_stats.csv")
    rows = []
    for bb in _bbd_all:
        for _, r in m[m.Backbone == bb].iterrows():
            rows.append(f"{bb} & {r['Variant']} & {int(r['n'])} & "
                        f"{r['Sensitivity_mean']:.4f} $\\pm$ {r['Sensitivity_std']:.4f} \\\\")
        for _, r in st[st.Backbone == bb].iterrows():
            res = ("n.s." if not r["significant"] else
                   ("Full more robust" if r["more_robust"] else "Full less robust"))
            rows.append(f" & \\quad {r['comparison']}: median $\\Delta$ {r['median_diff']:+.4f}, "
                        f"$p_{{\\text{{Holm}}}}$={_pfmt(r['p_holm'])} & & {res} \\\\")
        rows.append("\\addlinespace")
    rows = rows[:-1] if rows and rows[-1] == "\\addlinespace" else rows
    _save("T4_robustness_ablation.tex", _wrap(
        f"Robustness ablation (max-sensitivity $\\downarrow$, $n={ROBUST_ABL_N}$ images, "
        f"{ROBUST_ABL_SAMPLES} perturbations): per-image agreement weights (Full) vs.\\ uniform (Equal) vs.\\ "
        "static weights calibrated on the validation split (Fixed). Holm over all rows of this table.",
        "tab:rob_ablation", "llcc", "Backbone & Variant & $n$ & Max-sensitivity", rows))
except FileNotFoundError:
    log("[pub] robustness ablation CSV missing (run §16a) — skipped T4")

# T2b AGF vs Equal Fusion with CIs (replaces post-hoc 'best baseline')
rows = []
for _, r in STATS_PRIMARY.iterrows():
    rows.append(f"{r['backbone']} & {_MLAB[r['metric']]} & {_fx(r['mean_agf'], 4)} & {_fx(r['mean_ref'], 4)} & "
                f"${r['mean_paired_diff']:+.4f}$ [${r['mean_diff_CI_lo']:+.4f}$, ${r['mean_diff_CI_hi']:+.4f}$] & "
                f"{_pfmt(r['p_holm'])} \\\\")
_save("T2b_agf_vs_equal_ci.tex", _wrap(
    "AGF vs.\\ Equal Fusion: means on identical images and the mean paired difference with 95\\% "
    "bootstrap CI.", "tab:agf_equal_ci", "llcccc",
    "Backbone & Metric & AGF & Equal Fusion & Mean paired $\\Delta$ [95\\% CI] & $p_{\\text{Holm}}$", rows,
    notes=("The mean paired $\\Delta$ equals the difference of the two means on the same $n$ images. "
           "A CI above 0 favours AGF for insertion/deletion; a CI below 0 favours AGF for max-sensitivity.")))

# S1 detailed per-backbone metrics (descriptive)
for s in MODELS:
    if s["key"] not in ALL_RESULTS: continue
    bb = s["display"]; tag = bb.replace(" ", "").replace("-", "")
    x = pd.read_csv(f"{_TB}/{bb}/{tag}_xai_metrics.csv"); rows = []
    for _, r in x.iterrows():
        rows.append(" & ".join([_METH.get(r["Method"], r["Method"]), _fx(r["AOPC_Ins"]), _fx(r["AOPC_Del_comp"]),
                    _fx(r["Robustness_Sens"]), f"{int(r['Robustness_n'])}", _fx(r["Gini"]),
                    _fx(r["Entropy_norm"]), f"{r['Runtime_ms']:.0f}"]) + " \\\\")
    _save(f"S1_metrics_{tag}.tex", _wrap(
        f"Detailed descriptive metrics for {bb} (means; runtime of fused methods includes all three base methods).",
        f"tab:metrics_{tag}", "lccccccc",
        "Method & Ins.$\\uparrow$ & Del. (1$-$AUC)$\\uparrow$ & Sens.$\\downarrow$ & $n_{sens}$ & Gini & "
        "Norm. entropy & Runtime (ms)", rows))

# S2 per-class (descriptive, all backbones)
rows = []
for _, r in PER_CLASS_DF[PER_CLASS_DF.metric == "aopc_ins"].iterrows():
    rows.append(f"{r['backbone']} & {r['class']} & {int(r['n'])} & " +
                " & ".join(_fx(r[f'mean_{mk}']) for mk in ORDER) +
                f" & ${r['mean_paired_diff_agf_minus_equal']:+.4f}$ \\\\")
_save("S2_per_class_insertion.tex", _wrap(
    "Per-class mean insertion AOPC (descriptive; no per-class hypothesis tests).",
    "tab:perclass", "llc" + "c" * len(ORDER) + "c",
    "Backbone & Class & $n$ & " + " & ".join(_METH[DISP[k]] for k in ORDER) + " & AGF$-$Equal", rows))

# S3 corruption probe (Holm p)
try:
    ca = pd.read_csv(f"{_TB}/corruption_weighting_{PROBE_BACKBONE}.csv"); rows = []
    for _, r in ca.iterrows():
        star = "$^{*}$" if bool(r.get("significant", False)) else ""
        rows.append(f"{r['corruption'].replace('_', ' ')} & {int(r['sev_i'])} & {int(r['n'])} & "
                    f"${float(r['median_diff']):+.4f}$ & {_pfmt(r['p'])} & {_pfmt(r['p_holm'])}{star} \\\\")
    _pdisp = next(s["display"] for s in MODELS if s["key"] == PROBE_BACKBONE)
    _save("S3_corruption_weighting.tex", _wrap(
        f"AGF-Full vs.\\ AGF-Equal insertion AOPC under input corruption ({_pdisp} only; target class "
        "fixed on the clean image). $\\Delta$median $>0$ favours AGF-Full.",
        "tab:corruption", "lcccccc", "Corruption & Severity & $n$ & $\\Delta$median & $p$ & $p_{\\text{Holm}}$", rows,
        notes=f"$^{{*}}$ significant after Holm over the {len(ca)} conditions of this probe (single backbone; exploratory)."))
except FileNotFoundError:
    log("[pub] corruption table missing (run §16b) — skipped S3")

# S4 / S5 temperature and kernel ablations, every backbone on disk
_lab = {"equal": "Equal (uniform)", "linear": "Linear (no softmax)"}
_klab = {"equal": "Equal (uniform)", "kernel_ssim_cosine": "SSIM $+$ cosine",
         "kernel_pearson": "Pearson", "kernel_topk_dice": "Top-$k$ Dice (AGF)"}
for s in MODELS:
    for kind, fname, lab in (("temperature", f"agf_temperature_ablation_{s['key']}.csv", _lab),
                             ("kernel", f"agf_kernel_ablation_{s['key']}.csv", _klab)):
        p = os.path.join(_TB, fname)
        if not os.path.exists(p): continue
        te = pd.read_csv(p); rows = []
        for _, r in te.iterrows():
            v = str(r["variant"])
            name = lab.get(v, v.replace("softmax_t", "Softmax $\\tau{=}") + "$" if v.startswith("softmax_t") else v)
            if v == f"softmax_t{float(AGREEMENT_TAU)}": name += " (AGF)"
            star = "$^{*}$" if bool(r.get("beats_equal", False)) else ""
            ss = _fx(r["score_spread_mean"], 4) if "score_spread_mean" in te.columns else "--"
            rows.append(f"{name} & {ss} & {_fx(r['w_spread_mean'], 4)} & {_fx(r['aopc_ins_mean'], 4)} & "
                        f"${float(r['vs_equal_median_diff']):+.4f}$ & {_pfmt(r['vs_equal_p_holm'])}{star} \\\\")
        _n_beat = int(te.get("beats_equal", pd.Series(dtype=bool)).astype(bool).sum())
        _save(f"S{'4' if kind == 'temperature' else '5'}_{kind}_ablation_{s['key']}.tex", _wrap(
            f"{kind.capitalize()} ablation on {s['display']} (insertion AOPC, $n={N_ABL}$ images, vs.\\ uniform "
            f"weights). {_n_beat} of {len(te) - 1} variants differ significantly from uniform in its favour "
            "after Holm.", f"tab:{kind}_{s['key']}", "lccccc",
            "Variant & Score spread & Weight spread & Ins. & $\\Delta$median vs.\\ equal & $p_{\\text{Holm}}$",
            rows, notes="Reported, not used to select $\\tau$ or the kernel (both fixed a priori)."))

log(f"[pub] {len(_written)} publication tables -> Tables/publication/: " + ", ".join(_written))
mark_section("21_publication_tables")

[18:09:12] [pub] 17 publication tables -> Tables/publication/: T1_classification.tex, T2_xai_means.tex, T3a_primary_agf_vs_equal.tex, T3b_secondary_agf_vs_individual.tex, T4_robustness_ablation.tex, T2b_agf_vs_equal_ci.tex, S1_metrics_VGG16BN.tex, S1_metrics_DenseNet121.tex, S1_metrics_ResNet50.tex, S2_per_class_insertion.tex, S3_corruption_weighting.tex, S4_temperature_ablation_vgg16.tex, S5_kernel_ablation_vgg16.tex, S4_temperature_ablation_densenet121.tex, S5_kernel_ablation_densenet121.tex, S4_temperature_ablation_resnet50.tex, S5_kernel_ablation_resnet50.tex


## §23–24 Experiment manifest and reproducibility summary

`Exports/experiment_manifest.json` is the auditable record of the run: the frozen configuration, the
environment, checkpoint SHA-256s, the `xai_attribution.py` SHA-256, target layers and verified shapes,
prediction-parity results, subset seeds with index hashes, calibration weights, which sections completed,
and the full list of emitted files. Anything traceable to a run must be traceable through this file:
model + image ID + true class + predicted class + method + AGF configuration + seed + metric all live in
the per-image CSVs, keyed by the same `config_version`.

In [26]:
# ── §22  Consolidated exports (Excel workbook + master JSON + file list) ──
# 1) One Excel workbook, one sheet per registered table.
xlsx_path = os.path.join(DIRS["Exports"], "all_tables.xlsx")
try:
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
        if _EXCEL_SHEETS:
            for sheet, df in _EXCEL_SHEETS.items():
                df.to_excel(xw, sheet_name=sheet[:31], index=False)
        else:
            pd.DataFrame({"info": ["no tables produced"]}).to_excel(xw, index=False)
    log(f"Excel workbook: {xlsx_path} ({len(_EXCEL_SHEETS)} sheets)")
except Exception as e:
    log(f"[WARN] Excel export failed: {e}")

# 2) Master results JSON.
master_path = os.path.join(DIRS["Exports"], "master_results.json")
with open(master_path, "w") as f:
    json.dump({"config_version": CONFIG_VERSION, "config": FROZEN_CONFIG,
               "environment": ENV_INFO, "results": ALL_RESULTS}, f, indent=2, default=float)
log(f"Master JSON: {master_path}")

# 3) File manifest (everything under the output tree).
manifest = []
for d in DIRS.values():
    for root, _, files in os.walk(d):
        for fn in files:
            fp = os.path.join(root, fn)
            manifest.append({"file": os.path.relpath(fp, OUTPUT_DIR),
                             "size_kb": round(os.path.getsize(fp) / 1024, 1)})
df_manifest = pd.DataFrame(sorted(manifest, key=lambda r: r["file"]))
df_manifest.to_csv(os.path.join(DIRS["Exports"], "file_manifest.csv"), index=False)

print("=" * 68); print("OUTPUT MANIFEST"); print("=" * 68)
by_dir = defaultdict(lambda: [0, 0.0])
for r in manifest:
    top = r["file"].split(os.sep)[0]
    by_dir[top][0] += 1; by_dir[top][1] += r["size_kb"]
for top in sorted(by_dir):
    print(f"  {top:<12} {by_dir[top][0]:>4} files   {by_dir[top][1]/1024:>7.2f} MB")
print("-" * 68)
print(f"  TOTAL        {len(manifest):>4} files   "
      f"{sum(r['size_kb'] for r in manifest)/1024:>7.2f} MB")
mark_section("22_exports")

[18:09:12] Excel workbook: /kaggle/working/Exports/all_tables.xlsx (22 sheets)
[18:09:12] Master JSON: /kaggle/working/Exports/master_results.json
OUTPUT MANIFEST
  Exports         2 files      0.06 MB
  Figures        63 files    113.91 MB
  Logs            1 files      0.01 MB
  Results         3 files      0.01 MB
  Statistics     43 files      8.81 MB
  Tables        107 files      0.16 MB
--------------------------------------------------------------------
  TOTAL         219 files    122.96 MB


In [27]:
# ── §23  Experiment manifest ───────────────────────────────────────────────
MANIFEST = {
    "config_version": CONFIG_VERSION,
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "environment": ENV_INFO,
    "frozen_config": FROZEN_CONFIG,
    "code": {"xai_attribution_sha256": XAI_MODULE_SHA256,
             "preprocessing_version": PREPROC_VERSION},
    "data": {"test_dir": TEST_DIR, "n_test": len(sample_paths),
             "test_class_counts": {c: int(sample_labels.count(i)) for i, c in enumerate(class_names)},
             "train_dir": TRAIN_DIR, "n_val_split": len(VAL_PATHS),
             "val_split_verified_against_benchmark": VAL_SPLIT_VERIFIED,
             "n_calibration": len(CALIB_POSITIONS)},
    "checkpoints": {k: {"sha256": v} for k, v in CKPT_SHA.items()},
    "verification": {"prediction_parity": PARITY_DF.to_dict("records"),
                     "target_layers": LAYER_DF.to_dict("records")},
    "calibration_weights": {},
    "backbones": {},
    "statistics": {
        "primary": {"reference": DISP[PRIMARY_REFERENCE], "metrics": PRIMARY_METRICS,
                    "n_tests": int(STATS_PRIMARY["family_size"].iloc[0]) if len(STATS_PRIMARY) else 0,
                    "correction": MULTIPLE_CORRECTION, "alpha": ALPHA},
        "secondary": {"references": [DISP[b] for b in SECONDARY_BASELINES],
                      "n_tests": int(STATS_SECONDARY["family_size"].iloc[0]) if len(STATS_SECONDARY) else 0},
        "files": ["Statistics/stats_primary.csv", "Statistics/stats_secondary.csv",
                  "Statistics/stats_sentences.txt", "Statistics/cross_verdict_matrix.csv"],
    },
    "sections": SECTION_STATUS,
}
for s in MODELS:
    k = s["key"]
    if k in ALL_RESULTS:
        MANIFEST["backbones"][k] = {kk: ALL_RESULTS[k].get(kk) for kk in
                                    ("display", "backbone", "checkpoint", "target_layer",
                                     "target_shape", "subsets", "n_ok", "classification")}
    _cw = os.path.join(DIRS["Statistics"], "calibration", f"{k}_fixed_weights.json")
    if os.path.exists(_cw):
        with open(_cw) as f:
            MANIFEST["calibration_weights"][k] = json.load(f)

MANIFEST["files"] = sorted(
    os.path.relpath(os.path.join(root, fn), OUTPUT_DIR)
    for d in DIRS.values() for root, _, files in os.walk(d) for fn in files)
_mpath = os.path.join(DIRS["Exports"], "experiment_manifest.json")
with open(_mpath, "w") as f:
    json.dump(MANIFEST, f, indent=2, default=str)
log(f"Experiment manifest: {_mpath} ({len(MANIFEST['files'])} files listed)")
mark_section("23_manifest")

[18:09:12] Experiment manifest: /kaggle/working/Exports/experiment_manifest.json (220 files listed)


In [28]:
# ── §24  Reproducibility summary + final cleanup ───────────────────────────
print("=" * 88); print(f"REPRODUCIBILITY SUMMARY — config {CONFIG_VERSION}"); print("=" * 88)
print(f"  seeds            : global={SEED}, eval={EVAL_SEED}, sens={SENS_SEED}, abl={ABL_SEED}, "
      f"wabl={WABL_SEED}, robabl={ROBUST_ABL_SEED}, corrupt={CORRUPT_SEED}, calib={CALIB_SEED}")
print(f"  determinism      : cudnn.deterministic={torch.backends.cudnn.deterministic}, "
      f"benchmark={torch.backends.cudnn.benchmark}, precision={INFERENCE_PRECISION}")
print(f"  preprocessing    : {PREPROC_VERSION} (== benchmark cache)")
print(f"  AGF              : kernel={AGREEMENT_KERNEL}, tau={AGREEMENT_TAU}, q={TOPK_FRAC}, "
      f"rule={AGF_WEIGHT_RULE}")
print(f"  calibration      : {len(CALIB_POSITIONS)} images from the benchmark VALIDATION split "
      f"(verified={VAL_SPLIT_VERIFIED}); never used for evaluation")
print(f"  faithfulness     : {AOPC_N_STEPS} steps x {AOPC_PATCH_PCT:.0%} pixels, grey baseline "
      f"{AOPC_BASELINE_PIXEL}, {AOPC_SCORE}, {AOPC_TARGET_CLASS}, {AOPC_AGGREGATION}")
print(f"  robustness       : {SENS_SAMPLES} x {SENS_NOISE} (sigma={SENS_RADIUS}), "
      f"SG noise frozen={SENS_FREEZE_SG_NOISE}")
print(f"  statistics       : {STATS_TEST} ({STATS_ALTERNATIVE}), {MULTIPLE_CORRECTION} at alpha={ALPHA}; "
      f"primary family vs {DISP[PRIMARY_REFERENCE]}")
print("\n  section status:")
for k in sorted(SECTION_STATUS):
    print(f"    {k:<48} {SECTION_STATUS[k]}")
_expected = ["07_verification", "10_sanity_checks", "12_main_evaluation", "15_statistics",
             "16a_robustness_ablation", "16b_corruption_probe",
             "16c_weighting_kernel_temperature_ablation", "17_cross_architecture",
             "20_qualitative", "21_publication_tables", "22_exports", "23_manifest"]
_missing = [s for s in _expected if s not in SECTION_STATUS]
print(f"\n  sections NOT completed: {_missing if _missing else 'none'}")
if _missing:
    print("  -> results are INCOMPLETE; do not report tables that depend on the missing sections.")
print("\n  per-image traceability: Statistics/per_image/<key>_per_image.csv carries "
      "backbone, image_id, img_idx, true class, predicted class, method, metric values, "
      "config_version, subset seed and the AGF kernel/tau.")

plt.close("all")
free_gpu()
if torch.cuda.is_available():
    print(f"\nFinal VRAM: {torch.cuda.memory_allocated()/1e9:.3f} GB allocated | "
          f"{torch.cuda.memory_reserved()/1e9:.3f} GB reserved")
print("\nArtifacts:")
for name, d in DIRS.items():
    print(f"  {name:<11} {d}")
log("PIPELINE COMPLETE.")

REPRODUCIBILITY SUMMARY — config agf-final-1.0
  seeds            : global=42, eval=42, sens=43, abl=44, wabl=45, robabl=49, corrupt=42, calib=53
  determinism      : cudnn.deterministic=True, benchmark=False, precision=fp32
  preprocessing    : v2:removeblackborder(pad4,thr15,halfopen)+clahe(clip2.0,tile8x8) (== benchmark cache)
  AGF              : kernel=topk_dice, tau=1.0, q=0.1, rule=softmax
  calibration      : 100 images from the benchmark VALIDATION split (verified=True); never used for evaluation
  faithfulness     : 10 steps x 10% pixels, grey baseline 0.5, softmax_probability, clean_prediction_fixed, mean_over_steps_1_to_N
  robustness       : 10 x gaussian_iid_normalized (sigma=0.02), SG noise frozen=True
  statistics       : wilcoxon_signed_rank (two-sided), holm at alpha=0.05; primary family vs Equal Fusion

  section status:
    07_verification                                  completed @ 2026-09-18 13:24:45
    10_sanity_checks                                 completed 

## FINAL RUN CHECKLIST

Check these items before reporting any number from this notebook.

1. **Inputs attached:** benchmarking output with `models/*_best.pth`, `*_probe.npz`, `*_preds.npz`,
   `*_probs.npz`, `benchmark_results.json`; the brain-tumor dataset with **both** `Training/` and
   `Testing/`. Set `MODEL_DIR` in §1 if the slug differs.
2. **Run all, top to bottom, one session.** No cell is meant to be run out of order.
3. **§7 must pass:** `label_order_identical = True` for every backbone, `pred_agreement_vs_bench_clean`
   ≥ 0.98 (≥ 0.995 expected), and every target layer reported as 7×7 with ≥ 99 % non-negative activations.
   If parity fails, stop — the preprocessing is not the benchmark's and nothing downstream is comparable.
4. **§5b must pass:** `val_split_verified_against_benchmark = True` (the calibration split really is the
   benchmark's validation split).
5. **§15 determinism check:** `consistency_main_vs_ablation.csv` shows `max_abs_dev_ins` ≈ 0 (< 1e-5).
6. **§24 summary:** `sections NOT completed` is empty. If not, do not report anything from the missing
   sections.
7. **Report only from:** `Statistics/stats_primary.csv`, `stats_secondary.csv`,
   `Statistics/cross_metric_means.csv` and `Tables/publication/*.tex`. Every sentence about significance
   should be copyable from `Statistics/stats_sentences.txt`, per metric and per backbone.
8. **Never write** "AGF significantly outperforms Equal Fusion" unless the matching row of
   `stats_primary.csv` has `verdict = significant: AGF better` for that backbone **and** that metric; and
   never merge insertion, deletion and robustness into one verdict.
9. **Check `sign_concordant`.** Where it is `False` and the test is significant or a CI excludes zero,
   describe the distribution (paired scatter / histogram from the per-image CSVs) instead of quoting a
   single difference.
10. **Weight diagnostic:** report `weight_diagnostic_main.csv` and the §16c spread numbers alongside any
    statement about the agreement weighting. If the weights stay near 1/3, describe AGF as a consensus
    fusion with a calibrated weighting that was measured to be near-uniform on this data.
11. **Archive** `Exports/experiment_manifest.json` with the manuscript. Cite `config_version` in the paper
    so every table can be traced back to this run.